# 01 - Recolección de Datos

Este notebook centraliza la recolección y verificación de fuentes crudas antes de pasar a limpieza.



## Fuente 1: Stanford AI Index 2025

En este caso la descarga ya fue realizada y quedó guardada en `data/raw/STANFORD AI INDEX 25/`.

La lógica de extracción quedó encapsulada en un script reutilizable para no repetir una descarga tediosa desde cero.

Desde este notebook hacemos dos cosas:

1. Documentar cómo se obtuvo la fuente.

2. Cargar y visualizar los archivos crudos para confirmar que están disponibles en el repositorio antes de la limpieza.

In [1]:
from pathlib import Path

import subprocess

import sys



import pandas as pd

from IPython.display import display



NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

RAW_PATH = PROJECT_ROOT / "data" / "raw"

STANFORD_PATH = RAW_PATH / "STANFORD AI INDEX 25"

MANIFEST_PATH = STANFORD_PATH / "download_manifest.csv"

DOWNLOAD_SCRIPT = PROJECT_ROOT / "src" / "download_public_drive_folder.py"

STANFORD_DRIVE_FOLDER_ID = "1AxxxL9-AsaeMdDKtTNHCR1KqEJTsHCod"



if not STANFORD_PATH.exists():

    raise FileNotFoundError(f"No existe la carpeta esperada: {STANFORD_PATH}")



if not MANIFEST_PATH.exists():

    raise FileNotFoundError(f"No existe el manifiesto esperado: {MANIFEST_PATH}")



print(f"Project root: {PROJECT_ROOT}")

print(f"Stanford raw path: {STANFORD_PATH}")


Project root: /Users/francoia/Documents/MIA/Proyecto Data-Science/research
Stanford raw path: /Users/francoia/Documents/MIA/Proyecto Data-Science/research/data/raw/STANFORD AI INDEX 25


### Reutilizar la extracción sin volver a empezar

La descarga ya está hecha. Si en algún momento necesitas reconstruir esta fuente, usa el script reutilizable en `src/download_public_drive_folder.py`.

La celda siguiente deja el comando preparado, pero por defecto no vuelve a descargar nada.


In [2]:
RUN_STANFORD_DOWNLOAD = False



download_command = [

    sys.executable,

    str(DOWNLOAD_SCRIPT),

    STANFORD_DRIVE_FOLDER_ID,

    str(STANFORD_PATH),

]



print("Comando de extracción reproducible:")

print(" ".join(download_command))



if RUN_STANFORD_DOWNLOAD:

    subprocess.run(download_command, check=True)

else:

    print("Descarga omitida: la fuente ya fue almacenada en data/raw/STANFORD AI INDEX 25.")


Comando de extracción reproducible:
/Users/francoia/Documents/MIA/Proyecto Data-Science/research/.venv/bin/python /Users/francoia/Documents/MIA/Proyecto Data-Science/research/src/download_public_drive_folder.py 1AxxxL9-AsaeMdDKtTNHCR1KqEJTsHCod /Users/francoia/Documents/MIA/Proyecto Data-Science/research/data/raw/STANFORD AI INDEX 25
Descarga omitida: la fuente ya fue almacenada en data/raw/STANFORD AI INDEX 25.


### Inventario de la fuente Stanford

Leemos el manifiesto generado durante la descarga para verificar secciones, tipos de archivo y cobertura de la fuente cruda.


In [3]:
manifest_df = pd.read_csv(MANIFEST_PATH)

manifest_df["section"] = manifest_df["path"].str.split("/").str[0]

manifest_df["asset_type"] = manifest_df["path"].str.extract(r"/(Charts|Data)/", expand=False).fillna("root")



display(manifest_df.head())



summary_df = (

    manifest_df[manifest_df["type"] == "file"]

    .groupby(["section", "asset_type"])

    .size()

    .unstack(fill_value=0)

    .sort_index()

)



print(f"Total registros en manifiesto: {len(manifest_df)}")

display(summary_df)


,type,path,title,id,href,section,asset_type
0,folder,1. Research and Development,1. Research and Development,1ZkMuDF0KwLq9xTXFFQUI_0AYYGueKJh0,https://drive.google.com/drive/folders/1ZkMuDF...,1. Research and Development,root
1,folder,1. Research and Development/Charts,Charts,1AbvZGU_ZVoe34tyBxcUtShyjoDLPMCrp,https://drive.google.com/drive/folders/1AbvZGU...,1. Research and Development,root
2,file,1. Research and Development/Charts/fig_1.1.1.pdf,fig_1.1.1.pdf,1tdthxUt6SumxjBLKTz1HgGc8lojK6xMF,https://drive.google.com/file/d/1tdthxUt6Sumxj...,1. Research and Development,Charts
3,file,1. Research and Development/Charts/fig_1.1.10.pdf,fig_1.1.10.pdf,1aU_AeHerzGDrFBobhYDyl5RDPTPk1nsD,https://drive.google.com/file/d/1aU_AeHerzGDrF...,1. Research and Development,Charts
4,file,1. Research and Development/Charts/fig_1.1.11.pdf,fig_1.1.11.pdf,10VAvzvdo8rpb-7IahrUsBFeykmnLHqGx,https://drive.google.com/file/d/10VAvzvdo8rpb-...,1. Research and Development,Charts


Total registros en manifiesto: 661


asset_type,Charts,Data,root
section,,,
1. Research and Development,59,59,0
2. Technical Performance,51,49,0
3. Responsible AI,44,35,0
4. Economy,64,60,0
5. Science and Medicine,36,26,0
6. Policy and Governance,31,30,0
7. Education,30,30,0
8. Public Opinion,16,16,0
Changelog.rtf,0,0,1


### Visualización de archivos tabulares crudos

Aquí cargamos los CSV ya descargados para comprobar que la fuente está efectivamente disponible y legible desde el repositorio.


In [4]:
csv_inventory_df = (

    manifest_df[

        (manifest_df["type"] == "file")

        & (manifest_df["path"].str.endswith(".csv"))

    ]

    .copy()

)

csv_inventory_df["absolute_path"] = csv_inventory_df["path"].apply(lambda value: STANFORD_PATH / value)

csv_inventory_df["filename"] = csv_inventory_df["absolute_path"].apply(lambda value: value.name)



print(f"CSV disponibles: {len(csv_inventory_df)}")

display(csv_inventory_df[["section", "filename", "path"]].head(20))


CSV disponibles: 305


,section,filename,path
62,1. Research and Development,fig_1.1.1.csv,1. Research and Development/Data/fig_1.1.1.csv
63,1. Research and Development,fig_1.1.10.csv,1. Research and Development/Data/fig_1.1.10.csv
64,1. Research and Development,fig_1.1.11.csv,1. Research and Development/Data/fig_1.1.11.csv
65,1. Research and Development,fig_1.1.12.csv,1. Research and Development/Data/fig_1.1.12.csv
66,1. Research and Development,fig_1.1.13.csv,1. Research and Development/Data/fig_1.1.13.csv
67,1. Research and Development,fig_1.1.2.csv,1. Research and Development/Data/fig_1.1.2.csv
68,1. Research and Development,fig_1.1.3.csv,1. Research and Development/Data/fig_1.1.3.csv
69,1. Research and Development,fig_1.1.4.csv,1. Research and Development/Data/fig_1.1.4.csv
70,1. Research and Development,fig_1.1.5.csv,1. Research and Development/Data/fig_1.1.5.csv
71,1. Research and Development,fig_1.1.6.csv,1. Research and Development/Data/fig_1.1.6.csv


In [5]:
preview_paths = csv_inventory_df["absolute_path"].head(3).tolist()

preview_tables = []



for csv_path in preview_paths:

    preview_df = pd.read_csv(csv_path)

    preview_tables.append(

        {

            "file": csv_path.relative_to(STANFORD_PATH).as_posix(),

            "rows": len(preview_df),

            "columns": len(preview_df.columns),

            "columns_name": list(preview_df.columns),

        }

    )

    print(f"\nPreview: {csv_path.relative_to(STANFORD_PATH).as_posix()}")

    display(preview_df.head())



preview_summary_df = pd.DataFrame(preview_tables)

display(preview_summary_df)



Preview: 1. Research and Development/Data/fig_1.1.1.csv


,Year,Number of AI publications in CS (in thousands)
0,2013,101.885
1,2014,104.410
2,2015,105.736
3,2016,107.266
4,2017,116.937



Preview: 1. Research and Development/Data/fig_1.1.10.csv


,Year,Number of AI publications (in thousands),Label
0,2013,2.404,Robotics
1,2014,2.417,Robotics
2,2015,2.371,Robotics
3,2016,2.486,Robotics
4,2017,2.782,Robotics



Preview: 1. Research and Development/Data/fig_1.1.11.csv


,Number of highly cited publications in top 100,Geographic area,Label
0,50,United States,2023
1,34,China,2023
2,7,Germany,2023
3,7,Hong Kong,2023
4,6,Canada,2023


,file,rows,columns,columns_name
0,1. Research and Development/Data/fig_1.1.1.csv,11,2,"[Year, Number of AI publications in CS (in tho..."
1,1. Research and Development/Data/fig_1.1.10.csv,110,3,"[Year, Number of AI publications (in thousands..."
2,1. Research and Development/Data/fig_1.1.11.csv,30,3,[Number of highly cited publications in top 10...


## Fuente 2: World Bank WDI (World Development Indicators)

Desde esta fuente extraemos las **variables de control X2** del estudio: factores socioeconómicos e institucionales que permiten aislar el efecto de la regulación de IA sobre el ecosistema.

Además de las 4 variables core (PIB per cápita PPP, gasto en I+D, penetración de internet, educación terciaria), extraemos indicadores complementarios de gobernanza (World Governance Indicators), estructura económica y capital humano para mejorar la robustez del modelo.

**Criterio de selección de países:** se seleccionan los países que tienen cobertura en al menos 2 datasets de Stanford AI Index (variables Y), garantizando que no queden registros sin target. Esto produce un mínimo de 65 países.

**API:** `wbgapi` — wrapper oficial de la API del World Bank.

**Periodo:** 2013-2024 (alineado con la serie temporal de Stanford AI Index).

In [ ]:
import wbgapi as wb
import pycountry
import pandas as pd
from pathlib import Path
from IPython.display import display

# ── Paths ──────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RAW_PATH = PROJECT_ROOT / "data" / "raw"
WB_PATH = RAW_PATH / "World Bank WDI"
WB_PATH.mkdir(parents=True, exist_ok=True)

STANFORD_PATH = RAW_PATH / "STANFORD AI INDEX 25"

# ── Indicadores WDI a extraer (analíticos) ─────────────────────────────────
# Clave: código WDI → valor: nombre canónico del estudio
WDI_INDICATORS = {
    # --- Core X2 controls (guía metodológica) ---
    "NY.GDP.PCAP.PP.CD":  "gdp_per_capita_ppp",
    "GB.XPD.RSDV.GD.ZS":  "rd_expenditure",
    "IT.NET.USER.ZS":     "internet_penetration",
    "SE.TER.ENRR":        "tertiary_education",
    # --- Governance (WGI vía WDI) ---
    "GE.EST":             "government_effectiveness",
    "RQ.EST":             "regulatory_quality",
    "RL.EST":             "rule_of_law",
    "CC.EST":             "control_of_corruption",
    "VA.EST":             "voice_accountability",
    "PV.EST":             "political_stability",
    # --- Estructura económica ---
    "NY.GDP.MKTP.CD":     "gdp_current_usd",
    "SP.POP.TOTL":        "population",
    "SL.TLF.TOTL.IN":     "labor_force",
    "BX.KLT.DINV.CD.WD":  "fdi_net_inflows",
    "NE.EXP.GNFS.ZS":     "exports_pct_gdp",
    "FP.CPI.TOTL.ZG":     "inflation_consumer_prices",
    "SL.UEM.TOTL.ZS":     "unemployment_rate",
    # --- Capital humano e infraestructura ---
    "SE.XPD.TOTL.GD.ZS":  "education_expenditure_pct_gdp",
    "IT.CEL.SETS.P2":     "mobile_subscriptions_per100",
    "IP.PAT.RESD":        "patent_applications_residents",
}

# ── Indicadores WGI de metadata (calidad de medición) ─────────────────────
# Estos NO son variables analíticas X2; son metadata para auditar la
# precisión y robustez de los 6 estimadores WGI ya extraídos.
# Se guardan aparte en wdi_governance_metadata.csv.
#
# Decisión metodológica (ver VARIABLES_WORLD_BANK_WDI.md):
#   .STD.ERR  → Error estándar del estimador. Mide precisión.
#   .NO.SRC   → Número de fuentes usadas. Mide robustez del compuesto.
#   .PER.RNK  → EXCLUIDO: transformación monótona de .EST (redundante).
#   .PER.RNK.LOWER/.UPPER → EXCLUIDO: derivado de .STD.ERR (redundante).
WGI_METADATA_INDICATORS = {
    # --- Standard errors ---
    "GE.STD.ERR":  "government_effectiveness_se",
    "RQ.STD.ERR":  "regulatory_quality_se",
    "RL.STD.ERR":  "rule_of_law_se",
    "CC.STD.ERR":  "control_of_corruption_se",
    "VA.STD.ERR":  "voice_accountability_se",
    "PV.STD.ERR":  "political_stability_se",
    # --- Number of sources ---
    "GE.NO.SRC":   "government_effectiveness_nsrc",
    "RQ.NO.SRC":   "regulatory_quality_nsrc",
    "RL.NO.SRC":   "rule_of_law_nsrc",
    "CC.NO.SRC":   "control_of_corruption_nsrc",
    "VA.NO.SRC":   "voice_accountability_nsrc",
    "PV.NO.SRC":   "political_stability_nsrc",
}

# Periodo alineado con Stanford AI Index
YEARS = range(2013, 2025)

print(f"Indicadores WDI analíticos a extraer: {len(WDI_INDICATORS)}")
print(f"Indicadores WGI metadata a extraer:   {len(WGI_METADATA_INDICATORS)}")
print(f"Periodo: {min(YEARS)}-{max(YEARS)}")
print(f"Destino: {WB_PATH}")
print()
print("── Indicadores analíticos (20) ──")
for code, name in WDI_INDICATORS.items():
    print(f"  {code:25s} -> {name}")
print()
print("── Indicadores WGI metadata (12) ──")
for code, name in WGI_METADATA_INDICATORS.items():
    print(f"  {code:25s} -> {name}")

Indicadores WDI a extraer: 20
Periodo: 2013-2024
Destino: /Users/francoia/Documents/MIA/Proyecto Data-Science/research/data/raw/World Bank WDI

  NY.GDP.PCAP.PP.CD         -> gdp_per_capita_ppp
  GB.XPD.RSDV.GD.ZS         -> rd_expenditure
  IT.NET.USER.ZS            -> internet_penetration
  SE.TER.ENRR               -> tertiary_education
  GE.EST                    -> government_effectiveness
  RQ.EST                    -> regulatory_quality
  RL.EST                    -> rule_of_law
  CC.EST                    -> control_of_corruption
  VA.EST                    -> voice_accountability
  PV.EST                    -> political_stability
  NY.GDP.MKTP.CD            -> gdp_current_usd
  SP.POP.TOTL               -> population
  SL.TLF.TOTL.IN            -> labor_force
  BX.KLT.DINV.CD.WD         -> fdi_net_inflows
  NE.EXP.GNFS.ZS            -> exports_pct_gdp
  FP.CPI.TOTL.ZG            -> inflation_consumer_prices
  SL.UEM.TOTL.ZS            -> unemployment_rate
  SE.XPD.TOTL.GD.ZS  

/Users/francoia/Documents/MIA/Proyecto Data-Science/research/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Selección de países: intersección con Stanford AI Index

Seleccionamos solo los países que tienen cobertura en al menos **2 datasets de Stanford AI Index** (variables Y: patentes, startups, talento, skill penetration). Esto garantiza que cada país en el dataset final tenga al menos un target medible.

Luego mapeamos cada nombre a su código ISO3 (requerido por `wbgapi` y por la llave canónica `iso3` del estudio).

In [3]:
# ── Construir lista de países desde Stanford AI Index ──────────────────────
stanford_country_files = {
    "ai_patents":        ("1. Research and Development/Data/fig_1.2.4.csv",  "Geographic Area"),
    "ai_startups":       ("4. Economy/Data/fig_4.3.12.csv",                 "Geographic area"),
    "ai_startups_cumul": ("4. Economy/Data/fig_4.3.13.csv",                 "Geographic area"),
    "ai_talent":         ("4. Economy/Data/fig_4.2.17.csv",                 "Geographic area"),
    "ai_skill":          ("4. Economy/Data/fig_4.2.15.csv",                 "Geographic area"),
}

# Entidades no-país a excluir
NON_COUNTRIES = {
    "World", "Global", "Europe", "Asia-Pacific", "All geographies",
    "British Virgin Islands", "Cayman Islands", "Jersey", "Anguilla",
    "Saint Kitts And Nevis", "Puerto Rico", "Guam", "Hong Kong",
    "Developing markets (incl. India, Central/South America, MENA)",
    "Greater China (incl. Hong Kong, Taiwan, Macau)",
    "Greater China (incl. Hong Kong, Taiwan, and Macau)",
}

coverage = {}
for label, (path, col) in stanford_country_files.items():
    df = pd.read_csv(STANFORD_PATH / path)
    countries = set(df[col].dropna().str.strip().unique()) - NON_COUNTRIES
    for c in countries:
        coverage.setdefault(c, []).append(label)

# Filtrar: al menos 2 datasets Stanford
MIN_STANFORD_DATASETS = 2
selected_countries = {c for c, ds in coverage.items() if len(ds) >= MIN_STANFORD_DATASETS}

# ── Mapear nombre Stanford → ISO3 ─────────────────────────────────────────
# Correcciones manuales para nombres que pycountry no resuelve directamente
MANUAL_ISO3 = {
    "South Korea": "KOR",
    "Russia": "RUS",
    "Turkey": "TUR",
    "Czech Republic": "CZE",
    "Czechia": "CZE",
    "Slovak Republic": "SVK",
    "Slovakia": "SVK",
    "Taiwan": "TWN",
    "Vietnam": "VNM",
    "Viet Nam": "VNM",
    "Iran": "IRN",
    "Bolivia": "BOL",
    "Venezuela": "VEN",
    "Egypt": "EGY",
    "Tanzania": "TZA",
    "Moldova": "MDA",
    "Kosovo": "XKX",
    "Ivory Coast": "CIV",
}

def country_to_iso3(name):
    """Convierte nombre de país a ISO3."""
    if name in MANUAL_ISO3:
        return MANUAL_ISO3[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        # Intentar búsqueda fuzzy
        results = pycountry.countries.search_fuzzy(name)
        if results:
            return results[0].alpha_3
    return None

country_iso3_map = {}
unmapped = []
for c in sorted(selected_countries):
    iso3 = country_to_iso3(c)
    if iso3:
        country_iso3_map[c] = iso3
    else:
        unmapped.append(c)

# Eliminar duplicados por ISO3 (e.g. Czechia y Czech Republic → CZE)
iso3_to_name = {}
for name, iso3 in country_iso3_map.items():
    if iso3 not in iso3_to_name:
        iso3_to_name[iso3] = name

print(f"Países Stanford con {MIN_STANFORD_DATASETS}+ datasets: {len(selected_countries)}")
print(f"Mapeados a ISO3 exitosamente: {len(iso3_to_name)}")
if unmapped:
    print(f"Sin mapeo ISO3 (excluidos): {unmapped}")
print()

# Crear DataFrame resumen
country_df = pd.DataFrame([
    {
        "country_name_stanford": iso3_to_name[iso3],
        "iso3": iso3,
        "stanford_datasets": len(coverage.get(iso3_to_name[iso3], [])),
        "datasets": ", ".join(sorted(coverage.get(iso3_to_name[iso3], []))),
    }
    for iso3 in sorted(iso3_to_name.keys())
])

print(f"Países finales para extracción WDI: {len(country_df)}")
display(country_df)

Países Stanford con 2+ datasets: 65
Mapeados a ISO3 exitosamente: 64

Países finales para extracción WDI: 64


,country_name_stanford,iso3,stanford_datasets,datasets
0,United Arab Emirates,ARE,4,"ai_skill, ai_startups, ai_startups_cumul, ai_t..."
1,Argentina,ARG,4,"ai_patents, ai_skill, ai_startups_cumul, ai_ta..."
2,Australia,AUS,5,"ai_patents, ai_skill, ai_startups, ai_startups..."
3,Austria,AUT,5,"ai_patents, ai_skill, ai_startups, ai_startups..."
4,Belgium,BEL,5,"ai_patents, ai_skill, ai_startups, ai_startups..."
...,...,...,...,...
59,Taiwan,TWN,2,"ai_startups, ai_startups_cumul"
60,Uruguay,URY,3,"ai_skill, ai_startups_cumul, ai_talent"
61,United States,USA,5,"ai_patents, ai_skill, ai_startups, ai_startups..."
62,Vietnam,VNM,2,"ai_startups, ai_startups_cumul"


### Extracción de indicadores vía API del World Bank

Descargamos todos los indicadores para los países seleccionados usando `wbgapi`. La descarga genera un CSV por indicador temático + un CSV consolidado con todos los indicadores en formato long y uno en formato wide.

Los archivos se guardan en `data/raw/World Bank WDI/` sin transformaciones.

In [5]:
# ── Descargar indicadores WDI para los países seleccionados ────────────────
iso3_list = sorted(iso3_to_name.keys())
indicator_codes = list(WDI_INDICATORS.keys())

print(f"Descargando {len(indicator_codes)} indicadores para {len(iso3_list)} países...")
print(f"Periodo: {min(YEARS)}-{max(YEARS)}")
print()

all_records = []

for wb_code, canonical_name in WDI_INDICATORS.items():
    try:
        df_ind = wb.data.DataFrame(
            wb_code,
            economy=iso3_list,
            time=range(min(YEARS), max(YEARS) + 1),
            labels=True,
            columns="series",
            numericTimeKeys=True,
            skipBlanks=True,
        )
        if df_ind.empty:
            print(f"  WARN  {wb_code:25s} ({canonical_name}): sin datos")
            continue

        # El DataFrame tiene MultiIndex(economy, time) + columnas [Country, Time, <indicator>]
        df_ind = df_ind.reset_index()
        df_ind = df_ind.rename(columns={
            "economy": "iso3",
            "time": "year",
            "Country": "country_name",
            wb_code: "value",
        })
        df_ind = df_ind[["iso3", "year", "country_name", "value"]]
        df_ind["wb_indicator"] = wb_code
        df_ind["canonical_name"] = canonical_name
        all_records.append(df_ind)
        n_countries = df_ind["iso3"].nunique()
        n_years = df_ind["year"].nunique()
        print(f"  OK    {wb_code:25s} -> {canonical_name:35s} | {n_countries:3d} países | {n_years:2d} años | {len(df_ind):5d} obs")
    except Exception as e:
        print(f"  ERROR {wb_code:25s} ({canonical_name}): {e}")

if not all_records:
    raise RuntimeError("No se descargó ningún indicador. Revise la conexión a internet o los códigos WDI.")

# ── Concatenar todo en un DataFrame long ──────────────────────────────────
wdi_long_df = pd.concat(all_records, ignore_index=True)
print(f"\nTotal registros (long): {len(wdi_long_df)}")
print(f"Países con datos: {wdi_long_df['iso3'].nunique()}")
print(f"Indicadores descargados: {wdi_long_df['canonical_name'].nunique()}")

# ── Guardar CSV long ──────────────────────────────────────────────────────
long_path = WB_PATH / "wdi_all_indicators_long.csv"
wdi_long_df.to_csv(long_path, index=False)
print(f"\nGuardado: {long_path.relative_to(PROJECT_ROOT)}")

# ── Crear versión wide (países × años para cada indicador) ────────────────
wdi_wide_df = wdi_long_df.pivot_table(
    index=["iso3", "country_name", "year"],
    columns="canonical_name",
    values="value",
    aggfunc="first",
).reset_index()
wdi_wide_df.columns.name = None

wide_path = WB_PATH / "wdi_all_indicators_wide.csv"
wdi_wide_df.to_csv(wide_path, index=False)
print(f"Guardado: {wide_path.relative_to(PROJECT_ROOT)}")

# ── Guardar CSVs individuales por grupo temático ──────────────────────────
groups = {
    "wdi_core_controls.csv": [
        "gdp_per_capita_ppp", "rd_expenditure", "internet_penetration", "tertiary_education",
    ],
    "wdi_governance.csv": [
        "government_effectiveness", "regulatory_quality", "rule_of_law",
        "control_of_corruption", "voice_accountability", "political_stability",
    ],
    "wdi_economic_structure.csv": [
        "gdp_current_usd", "population", "labor_force", "fdi_net_inflows",
        "exports_pct_gdp", "inflation_consumer_prices", "unemployment_rate",
    ],
    "wdi_human_capital_infra.csv": [
        "education_expenditure_pct_gdp", "mobile_subscriptions_per100",
        "patent_applications_residents",
    ],
}

for filename, vars_list in groups.items():
    subset = wdi_long_df[wdi_long_df["canonical_name"].isin(vars_list)]
    if not subset.empty:
        out = WB_PATH / filename
        subset.to_csv(out, index=False)
        print(f"Guardado: {out.relative_to(PROJECT_ROOT)} ({len(subset)} obs)")

print("\nExtracción WDI completada.")

Descargando 20 indicadores para 64 países...
Periodo: 2013-2024

  OK    NY.GDP.PCAP.PP.CD         -> gdp_per_capita_ppp                  |  63 países | 12 años |   756 obs
  OK    GB.XPD.RSDV.GD.ZS         -> rd_expenditure                      |  62 países | 11 años |   547 obs
  OK    IT.NET.USER.ZS            -> internet_penetration                |  63 países | 12 años |   728 obs
  OK    SE.TER.ENRR               -> tertiary_education                  |  62 países | 12 años |   653 obs
  OK    GE.EST                    -> government_effectiveness            |  63 países | 11 años |   693 obs
  OK    RQ.EST                    -> regulatory_quality                  |  63 países | 11 años |   693 obs
  OK    RL.EST                    -> rule_of_law                         |  63 países | 11 años |   693 obs
  OK    CC.EST                    -> control_of_corruption               |  63 países | 11 años |   693 obs
  OK    VA.EST                    -> voice_accountability              

### Extracción de metadata WGI (calidad de medición)

Para cada uno de los 6 indicadores de gobernanza WGI se extraen dos companions de metadata:
- **Standard Error** (`.STD.ERR`): precisión del estimador compuesto.
- **Number of Sources** (`.NO.SRC`): cantidad de fuentes subyacentes usadas.

Estos **no son variables analíticas X2**; son metadata de calidad que permite auditar la confiabilidad de los estimadores WGI. Se guardan en un archivo separado (`wdi_governance_metadata.csv`) para no inflar los outputs analíticos principales.

**Fuente**: Dataset oficial WGI 2025 Revision publicado en [worldbank.org/publication/worldwide-governance-indicators](https://www.worldbank.org/en/publication/worldwide-governance-indicators). Se descarga el Excel con todos los indicadores y se parsean las columnas de Standard Error y Number of Sources. Los indicadores `.STD.ERR` y `.NO.SRC` **no están disponibles** en la API WDI estándar (source 2); solo existen en el dataset nativo WGI (source 57), cuyo endpoint API es extremadamente lento. Por eso se usa la descarga directa del Excel.

**Indicadores excluidos** (documentado en `VARIABLES_WORLD_BANK_WDI.md`):
- `.PER.RNK` (percentile rank): transformación monótona de `.EST`, introduce redundancia perfecta.
- `.PER.RNK.LOWER` / `.PER.RNK.UPPER` (CI bounds): derivados de `.STD.ERR`, redundantes.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Descarga y parseo de metadata WGI (STD.ERR + NO.SRC) desde Excel oficial
# ══════════════════════════════════════════════════════════════════════════
# Los indicadores .STD.ERR y .NO.SRC NO están disponibles en la API WDI
# estándar (source 2). Existen en source 57 (WGI nativo) pero su API es
# extremadamente lenta por datos versionados. Usamos la descarga directa
# del Excel oficial WGI 2025 Revision desde worldbank.org.
# ══════════════════════════════════════════════════════════════════════════
import requests, io, tempfile

WGI_EXCEL_URL = (
    "https://www.worldbank.org/content/dam/sites/govindicators/doc/"
    "wgidataset_with_sourcedata-2025.xlsx"
)

# Mapeo de hoja Excel → prefijo canónico de la dimensión WGI
WGI_SHEET_MAP = {
    "va": "voice_accountability",
    "pv": "political_stability",
    "ge": "government_effectiveness",
    "rq": "regulatory_quality",
    "rl": "rule_of_law",
    "cc": "control_of_corruption",
}

print(f"Descargando WGI Excel dataset desde worldbank.org...")
print(f"URL: {WGI_EXCEL_URL}")
r = requests.get(WGI_EXCEL_URL, timeout=120, allow_redirects=True)
assert r.status_code == 200, f"Error HTTP {r.status_code}"
print(f"Descargado: {len(r.content) / 1024:.0f} KB")

# Guardar temporalmente
wgi_excel_tmp = tempfile.NamedTemporaryFile(suffix=".xlsx", delete=False)
wgi_excel_tmp.write(r.content)
wgi_excel_tmp.close()
print(f"Guardado temporalmente: {wgi_excel_tmp.name}")

# ── Parsear cada hoja: extraer Standard Error + Number of Sources ─────────
meta_records = []
study_iso3 = set(iso3_list)

for sheet, dimension in WGI_SHEET_MAP.items():
    df_sheet = pd.read_excel(wgi_excel_tmp.name, sheet_name=sheet)
    df_sheet = df_sheet[df_sheet["Economy (code)"].isin(study_iso3)]
    df_sheet = df_sheet[df_sheet["Year"].isin(YEARS)]

    n_c = df_sheet["Economy (code)"].nunique()
    n_y = df_sheet["Year"].nunique()

    # Standard Error
    se = df_sheet[["Economy (code)", "Year", "Economy (name)",
                    "Standard error (estimate)"]].copy()
    se.columns = ["iso3", "year", "country_name", "value"]
    se["wb_indicator"] = sheet.upper() + ".STD.ERR"
    se["canonical_name"] = dimension + "_se"
    meta_records.append(se)

    # Number of Sources
    nsrc = df_sheet[["Economy (code)", "Year", "Economy (name)",
                      "Number of sources"]].copy()
    nsrc.columns = ["iso3", "year", "country_name", "value"]
    nsrc["wb_indicator"] = sheet.upper() + ".NO.SRC"
    nsrc["canonical_name"] = dimension + "_nsrc"
    meta_records.append(nsrc)

    print(f"  {sheet.upper():2s} ({dimension:35s}) | {n_c:3d} países | {n_y:2d} años | "
          f"SE=[{se['value'].min():.4f}, {se['value'].max():.4f}] | "
          f"NSRC=[{int(nsrc['value'].min())}, {int(nsrc['value'].max())}]")

# ── Consolidar y guardar ──────────────────────────────────────────────────
wgi_meta_df = pd.concat(meta_records, ignore_index=True)
wgi_meta_df["year"] = wgi_meta_df["year"].astype(int)
wgi_meta_df = wgi_meta_df.dropna(subset=["value"])

meta_path = WB_PATH / "wdi_governance_metadata.csv"
wgi_meta_df.to_csv(meta_path, index=False)

print(f"\nGuardado: {meta_path.relative_to(PROJECT_ROOT)} ({len(wgi_meta_df)} obs)")
print(f"  Países: {wgi_meta_df['iso3'].nunique()}")
print(f"  Indicadores: {wgi_meta_df['canonical_name'].nunique()}")
print(f"  Años: {sorted(wgi_meta_df['year'].unique())}")

# Limpiar archivo temporal
import os
os.unlink(wgi_excel_tmp.name)

print("\nExtracción WGI metadata completada.")

### Verificación y cobertura de la extracción WDI

Comprobamos que los datos descargados tienen cobertura suficiente: cuántos países tienen datos por indicador, y cuál es la cobertura temporal.

In [6]:
# ── Tabla de cobertura por indicador ───────────────────────────────────────
coverage_summary = (
    wdi_long_df
    .groupby("canonical_name")
    .agg(
        paises_con_dato=("iso3", "nunique"),
        anios_con_dato=("year", "nunique"),
        total_obs=("value", "count"),
        valor_min=("value", "min"),
        valor_max=("value", "max"),
        nulls=("value", lambda x: x.isna().sum()),
    )
    .sort_values("paises_con_dato", ascending=False)
)

print("Cobertura por indicador:")
display(coverage_summary)

# ── Cobertura por país: cuántos indicadores tiene cada país ───────────────
country_coverage = (
    wdi_long_df
    .groupby(["iso3", "country_name"])
    .agg(
        indicadores_disponibles=("canonical_name", "nunique"),
        obs_totales=("value", "count"),
        anio_min=("year", "min"),
        anio_max=("year", "max"),
    )
    .sort_values("indicadores_disponibles", ascending=False)
    .reset_index()
)

print(f"\nCobertura por país ({len(country_coverage)} países):")
display(country_coverage)

# ── Archivos generados ────────────────────────────────────────────────────
print("\nArchivos guardados en data/raw/World Bank WDI/:")
for f in sorted(WB_PATH.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:8.1f} KB")

Cobertura por indicador:


,paises_con_dato,anios_con_dato,total_obs,valor_min,valor_max,nulls
canonical_name,,,,,,
control_of_corruption,63,11,693,-1.283923e+00,2.402638e+00,0
internet_penetration,63,12,728,1.000000e+01,1.000000e+02,0
unemployment_rate,63,12,756,2.490000e-01,3.400700e+01,0
rule_of_law,63,11,693,-1.196590e+00,2.124762e+00,0
regulatory_quality,63,11,693,-1.155945e+00,2.308591e+00,0
population,63,12,756,3.237640e+05,1.450936e+09,0
political_stability,63,11,693,-2.130276e+00,1.619806e+00,0
labor_force,63,12,756,1.925590e+05,7.811879e+08,0
mobile_subscriptions_per100,63,11,692,6.431430e+01,2.204120e+02,0



Cobertura por país (63 países):


,iso3,country_name,indicadores_disponibles,obs_totales,anio_min,anio_max
0,ARE,United Arab Emirates,20,210,2013,2024
1,PHL,Philippines,20,219,2013,2024
2,JOR,Jordan,20,215,2013,2024
3,JPN,Japan,20,223,2013,2024
4,KAZ,Kazakhstan,20,226,2013,2024
...,...,...,...,...,...,...
58,IND,India,20,221,2013,2024
59,ZAF,South Africa,20,226,2013,2024
60,EGY,"Egypt, Arab Rep.",19,216,2013,2024
61,CMR,Cameroon,18,207,2013,2024



Archivos guardados en data/raw/World Bank WDI/:
  wdi_all_indicators_long.csv                      879.4 KB
  wdi_all_indicators_wide.csv                      207.8 KB
  wdi_core_controls.csv                            170.9 KB
  wdi_economic_structure.csv                       325.5 KB
  wdi_governance.csv                               255.5 KB
  wdi_human_capital_infra.csv                      127.7 KB


In [7]:
# ── Preview del dataset wide (formato final esperado) ─────────────────────
print("Preview del dataset en formato wide (país-año × indicadores):")
print(f"Shape: {wdi_wide_df.shape}")
print(f"Columnas: {list(wdi_wide_df.columns)}")
print()
display(wdi_wide_df.head(10))

# ── Resumen de missingness por columna ────────────────────────────────────
miss = wdi_wide_df.isnull().mean().sort_values(ascending=False)
miss_df = pd.DataFrame({"columna": miss.index, "pct_missing": (miss.values * 100).round(1)})
print("\nPorcentaje de valores faltantes por columna:")
display(miss_df[miss_df["pct_missing"] > 0])

Preview del dataset en formato wide (país-año × indicadores):
Shape: (756, 23)
Columnas: ['iso3', 'country_name', 'year', 'control_of_corruption', 'education_expenditure_pct_gdp', 'exports_pct_gdp', 'fdi_net_inflows', 'gdp_current_usd', 'gdp_per_capita_ppp', 'government_effectiveness', 'inflation_consumer_prices', 'internet_penetration', 'labor_force', 'mobile_subscriptions_per100', 'patent_applications_residents', 'political_stability', 'population', 'rd_expenditure', 'regulatory_quality', 'rule_of_law', 'tertiary_education', 'unemployment_rate', 'voice_accountability']



,iso3,country_name,year,control_of_corruption,education_expenditure_pct_gdp,exports_pct_gdp,fdi_net_inflows,gdp_current_usd,gdp_per_capita_ppp,government_effectiveness,...,mobile_subscriptions_per100,patent_applications_residents,political_stability,population,rd_expenditure,regulatory_quality,rule_of_law,tertiary_education,unemployment_rate,voice_accountability
0,ARE,United Arab Emirates,2013,1.273543,NaN,95.755184,9.764915e+09,4.096327e+11,87526.325480,1.190101,...,205.105,21.0,0.894792,7693031.0,NaN,0.774973,0.613428,NaN,2.125,-1.019597
1,ARE,United Arab Emirates,2014,1.198810,NaN,94.773856,1.107154e+10,4.249359e+11,87477.734196,1.451583,...,204.192,29.0,0.768909,8059440.0,0.67523,0.987515,0.651002,NaN,1.999,-1.061466
2,ARE,United Arab Emirates,2015,1.034574,NaN,93.862543,8.550902e+09,3.819730e+11,73986.063640,1.500492,...,206.839,35.0,0.748049,8505237.0,0.86570,1.096597,0.614162,NaN,1.793,-1.112433
3,ARE,United Arab Emirates,2016,1.131677,NaN,94.474753,9.604773e+09,3.817171e+11,69986.983568,1.410187,...,220.412,57.0,0.549646,8935095.0,0.93180,0.959918,0.819038,NaN,1.640,-1.050076
4,ARE,United Arab Emirates,2017,1.095754,NaN,94.737396,1.035422e+10,4.033650e+11,70281.888950,1.408903,...,214.701,63.0,0.601950,9223225.0,NaN,1.000955,0.768335,NaN,2.462,-1.096624
5,ARE,United Arab Emirates,2018,1.117051,NaN,90.842958,1.038529e+10,4.405601e+11,77446.226993,1.426340,...,214.842,57.0,0.689256,9346701.0,1.26389,0.919855,0.774245,53.211328,2.236,-1.122660
6,ARE,United Arab Emirates,2019,1.072382,3.86021,93.302309,1.787466e+10,4.339262e+11,79816.301090,1.380270,...,209.033,55.0,0.667538,9445785.0,1.31065,0.968138,0.808277,53.881466,2.331,-1.139524
7,ARE,United Arab Emirates,2020,1.083614,3.98418,98.072468,1.988447e+10,3.571619e+11,66790.682266,1.285267,...,194.468,39.0,0.592605,9401038.0,1.48831,1.074331,0.875136,54.974700,4.294,-1.178066
8,ARE,United Arab Emirates,2021,1.150395,3.89403,100.642574,2.066712e+10,4.224414e+11,68580.148542,1.363703,...,186.301,69.0,0.597572,9575152.0,1.49469,1.006123,0.798968,56.452033,3.105,-1.191426
9,ARE,United Arab Emirates,2022,1.155614,NaN,102.051989,2.273656e+10,5.114034e+11,75071.658630,1.299128,...,195.625,NaN,0.744183,10074977.0,NaN,1.034781,0.836057,60.418823,2.872,-1.127491



Porcentaje de valores faltantes por columna:


,columna,pct_missing
0,patent_applications_residents,29.4
1,rd_expenditure,27.6
2,education_expenditure_pct_gdp,24.9
3,tertiary_education,13.6
4,mobile_subscriptions_per100,8.5
5,voice_accountability,8.3
6,control_of_corruption,8.3
7,rule_of_law,8.3
8,regulatory_quality,8.3
9,political_stability,8.3


### QA específico WGI — Validación de gobernanza y metadata

Verificaciones focalizadas en el bloque WGI:
1. **Escala**: los 6 `.EST` deben estar en rango [-2.5, 2.5].
2. **Cobertura país-año**: cada indicador debe cubrir los mismos 63 países × 11 años (2013-2023).
3. **Rezago estructural 2024**: WGI se publica ~septiembre del año siguiente; 2024 no debe tener datos.
4. **Duplicados**: no debe haber duplicados `iso3 + year + canonical_name`.
5. **Metadata vs Core**: los indicadores metadata (`.STD.ERR`, `.NO.SRC`) deben cubrir los mismos países y años que los `.EST`.
6. **Consistencia de escala metadata**: `.STD.ERR` debe ser > 0; `.NO.SRC` debe ser entero positivo.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# QA ESPECÍFICO WGI — GOBERNANZA + METADATA
# ══════════════════════════════════════════════════════════════════════════
wgi_core_names = [
    "government_effectiveness", "regulatory_quality", "rule_of_law",
    "control_of_corruption", "voice_accountability", "political_stability",
]
wgi_core = wdi_long_df[wdi_long_df["canonical_name"].isin(wgi_core_names)].copy()

problems = []

# ── 1. Escala: .EST debe estar en [-2.5, 2.5] ────────────────────────────
out_of_range = wgi_core[(wgi_core["value"] < -2.5) | (wgi_core["value"] > 2.5)]
if len(out_of_range) > 0:
    problems.append(f"ESCALA: {len(out_of_range)} obs fuera de [-2.5, 2.5]")
    print(f"⚠ ESCALA: {len(out_of_range)} observaciones fuera de rango [-2.5, 2.5]")
    display(out_of_range.head(10))
else:
    print("✓ Escala: todos los .EST en rango [-2.5, 2.5]")

# ── 2. Cobertura país-año por indicador ──────────────────────────────────
print("\n── Cobertura WGI por indicador ──")
for ind in wgi_core_names:
    sub = wgi_core[wgi_core["canonical_name"] == ind]
    n_c = sub["iso3"].nunique()
    yrs = sorted(sub["year"].unique())
    yr_range = f"{yrs[0]}-{yrs[-1]}" if yrs else "N/A"
    miss = sub["value"].isna().mean() * 100
    status = "✓" if n_c == 63 and len(yrs) == 11 else "⚠"
    print(f"  {status} {ind:35s} | {n_c:3d} países | {len(yrs):2d} años ({yr_range}) | miss={miss:.1f}%")
    if n_c != 63:
        problems.append(f"COBERTURA: {ind} tiene {n_c} países (esperados 63)")

# ── 3. Rezago estructural 2024 ───────────────────────────────────────────
wgi_2024 = wgi_core[wgi_core["year"] == 2024]
if len(wgi_2024) > 0:
    print(f"\n⚠ REZAGO 2024: se encontraron {len(wgi_2024)} obs WGI para 2024 — verificar publicación")
    problems.append(f"REZAGO: {len(wgi_2024)} obs WGI con year=2024 (no debería existir)")
else:
    print("\n✓ Rezago 2024: no hay datos WGI para 2024 (esperado — WGI se publica con rezago)")

# ── 4. Duplicados ────────────────────────────────────────────────────────
dups = wgi_core.duplicated(subset=["iso3", "year", "canonical_name"], keep=False)
if dups.sum() > 0:
    problems.append(f"DUPLICADOS: {dups.sum()} registros duplicados en WGI core")
    print(f"\n⚠ DUPLICADOS: {dups.sum()} registros duplicados")
else:
    print("✓ Duplicados: ningún duplicado en iso3 + year + canonical_name")

# ── 5. Metadata vs Core (si se descargó metadata) ────────────────────────
if not wgi_meta_df.empty:
    print("\n── Validación WGI Metadata ──")
    meta_countries = set(wgi_meta_df["iso3"].unique())
    core_countries = set(wgi_core["iso3"].unique())
    meta_years = set(wgi_meta_df["year"].unique())
    core_years = set(wgi_core["year"].unique())

    if meta_countries == core_countries:
        print(f"✓ Metadata cubre los mismos {len(meta_countries)} países que core")
    else:
        diff = core_countries - meta_countries
        if diff:
            problems.append(f"METADATA: {len(diff)} países core sin metadata: {sorted(diff)}")
            print(f"⚠ Países core sin metadata: {sorted(diff)}")

    if meta_years == core_years:
        print(f"✓ Metadata cubre los mismos años que core: {sorted(meta_years)}")
    else:
        print(f"⚠ Años metadata: {sorted(meta_years)} vs core: {sorted(core_years)}")

    # 6. Consistencia de escala metadata
    se_cols = [c for c in wgi_meta_df["canonical_name"].unique() if c.endswith("_se")]
    nsrc_cols = [c for c in wgi_meta_df["canonical_name"].unique() if c.endswith("_nsrc")]

    se_data = wgi_meta_df[wgi_meta_df["canonical_name"].isin(se_cols)]
    if len(se_data) > 0:
        neg_se = se_data[se_data["value"] <= 0]
        if len(neg_se) > 0:
            problems.append(f"SE: {len(neg_se)} standard errors ≤ 0")
            print(f"⚠ {len(neg_se)} standard errors ≤ 0")
        else:
            print(f"✓ Standard errors: todos > 0 (rango {se_data['value'].min():.4f} - {se_data['value'].max():.4f})")

    nsrc_data = wgi_meta_df[wgi_meta_df["canonical_name"].isin(nsrc_cols)]
    if len(nsrc_data) > 0:
        non_int = nsrc_data[nsrc_data["value"] != nsrc_data["value"].round()]
        if len(non_int) > 0:
            problems.append(f"NSRC: {len(non_int)} number-of-sources no enteros")
        else:
            print(f"✓ Number of sources: todos enteros positivos "
                  f"(rango {int(nsrc_data['value'].min())}-{int(nsrc_data['value'].max())})")

    # Resumen por indicador metadata
    print("\n── Cobertura metadata por indicador ──")
    for ind in sorted(wgi_meta_df["canonical_name"].unique()):
        sub = wgi_meta_df[wgi_meta_df["canonical_name"] == ind]
        print(f"  {ind:40s} | {sub['iso3'].nunique():3d} países | "
              f"{sub['year'].nunique():2d} años | {len(sub):5d} obs")
else:
    print("\n⚠ WGI metadata no disponible (API no respondió). Reejecutar cuando la API esté activa.")

# ── Resumen final ─────────────────────────────────────────────────────────
print(f"\n{'=' * 70}")
if problems:
    print(f"QA WGI COMPLETADO: {len(problems)} problemas encontrados")
    for p in problems:
        print(f"  ⚠ {p}")
else:
    print("QA WGI COMPLETADO: 0 problemas — bloque de gobernanza limpio")

# ── Archivos WGI generados ───────────────────────────────────────────────
print(f"\nArchivos WGI en {WB_PATH.relative_to(PROJECT_ROOT)}:")
for f in sorted(WB_PATH.glob("wdi_governance*.csv")):
    df_tmp = pd.read_csv(f)
    print(f"  {f.name:45s} | {df_tmp.shape[0]:6d} rows | {df_tmp.shape[1]:2d} cols")

## Fuente 3: OECD — AI Policy Observatory, STI Scoreboard & MSTI

La OCDE ofrece múltiples fuentes de datos que aportan variables **Y** (targets), **X2** (controles) y metadatos útiles para este estudio. Se extraen datos de tres familias:

### 3a. OECD STI Scoreboard (vía GitHub SDMX XML)
Datos de AI publications, AI patents, VC investment y otros indicadores de ciencia, tecnología e innovación publicados en el STI Scoreboard de la OCDE. Se descargan como XML SDMX desde el repositorio oficial en GitHub.

### 3b. OECD SDMX API — MSTI (Main Science & Technology Indicators)
Indicadores macro de I+D obtenidos directamente de la API SDMX pública de la OCDE. Complementan las variables X2 de World Bank.

### 3c. OECD.AI Visualization Metadata
Catálogo de 191 visualizaciones del portal OECD.AI con metadatos sobre inversión VC, talento IA, publicaciones, compute, etc. Se guarda el catálogo como referencia.

**Cobertura:** ~62 países (miembros OECD + socios clave)
**Periodo:** 2009–2024 (varía por indicador)
**Variables canónicas objetivo:**
- `ai_publications_frac` (Y proxy) — Publicaciones científicas de IA, conteo fraccionado
- `ai_publications_top10_frac` (Y proxy) — Publicaciones IA top 10% citadas
- `ai_patents_pct` (Y) — Patentes IA vía PCT
- `ai_patents_ip5` (Y) — Patentes IA en oficinas IP5
- `vc_seed_pct_gdp` (Y proxy) — VC seed como % del PIB
- `vc_startup_pct_gdp` (Y proxy) — VC startup como % del PIB
- `vc_later_pct_gdp` (Y proxy) — VC later-stage como % del PIB
- `researchers_per_1000_employed` (X2) — Investigadores por cada mil empleados
- `berd_pct_gdp` (X2) — Gasto I+D empresarial como % del PIB
- Indicadores adicionales de MSTI por país-año

In [4]:
# ══════════════════════════════════════════════════════════════════════════
# 3.0  SETUP — Imports, Paths y definición de indicadores OECD
# ══════════════════════════════════════════════════════════════════════════
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import json
import time
from pathlib import Path
from io import StringIO

OECD_PATH = Path("../data/raw/OECD/")
OECD_PATH.mkdir(parents=True, exist_ok=True)

# ── Indicadores del STI Scoreboard (GitHub SDMX XML) ─────────────────────
# Cada clave = nombre del archivo en GitHub, valor = dict con metadata
STI_SCOREBOARD_INDICATORS = {
    # --- Y: AI Publications ---
    "AIPUBS_NBFRAC_V8.txt": {
        "canonical": "ai_publications_frac",
        "description": "Total AI scientific publications (fractional count)",
        "role": "Y_proxy",
        "loc_key": "LOCATION",
    },
    "TOP10_AI_NBFRAC_V8.txt": {
        "canonical": "ai_publications_top10_frac",
        "description": "Top 10% cited AI scientific publications (fractional count)",
        "role": "Y_proxy",
        "loc_key": "LOCATION",
    },
    "FPUBS_1702_NBFRAC_V8.txt": {
        "canonical": "ai_publications_scopus_frac",
        "description": "AI publications in Scopus (fractional count) — field 1702",
        "role": "Y_proxy",
        "loc_key": "LOCATION",
    },
    # --- Y: AI Patents ---
    "PCTAI_NB_V8.txt": {
        "canonical": "ai_patents_pct",
        "description": "AI-related patent families filed under PCT",
        "role": "Y",
        "loc_key": "REF_AREA",
    },
    "IP5AI_NB_V8.txt": {
        "canonical": "ai_patents_ip5",
        "description": "AI-related patent families filed at IP5 offices",
        "role": "Y",
        "loc_key": "REF_AREA",
    },
    # --- Y proxy: Venture Capital ---
    "VCSEED_XGDP_V8.txt": {
        "canonical": "vc_seed_pct_gdp",
        "description": "Venture capital — seed stage as % of GDP",
        "role": "Y_proxy",
        "loc_key": "LOCATION",
    },
    "VCSTART_XGDP_V8.txt": {
        "canonical": "vc_startup_pct_gdp",
        "description": "Venture capital — start-up stage as % of GDP",
        "role": "Y_proxy",
        "loc_key": "LOCATION",
    },
    "VCLATE_XGDP_V8.txt": {
        "canonical": "vc_later_pct_gdp",
        "description": "Venture capital — later stage as % of GDP",
        "role": "Y_proxy",
        "loc_key": "LOCATION",
    },
    # --- X2: Researchers ---
    "TP_RSXEM_V8.txt": {
        "canonical": "researchers_per_1000_employed",
        "description": "Total researchers (FTE) per 1000 total employment",
        "role": "X2",
        "loc_key": "LOCATION",
    },
    # --- Extra: AI quality & collaboration ---
    "TOP10_AI_X_V8.txt": {
        "canonical": "ai_publications_top10_pct",
        "description": "% of AI publications among world's 10% top-cited",
        "role": "extra",
        "loc_key": "LOCATION",
    },
    "AI_INTL_X_V8.txt": {
        "canonical": "ai_publications_intl_collab_pct",
        "description": "% of AI publications involving international collaboration",
        "role": "extra",
        "loc_key": "LOCATION",
    },
    # --- Extra: PhD/ICT enrolment ---
    "PHDENR_ENG_XT_V8.txt": {
        "canonical": "phd_enrolment_engineering_pct",
        "description": "PhD enrolment in engineering as % of all PhD",
        "role": "extra",
        "loc_key": "LOCATION",
    },
}

# ── Indicadores MSTI via SDMX API ────────────────────────────────────────
# Codes verificados contra la respuesta real de la API (USA, 2020)
# MEASURE codes: G=GERD, B=BERD, H=HERD, GV=GOVERD
# UNIT_MEASURE: PT_B1GQ = porcentaje del PIB
MSTI_MEASURES = {
    "G": {
        "canonical": "gerd_pct_gdp",
        "description": "Gross domestic expenditure on R&D as % of GDP",
        "unit": "PT_B1GQ",
    },
    "B": {
        "canonical": "berd_pct_gdp",
        "description": "Business enterprise R&D expenditure as % of GDP",
        "unit": "PT_B1GQ",
    },
    "H": {
        "canonical": "herd_pct_gdp",
        "description": "Higher education R&D expenditure as % of GDP",
        "unit": "PT_B1GQ",
    },
    "GV": {
        "canonical": "goverd_pct_gdp",
        "description": "Government R&D expenditure as % of GDP",
        "unit": "PT_B1GQ",
    },
}

GITHUB_BASE = "https://raw.githubusercontent.com/STIScoreboard/STI.Scoreboard/main/"
SDMX_BASE = "https://sdmx.oecd.org/public/rest/data"

print(f"Ruta de salida: {OECD_PATH.resolve()}")
print(f"STI Scoreboard: {len(STI_SCOREBOARD_INDICATORS)} indicadores")
print(f"MSTI SDMX:      {len(MSTI_MEASURES)} indicadores")
print(f"Total:          {len(STI_SCOREBOARD_INDICATORS) + len(MSTI_MEASURES)} indicadores a extraer")

Ruta de salida: /Users/francoia/Documents/MIA/Proyecto Data-Science/research/data/raw/OECD
STI Scoreboard: 12 indicadores
MSTI SDMX:      4 indicadores
Total:          16 indicadores a extraer


### 3a. Extracción STI Scoreboard — AI Publications, AI Patents, VC, Researchers

Se descargan los archivos SDMX XML del repositorio GitHub oficial del OECD STI Scoreboard. Cada archivo contiene series temporales por país (ISO3 alpha-3) con observaciones año por año.

In [2]:
# ══════════════════════════════════════════════════════════════════════════
# 3a. Descarga y parseo — STI Scoreboard (GitHub SDMX XML)
# ══════════════════════════════════════════════════════════════════════════

def parse_sti_sdmx_xml(xml_text: str, loc_key: str = "LOCATION") -> pd.DataFrame:
    """Parsea un archivo SDMX XML del STI Scoreboard y retorna un DataFrame largo."""
    root = ET.fromstring(xml_text)
    rows = []
    for elem in root.iter():
        if "Series" in elem.tag:
            country = elem.attrib.get(loc_key, elem.attrib.get("LOCATION", elem.attrib.get("REF_AREA", "")))
            for obs in elem:
                if "Obs" in obs.tag:
                    year = obs.attrib.get("TIME_PERIOD", "")
                    value = obs.attrib.get("OBS_VALUE", "")
                    if country and year and value:
                        rows.append({
                            "iso3": country,
                            "year": int(year),
                            "value": float(value),
                        })
    return pd.DataFrame(rows)


# ── Descargar y parsear cada indicador ────────────────────────────────────
sti_dfs = {}
sti_errors = []

for filename, meta in STI_SCOREBOARD_INDICATORS.items():
    canonical = meta["canonical"]
    url = GITHUB_BASE + filename
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        df = parse_sti_sdmx_xml(r.text, loc_key=meta["loc_key"])
        df = df.rename(columns={"value": canonical})
        # Filtrar periodo de interés
        df = df[df["year"].between(2009, 2024)]
        sti_dfs[canonical] = df
        countries = df["iso3"].nunique()
        years = f"{df['year'].min()}-{df['year'].max()}"
        print(f"  ✓ {canonical:40s} | {len(df):5d} obs | {countries:3d} países | {years}")
    except Exception as e:
        sti_errors.append((canonical, str(e)))
        print(f"  ✗ {canonical:40s} | ERROR: {e}")

print(f"\nResumen: {len(sti_dfs)}/{len(STI_SCOREBOARD_INDICATORS)} indicadores descargados")
if sti_errors:
    print(f"Errores: {len(sti_errors)}")
    for name, err in sti_errors:
        print(f"  - {name}: {err}")

  ✓ ai_publications_frac                     |   945 obs |  63 países | 2009-2023
  ✓ ai_publications_top10_frac               |   941 obs |  63 países | 2009-2023
  ✓ ai_publications_scopus_frac              |   930 obs |  62 países | 2009-2023
  ✓ ai_patents_pct                           |  1248 obs | 104 países | 2009-2020
  ✓ ai_patents_ip5                           |  1248 obs | 104 países | 2009-2020
  ✓ vc_seed_pct_gdp                          |   511 obs |  33 países | 2009-2024
  ✓ vc_startup_pct_gdp                       |   511 obs |  33 países | 2009-2024
  ✓ vc_later_pct_gdp                         |   511 obs |  33 países | 2009-2024
  ✓ researchers_per_1000_employed            |   648 obs |  47 países | 2009-2023
  ✓ ai_publications_top10_pct                |   941 obs |  63 países | 2009-2023
  ✓ ai_publications_intl_collab_pct          |   941 obs |  63 países | 2009-2023
  ✓ phd_enrolment_engineering_pct            |  1123 obs |  45 países | 2013-2023

Resumen: 12/12 

### 3b. Extracción MSTI — Main Science & Technology Indicators (SDMX API)

Indicadores macro de I+D descargados directamente desde la API SDMX pública de la OCDE. Estos complementan la variable `rd_expenditure` del World Bank WDI con desgloses más detallados (BERD, HERD, GOVERD).

In [5]:
# ══════════════════════════════════════════════════════════════════════════
# 3b. Descarga — MSTI via SDMX API
# ══════════════════════════════════════════════════════════════════════════
# Endpoint: OECD.STI.STP,DSD_MSTI@DF_MSTI
# Dimensions: REF_AREA.FREQ.MEASURE.UNIT_MEASURE.PRICE_BASE.TRANSFORMATION

msti_dfs = {}
msti_errors = []

for measure_code, meta in MSTI_MEASURES.items():
    canonical = meta["canonical"]
    # Build SDMX query for all countries, annual, specific measure
    # REF_AREA=. (all), FREQ=A, MEASURE=<code>, UNIT_MEASURE=<unit>, PRICE_BASE=_Z, TRANSFORMATION=_Z
    url = (
        f"{SDMX_BASE}/OECD.STI.STP,DSD_MSTI@DF_MSTI/"
        f".A.{measure_code}.{meta['unit']}._Z._Z"
        f"?startPeriod=2013&endPeriod=2024"
    )
    try:
        r = requests.get(
            url,
            timeout=90,
            headers={"Accept": "application/vnd.sdmx.data+csv;file=true"},
        )
        r.raise_for_status()

        # Parse CSV response
        from io import StringIO
        df = pd.read_csv(StringIO(r.text))

        # Keep only needed columns
        df = df[["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={
            "REF_AREA": "iso3",
            "TIME_PERIOD": "year",
            "OBS_VALUE": canonical,
        })
        df["year"] = df["year"].astype(int)
        df[canonical] = pd.to_numeric(df[canonical], errors="coerce")
        df = df.dropna(subset=[canonical])

        msti_dfs[canonical] = df
        countries = df["iso3"].nunique()
        years = f"{df['year'].min()}-{df['year'].max()}"
        print(f"  ✓ {canonical:25s} | {len(df):5d} obs | {countries:3d} países | {years}")

    except Exception as e:
        msti_errors.append((canonical, str(e)))
        print(f"  ✗ {canonical:25s} | ERROR: {e}")

    time.sleep(1)  # Rate limiting cortesía

print(f"\nResumen: {len(msti_dfs)}/{len(MSTI_MEASURES)} indicadores MSTI descargados")
if msti_errors:
    print(f"Errores: {len(msti_errors)}")
    for name, err in msti_errors:
        print(f"  - {name}: {err}")

  ✓ gerd_pct_gdp              |   546 obs |  49 países | 2013-2024
  ✓ berd_pct_gdp              |   550 obs |  49 países | 2013-2024
  ✓ herd_pct_gdp              |   553 obs |  49 países | 2013-2024
  ✓ goverd_pct_gdp            |   553 obs |  49 países | 2013-2024

Resumen: 4/4 indicadores MSTI descargados


### 3c. Catálogo OECD.AI Visualizations

Se extrae el catálogo completo de las 191 visualizaciones del portal OECD.AI vía WordPress REST API. Esto incluye metadatos de inversión VC, talento IA, publicaciones, GPU/compute, incidentes, etc. Los datos detrás de las visualizaciones están alojados en Flourish (acceso restringido), pero el catálogo sirve como referencia y traza documental.

In [6]:
# ══════════════════════════════════════════════════════════════════════════
# 3c. Catálogo OECD.AI Visualizations (WordPress REST API)
# ══════════════════════════════════════════════════════════════════════════

all_vizs = []
for page in range(1, 5):
    url = f"https://wp.oecd.ai/wp-json/wp/v2/visualizations?per_page=100&page={page}"
    try:
        r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200:
            data = r.json()
            all_vizs.extend(data)
            if len(data) < 100:
                break
        else:
            break
    except Exception as e:
        print(f"Error page {page}: {e}")
        break

# Extraer metadata relevante
catalog_rows = []
for viz in all_vizs:
    title = viz.get("title", {}).get("rendered", "")
    vid = viz.get("id", "")
    acf = viz.get("acf", {})
    chart_id = acf.get("chartId", "")
    iframe_url = acf.get("iframeurl", "")
    short_title = acf.get("shortTitle", "")

    # Categorizar
    tl = title.lower()
    if any(k in tl for k in ["polic", "regulat", "legislat", "law", "governance", "strateg"]):
        category = "AI_POLICY"
    elif any(k in tl for k in ["investment", "venture", "funding", "vc "]):
        category = "AI_INVESTMENT"
    elif any(k in tl for k in ["talent", "skill", "job", "employ", "worker", "developer"]):
        category = "AI_TALENT"
    elif any(k in tl for k in ["patent", "publication", "research", "scientific", "model", "foundation"]):
        category = "AI_RESEARCH_MODELS"
    elif any(k in tl for k in ["incident", "risk", "hazard", "harm"]):
        category = "AI_INCIDENTS"
    elif any(k in tl for k in ["compute", "gpu", "cloud", "infrastructure"]):
        category = "AI_INFRA"
    elif any(k in tl for k in ["adoption", "company", "enterprise", "firm"]):
        category = "AI_ECOSYSTEM"
    else:
        category = "OTHER"

    catalog_rows.append({
        "viz_id": vid,
        "title": title,
        "short_title": short_title,
        "category": category,
        "chart_id": chart_id,
        "iframe_url": iframe_url,
    })

oecd_ai_catalog = pd.DataFrame(catalog_rows)

# Guardar catálogo
catalog_path = OECD_PATH / "oecd_ai_visualizations_catalog.csv"
oecd_ai_catalog.to_csv(catalog_path, index=False)

print(f"Catálogo OECD.AI: {len(oecd_ai_catalog)} visualizaciones")
print(f"\nDistribución por categoría:")
display(oecd_ai_catalog["category"].value_counts().to_frame("count"))
print(f"\nGuardado en: {catalog_path}")

Catálogo OECD.AI: 191 visualizaciones

Distribución por categoría:


,count
category,
OTHER,58
AI_RESEARCH_MODELS,53
AI_TALENT,44
AI_INVESTMENT,30
AI_INFRA,4
AI_POLICY,2



Guardado en: ../data/raw/OECD/oecd_ai_visualizations_catalog.csv


### 3d. Consolidación y guardado de todos los indicadores OECD

In [7]:
# ══════════════════════════════════════════════════════════════════════════
# 3d. CONSOLIDACIÓN — Merge de todos los indicadores en formato long y wide
# ══════════════════════════════════════════════════════════════════════════

# ── 1. Unificar todos los DataFrames en formato long ──────────────────────
all_long_rows = []

# STI Scoreboard indicators
for canonical, df in sti_dfs.items():
    for _, row in df.iterrows():
        all_long_rows.append({
            "iso3": row["iso3"],
            "year": row["year"],
            "indicator": canonical,
            "value": row[canonical],
            "source": "OECD_STI_Scoreboard",
        })

# MSTI indicators
for canonical, df in msti_dfs.items():
    for _, row in df.iterrows():
        all_long_rows.append({
            "iso3": row["iso3"],
            "year": row["year"],
            "indicator": canonical,
            "value": row[canonical],
            "source": "OECD_MSTI_SDMX",
        })

oecd_long_df = pd.DataFrame(all_long_rows)
print(f"Dataset OECD long: {oecd_long_df.shape}")
print(f"  Indicadores: {oecd_long_df['indicator'].nunique()}")
print(f"  Países:      {oecd_long_df['iso3'].nunique()}")
print(f"  Período:     {oecd_long_df['year'].min()}-{oecd_long_df['year'].max()}")
print(f"  Obs totales: {len(oecd_long_df)}")

# ── 2. Crear formato wide (país-año × indicadores) ───────────────────────
oecd_wide_df = oecd_long_df.pivot_table(
    index=["iso3", "year"],
    columns="indicator",
    values="value",
    aggfunc="first",
).reset_index()

# Aplanar columnas multiindex
oecd_wide_df.columns.name = None
print(f"\nDataset OECD wide: {oecd_wide_df.shape}")

# ── 3. Guardar CSVs ──────────────────────────────────────────────────────
oecd_long_df.to_csv(OECD_PATH / "oecd_all_indicators_long.csv", index=False)
oecd_wide_df.to_csv(OECD_PATH / "oecd_all_indicators_wide.csv", index=False)

# Guardar también por subfamilia temática
sti_indicators = [c for c in oecd_wide_df.columns if c not in ("iso3", "year")
                  and any(c == meta["canonical"] for meta in STI_SCOREBOARD_INDICATORS.values())]
msti_indicators = [c for c in oecd_wide_df.columns if c not in ("iso3", "year")
                   and any(c == meta["canonical"] for meta in MSTI_MEASURES.values())]

if sti_indicators:
    oecd_wide_df[["iso3", "year"] + sti_indicators].dropna(how="all", subset=sti_indicators).to_csv(
        OECD_PATH / "oecd_sti_scoreboard.csv", index=False
    )
if msti_indicators:
    oecd_wide_df[["iso3", "year"] + msti_indicators].dropna(how="all", subset=msti_indicators).to_csv(
        OECD_PATH / "oecd_msti.csv", index=False
    )

# ── 4. Resumen de archivos generados ─────────────────────────────────────
print("\nArchivos guardados en data/raw/OECD/:")
for f in sorted(OECD_PATH.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s} {size_kb:8.1f} KB")

Dataset OECD long: (12700, 5)
  Indicadores: 16
  Países:      107
  Período:     2009-2024
  Obs totales: 12700

Dataset OECD wide: (1515, 18)

Archivos guardados en data/raw/OECD/:
  oecd_ai_visualizations_catalog.csv                     27.4 KB
  oecd_all_indicators_long.csv                          772.5 KB
  oecd_all_indicators_wide.csv                          170.0 KB
  oecd_msti.csv                                          30.6 KB
  oecd_sti_scoreboard.csv                               140.4 KB


In [4]:
# ══════════════════════════════════════════════════════════════════════════
# 3e. VERIFICACIÓN — Cobertura por indicador y cruce con muestra del estudio
# ══════════════════════════════════════════════════════════════════════════

# ── Cobertura por indicador ───────────────────────────────────────────────
coverage = oecd_long_df.groupby("indicator").agg(
    n_countries=("iso3", "nunique"),
    n_obs=("value", "count"),
    year_min=("year", "min"),
    year_max=("year", "max"),
    mean_value=("value", "mean"),
).sort_values("n_countries", ascending=False)

print("Cobertura por indicador OECD:")
display(coverage)

# ── Cruce con los 86 países del estudio ──────────────────────────────────
study_countries_63 = [
    "ARE","ARG","AUS","AUT","BEL","BGR","BRA","CAN","CHE","CHL","CHN","CMR",
    "COL","CRI","CYP","CZE","DEU","DNK","EGY","ESP","EST","FIN","FRA","GBR",
    "GRC","HRV","HUN","IDN","IND","IRL","ISL","ISR","ITA","JOR","JPN","KAZ",
    "KOR","LTU","LUX","LVA","MEX","MYS","NGA","NLD","NOR","NZL","PER","PHL",
    "POL","PRT","ROU","RUS","SAU","SGP","SVN","SWE","THA","TUN","TUR","URY",
    "USA","VNM","ZAF",
]
study_solo_y = [
    "ARM","BGD","BHR","BLR","BLZ","BRB","ECU","GHA","KEN","LBN","LKA","MAR",
    "MLT","MNG","MUS","PAK","PAN","SRB","SVK","SYC","TWN","UGA","UKR",
]
all_study = set(study_countries_63 + study_solo_y)

# Countries in OECD data that are in our study
oecd_countries = set(oecd_long_df["iso3"].unique())
# OECD uses GBR not UK, but some files might use UK
# Also handle: KOR, TUR mapping
matched = oecd_countries & all_study
oecd_only = oecd_countries - all_study
study_missing = all_study - oecd_countries

# Check if UK is used instead of GBR
if "UK" in oecd_countries and "GBR" not in oecd_countries:
    print("NOTA: OECD usa 'UK' en lugar de 'GBR'")

print(f"\n═══ Cruce OECD ↔ Muestra del estudio ═══")
print(f"Países en OECD data:          {len(oecd_countries)}")
print(f"Países en muestra estudio:    {len(all_study)}")
print(f"Intersección:                 {len(matched)}")
print(f"Solo en OECD (no en muestra): {len(oecd_only)}")
print(f"En muestra sin OECD:          {len(study_missing)}")

if study_missing:
    print(f"\nPaíses de la muestra SIN datos OECD ({len(study_missing)}):")
    print(f"  {sorted(study_missing)}")

# ── Missingness en formato wide ──────────────────────────────────────────
print("\nMissingness por columna (solo países del estudio):")
study_wide = oecd_wide_df[oecd_wide_df["iso3"].isin(all_study)]
if len(study_wide) > 0:
    miss = study_wide.drop(columns=["iso3", "year"]).isnull().mean().sort_values(ascending=False)
    miss_df = pd.DataFrame({"indicator": miss.index, "pct_missing": (miss.values * 100).round(1)})
    display(miss_df)

NameError: name 'oecd_long_df' is not defined

In [9]:
# Auxiliar: contar indicadores OECD por país de la muestra
per_country = oecd_long_df[oecd_long_df["iso3"].isin(all_study)].groupby("iso3")["indicator"].nunique().reset_index()
per_country.columns = ["iso3", "n_oecd_indicators"]
per_country = per_country.sort_values("iso3")
print(per_country.to_string(index=False))

iso3  n_oecd_indicators
 ARE                  7
 ARG                 11
 ARM                  2
 AUS                 15
 AUT                 16
 BEL                 16
 BGR                 16
 BLR                  2
 BRA                  8
 CAN                 16
 CHE                 16
 CHL                 13
 CHN                 12
 COL                 12
 CRI                 13
 CYP                  8
 CZE                 16
 DEU                 16
 DNK                 16
 ECU                  2
 EGY                  7
 ESP                 16
 EST                 16
 FIN                 16
 FRA                 16
 GBR                 16
 GRC                 16
 HRV                 16
 HUN                 16
 IDN                  7
 IND                  7
 IRL                 16
 ISL                 13
 ISR                 15
 ITA                 16
 JOR                  2
 JPN                 16
 KAZ                  7
 KEN                  2
 KOR                 13
 LBN            

## Fuente 4: EC-OECD AI Policy Database (via API buddyweb)

Extraemos la base completa de **iniciativas de política de IA** del observatorio EC-OECD (OECD.AI Policy Navigator). La fuente contiene ~2,200 iniciativas de 73+ países con campos estructurados que permiten construir las **6 variables X1** del estudio:

| Variable X1 | Derivación desde API |
|---|---|
| `has_ai_law` | count(Binding) > 0 por país-año |
| `regulatory_approach` | Categorización de `initiativeType` |
| `year_enacted` | `startYear` de la primera regulación Binding |
| `regulatory_intensity` | Composite: n° Binding + tipos de regulación |
| `enforcement_level` | Presencia de "governance bodies" |
| `thematic_coverage` | Conteo de tags distintos por país |

**API descubierta:** `https://oecd-ai.case-api.buddyweb.fr/policy-initiatives`
- Paginación: 20 items/página, ~111 páginas
- Respuesta JSON con 55+ campos estructurados por iniciativa
- Campos clave: `category`, `extentBinding`, `startYear`, `initiativeType`, `tags`, `gaiinCountry`

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# 4.0  Extracción completa de la API EC-OECD AI Policy
# ══════════════════════════════════════════════════════════════════════════
import requests
import pandas as pd
import json
import time
from pathlib import Path
from collections import Counter, defaultdict

POLICY_API = "https://oecd-ai.case-api.buddyweb.fr/policy-initiatives"
POLICY_PATH = Path("../data/raw/OECD/")
POLICY_PATH.mkdir(parents=True, exist_ok=True)

# ── Mapeo de nombres OECD Policy → ISO3 del estudio ──────────────────────
POLICY_NAME_TO_ISO3 = {
    "Argentina": "ARG", "Armenia": "ARM", "Australia": "AUS",
    "Austria": "AUT", "Bangladesh": "BGD", "Barbados": "BRB",
    "Belgium": "BEL", "Belize": "BLZ", "Brazil": "BRA",
    "Bulgaria": "BGR", "Cameroon": "CMR", "Canada": "CAN",
    "Chile": "CHL", "China (People's Republic of)": "CHN",
    "Colombia": "COL", "Costa Rica": "CRI", "Croatia": "HRV",
    "Cyprus": "CYP", "Czech Republic": "CZE", "Denmark": "DNK",
    "Ecuador": "ECU", "Egypt": "EGY", "Estonia": "EST",
    "Finland": "FIN", "France": "FRA", "Germany": "DEU",
    "Ghana": "GHA", "Greece": "GRC", "Hungary": "HUN",
    "Iceland": "ISL", "India": "IND", "Indonesia": "IDN",
    "Ireland": "IRL", "Israel": "ISR", "Italy": "ITA",
    "Japan": "JPN", "Jordan": "JOR", "Kazakhstan": "KAZ",
    "Kenya": "KEN", "Korea": "KOR", "Latvia": "LVA",
    "Lebanon": "LBN", "Lithuania": "LTU", "Luxembourg": "LUX",
    "Malaysia": "MYS", "Malta": "MLT", "Mexico": "MEX",
    "Mongolia": "MNG", "Morocco": "MAR", "Netherlands": "NLD",
    "New Zealand": "NZL", "Nigeria": "NGA", "Norway": "NOR",
    "Pakistan": "PAK", "Panama": "PAN", "Peru": "PER",
    "Philippines": "PHL", "Poland": "POL", "Portugal": "PRT",
    "Romania": "ROU", "Russian Federation": "RUS",
    "Saudi Arabia": "SAU", "Serbia": "SRB", "Singapore": "SGP",
    "Slovak Republic": "SVK", "Slovenia": "SVN", "South Africa": "ZAF",
    "Spain": "ESP", "Sri Lanka": "LKA", "Sweden": "SWE",
    "Switzerland": "CHE", "Thailand": "THA", "Tunisia": "TUN",
    "Türkiye": "TUR", "Ukraine": "UKR",
    "United Arab Emirates": "ARE", "United Kingdom": "GBR",
    "United States": "USA", "Uruguay": "URY", "Viet Nam": "VNM",
    # Países adicionales en la API pero fuera de la muestra
    "Bahrain": "BHR", "Belarus": "BLR", "Mauritius": "MUS",
    "Seychelles": "SYC", "Taiwan": "TWN", "Uganda": "UGA",
}

# ── Descarga paginada de todas las iniciativas ────────────────────────────
print("Descargando iniciativas de política de IA desde API EC-OECD...")
all_items = []
page = 1
errors = []

while True:
    try:
        r = requests.get(f"{POLICY_API}?page={page}", timeout=30)
        r.raise_for_status()
        data = r.json()
        all_items.extend(data["data"])
        total = data["total"]
        last_page = data["lastPage"]
        
        if page == 1:
            print(f"  Total iniciativas: {total}, Páginas: {last_page}")
        if page % 20 == 0:
            print(f"  Página {page}/{last_page} ({len(all_items)} items)...")
        
        if page >= last_page:
            break
        page += 1
        time.sleep(0.15)  # Rate limiting cortés
    except Exception as e:
        errors.append(f"Página {page}: {e}")
        if len(errors) > 5:
            print(f"  Demasiados errores, deteniendo en página {page}")
            break
        time.sleep(2)
        page += 1

print(f"\nDescarga completada: {len(all_items)} iniciativas en {page} páginas")
if errors:
    print(f"  Errores: {len(errors)}")
    for e in errors:
        print(f"    {e}")

Descargando iniciativas de política de IA desde API EC-OECD...
  Total iniciativas: 2218, Páginas: 111
  Página 20/111 (400 items)...
  Página 40/111 (800 items)...
  Página 60/111 (1200 items)...
  Página 80/111 (1600 items)...
  Página 100/111 (2000 items)...

Descarga completada: 2218 iniciativas en 111 páginas


### 4a. Estructuración y guardado del dataset crudo de políticas

Se extraen los campos más relevantes de cada iniciativa y se mapea el nombre de país a ISO3. Se guarda el dataset crudo completo y un subconjunto filtrado a los 86 países del estudio.

In [11]:
# ══════════════════════════════════════════════════════════════════════════
# 4a. Estructuración del dataset crudo de políticas
# ══════════════════════════════════════════════════════════════════════════

def extract_policy_fields(item):
    """Extrae campos relevantes de una iniciativa de política."""
    gc = item.get("gaiinCountry") or {}
    country_name = gc.get("name", "") if isinstance(gc, dict) else ""
    
    it = item.get("initiativeType") or {}
    it_name = it.get("name", "") if isinstance(it, dict) else ""
    it_category = it.get("category", "") if isinstance(it, dict) else ""
    
    tags = item.get("tags") or []
    tag_names = [t.get("name", "") for t in tags if isinstance(t, dict)] if isinstance(tags, list) else []
    
    # Campos de targeting/sector
    target_groups = item.get("targetGroups") or []
    tg_names = [t.get("name", "") for t in target_groups if isinstance(t, dict)] if isinstance(target_groups, list) else []
    
    sectors = item.get("sectors") or []
    sec_names = [s.get("name", "") for s in sectors if isinstance(s, dict)] if isinstance(sectors, list) else []
    
    # Campos de policy instruments
    policy_instruments = item.get("policyInstruments") or []
    pi_names = [p.get("name", "") for p in policy_instruments if isinstance(p, dict)] if isinstance(policy_instruments, list) else []
    
    return {
        "id": item.get("id"),
        "english_name": item.get("englishName", ""),
        "slug": item.get("slug", ""),
        "country_name": country_name,
        "iso3": POLICY_NAME_TO_ISO3.get(country_name, ""),
        "category": item.get("category", ""),
        "extent_binding": item.get("extentBinding", ""),
        "start_year": item.get("startYear"),
        "end_year": item.get("endYear"),
        "status": item.get("status", ""),
        "initiative_type": it_name,
        "initiative_type_category": it_category,
        "tags": "|".join(tag_names),
        "n_tags": len(tag_names),
        "target_groups": "|".join(tg_names),
        "sectors": "|".join(sec_names),
        "policy_instruments": "|".join(pi_names),
        "n_policy_instruments": len(pi_names),
        "description": (item.get("description", "") or "")[:500],
    }

# ── Construir DataFrame ──────────────────────────────────────────────────
policy_rows = [extract_policy_fields(item) for item in all_items]
policy_raw_df = pd.DataFrame(policy_rows)

# ── Resumen general ──────────────────────────────────────────────────────
print(f"Dataset crudo de políticas: {policy_raw_df.shape}")
print(f"  Países únicos: {policy_raw_df['country_name'].nunique()}")
print(f"  Con ISO3 mapeado: {policy_raw_df[policy_raw_df['iso3'] != '']['iso3'].nunique()}")
print(f"  Rango temporal: {policy_raw_df['start_year'].min()}-{policy_raw_df['start_year'].max()}")
print(f"\nDistribución por extent_binding:")
print(policy_raw_df["extent_binding"].value_counts(dropna=False).to_string())
print(f"\nTop 15 categorías:")
print(policy_raw_df["category"].value_counts().head(15).to_string())
print(f"\nTop 15 initiative_type:")
print(policy_raw_df["initiative_type"].value_counts().head(15).to_string())

# ── Guardar dataset crudo completo ────────────────────────────────────────
raw_all_path = POLICY_PATH / "oecd_ai_policy_initiatives_all.csv"
policy_raw_df.to_csv(raw_all_path, index=False)
print(f"\nGuardado completo: {raw_all_path.name} ({raw_all_path.stat().st_size/1024:.1f} KB)")

# ── Filtrar a los 86 países del estudio ──────────────────────────────────
study_iso3_set = set(study_countries_63 + study_solo_y)  # 63 + 23 = 86
policy_study_df = policy_raw_df[policy_raw_df["iso3"].isin(study_iso3_set)].copy()

study_path = POLICY_PATH / "oecd_ai_policy_initiatives_study.csv"
policy_study_df.to_csv(study_path, index=False)
print(f"Guardado muestra: {study_path.name} ({study_path.stat().st_size/1024:.1f} KB)")
print(f"  Iniciativas del estudio: {len(policy_study_df)}")
print(f"  Países del estudio con datos: {policy_study_df['iso3'].nunique()}/86")

# Países del estudio sin cobertura en esta fuente
covered = set(policy_study_df["iso3"].unique())
missing_policy = study_iso3_set - covered
if missing_policy:
    print(f"  Países sin datos policy: {sorted(missing_policy)}")

Dataset crudo de políticas: (2218, 19)
  Países únicos: 74
  Con ISO3 mapeado: 68
  Rango temporal: 1956-2026

Distribución por extent_binding:
extent_binding
Non-binding    1842
None            245
Binding         131

Top 15 categorías:
category
AI policy initiatives, programmes and projects                               1550
Regulations, guidelines and standards                                         381
National – AI governance bodies or mechanisms                                 135
AI Policy Frameworks and Initiatives (intergovernmental or supranational)      81
National – Strategy                                                            68
AI Governance Bodies and Mechanisms (intergovernmental or supranational)        3

Top 15 initiative_type:
initiative_type
AI Research Centres/Centres of Excellence/Expertise, specialised AI teams, labs                 224
Other AI policy initiatives, programmes and projects                                            184
Networks, communiti

### 4b. Construcción de variables X1 derivadas por país-año

Se construyen las 6 variables X1 del estudio a partir de los datos crudos de políticas. El nivel de análisis es **país-año** (un registro por combinación iso3 × año), acumulando las iniciativas vigentes hasta cada año.

In [12]:
# ══════════════════════════════════════════════════════════════════════════
# 4b. Construcción de variables X1 por país-año
# ══════════════════════════════════════════════════════════════════════════
# Se usa una lógica acumulativa: para cada país y año, se contabilizan todas
# las iniciativas cuyo startYear <= año en cuestión (y que no hayan terminado).

# Periodo de análisis alineado con el estudio
STUDY_YEARS = range(2013, 2025)

# Filtrar solo países del estudio con datos
df = policy_study_df.copy()
df["start_year"] = pd.to_numeric(df["start_year"], errors="coerce")
df["end_year"] = pd.to_numeric(df["end_year"], errors="coerce")

# ── Clasificación de regulatory_approach ──────────────────────────────────
def classify_approach(initiatives_for_country):
    """Clasifica el enfoque regulatorio de un país basado en sus iniciativas."""
    types = initiatives_for_country["initiative_type"].str.lower().tolist()
    cats = initiatives_for_country["category"].str.lower().tolist()
    binding = initiatives_for_country["extent_binding"].tolist()
    
    has_binding_regulation = any(
        b == "Binding" and ("regulation" in str(c) or "standard" in str(c))
        for b, c in zip(binding, cats)
    )
    has_strategy = any("strategy" in str(t) or "strategy" in str(c) for t, c in zip(types, cats))
    has_governance = any("governance" in str(c) for c in cats)
    
    if has_binding_regulation and has_strategy:
        return "comprehensive"
    elif has_binding_regulation:
        return "regulation_focused"
    elif has_strategy:
        return "strategy_led"
    else:
        return "light_touch"

# ── Construir panel país-año ──────────────────────────────────────────────
x1_rows = []
study_countries_policy = sorted(df["iso3"].unique())

for iso3 in study_countries_policy:
    country_df = df[df["iso3"] == iso3]
    
    for year in STUDY_YEARS:
        # Iniciativas vigentes: startYear <= year AND (endYear >= year OR endYear is NaN)
        active = country_df[
            (country_df["start_year"] <= year) &
            ((country_df["end_year"] >= year) | country_df["end_year"].isna())
        ]
        
        # 1. has_ai_law: ¿hay al menos una iniciativa Binding vigente?
        binding_active = active[active["extent_binding"] == "Binding"]
        has_ai_law = 1 if len(binding_active) > 0 else 0
        
        # 2. regulatory_approach: categorización del enfoque
        if len(active) > 0:
            regulatory_approach = classify_approach(active)
        else:
            regulatory_approach = "none"
        
        # 3. year_enacted: año de la primera regulación Binding del país (acumulativo)
        all_binding = country_df[country_df["extent_binding"] == "Binding"]
        if len(all_binding) > 0:
            first_binding_year = all_binding["start_year"].min()
            year_enacted = first_binding_year if first_binding_year <= year else None
        else:
            year_enacted = None
        
        # 4. regulatory_intensity: n° binding vigentes + n° tipos de regulación distintos
        n_binding = len(binding_active)
        n_regulation_types = active["initiative_type"].nunique() if len(active) > 0 else 0
        regulatory_intensity = n_binding + n_regulation_types
        
        # 5. enforcement_level: escala basada en governance bodies + binding count
        has_governance = any("governance" in str(c).lower() for c in active["category"].tolist())
        if has_governance and n_binding >= 3:
            enforcement_level = "high"
        elif has_governance or n_binding >= 2:
            enforcement_level = "medium"
        elif n_binding >= 1:
            enforcement_level = "low"
        else:
            enforcement_level = "none"
        
        # 6. thematic_coverage: n° de tags distintos de iniciativas vigentes
        all_tags = set()
        for tags_str in active["tags"].dropna():
            if tags_str:
                all_tags.update(t.strip() for t in tags_str.split("|") if t.strip())
        thematic_coverage = len(all_tags)
        
        # ── Variables adicionales derivadas (complementarias) ─────────
        n_total_active = len(active)
        n_nonbinding = len(active[active["extent_binding"] == "Non-binding"])
        n_strategies = len(active[active["category"].str.contains("Strategy", case=False, na=False)])
        n_regulations = len(active[active["category"].str.contains("Regulation|standard", case=False, na=False)])
        n_sectors = active["sectors"].str.split("|").explode().nunique() if len(active) > 0 else 0
        n_policy_instruments = active["policy_instruments"].str.split("|").explode().nunique() if len(active) > 0 else 0
        
        x1_rows.append({
            "iso3": iso3,
            "year": year,
            # ── 6 variables X1 core ──
            "has_ai_law": has_ai_law,
            "regulatory_approach": regulatory_approach,
            "year_enacted": year_enacted,
            "regulatory_intensity": regulatory_intensity,
            "enforcement_level": enforcement_level,
            "thematic_coverage": thematic_coverage,
            # ── Variables complementarias ──
            "n_total_initiatives": n_total_active,
            "n_binding": n_binding,
            "n_nonbinding": n_nonbinding,
            "n_strategies": n_strategies,
            "n_regulations": n_regulations,
            "n_sectors_covered": n_sectors,
            "n_policy_instruments": n_policy_instruments,
        })

x1_df = pd.DataFrame(x1_rows)

# ── Resumen ───────────────────────────────────────────────────────────────
print(f"Panel X1 construido: {x1_df.shape}")
print(f"  Países: {x1_df['iso3'].nunique()}")
print(f"  Periodo: {x1_df['year'].min()}-{x1_df['year'].max()}")
print(f"\nDistribución has_ai_law (todos los años):")
print(x1_df["has_ai_law"].value_counts().to_string())
print(f"\nDistribución regulatory_approach (2024):")
print(x1_df[x1_df["year"] == 2024]["regulatory_approach"].value_counts().to_string())
print(f"\nDistribución enforcement_level (2024):")
print(x1_df[x1_df["year"] == 2024]["enforcement_level"].value_counts().to_string())
print(f"\nEstadísticas descriptivas variables numéricas (2024):")
cols_num = ["has_ai_law", "regulatory_intensity", "thematic_coverage",
            "n_total_initiatives", "n_binding", "n_nonbinding", "n_regulations"]
print(x1_df[x1_df["year"] == 2024][cols_num].describe().round(2).to_string())

Panel X1 construido: (816, 15)
  Países: 68
  Periodo: 2013-2024

Distribución has_ai_law (todos los años):
has_ai_law
0    602
1    214

Distribución regulatory_approach (2024):
regulatory_approach
light_touch           24
strategy_led          17
regulation_focused    16
comprehensive          9
none                   2

Distribución enforcement_level (2024):
enforcement_level
medium    34
none      18
low        9
high       7

Estadísticas descriptivas variables numéricas (2024):
       has_ai_law  regulatory_intensity  thematic_coverage  n_total_initiatives  n_binding  n_nonbinding  n_regulations
count        68.0                 68.00              68.00                68.00      68.00         68.00          68.00
mean          0.5                 10.93              12.63                17.06       1.25         13.82           4.12
std           0.5                  7.44               8.78                14.24       2.37         11.95           5.57
min           0.0              

### 4c. Guardado de variables X1 y verificación cruzada

Se guardan las variables X1 derivadas en formato panel (país × año) y se cruza con las variables Y y X2 existentes para evaluar la completitud del dataset integrado.

In [13]:
# ══════════════════════════════════════════════════════════════════════════
# 4c. Guardado de X1 y verificación cruzada
# ══════════════════════════════════════════════════════════════════════════

# ── Guardar X1 panel ──────────────────────────────────────────────────────
x1_path = POLICY_PATH / "oecd_x1_policy_variables.csv"
x1_df.to_csv(x1_path, index=False)
print(f"Variables X1 guardadas: {x1_path.name} ({x1_path.stat().st_size/1024:.1f} KB)")

# ── Guardar solo las 6 variables X1 core en formato limpio ────────────────
x1_core_cols = ["iso3", "year", "has_ai_law", "regulatory_approach",
                "year_enacted", "regulatory_intensity", "enforcement_level",
                "thematic_coverage"]
x1_core_df = x1_df[x1_core_cols].copy()
x1_core_path = POLICY_PATH / "oecd_x1_core.csv"
x1_core_df.to_csv(x1_core_path, index=False)
print(f"Variables X1 core: {x1_core_path.name} ({x1_core_path.stat().st_size/1024:.1f} KB)")

# ── Snapshot 2024: tabla resumen por país ─────────────────────────────────
snapshot_2024 = x1_df[x1_df["year"] == 2024].sort_values("iso3").reset_index(drop=True)
snapshot_path = POLICY_PATH / "oecd_x1_snapshot_2024.csv"
snapshot_2024.to_csv(snapshot_path, index=False)
print(f"Snapshot 2024: {snapshot_path.name}")

# ── Verificación cruzada con cobertura OECD existente ─────────────────────
print("\n" + "=" * 70)
print("VERIFICACIÓN CRUZADA DE COBERTURA")
print("=" * 70)

# Países del estudio
all_study_set = set(study_countries_63 + study_solo_y)
x1_countries = set(x1_df["iso3"].unique())
oecd_quant_countries = set(oecd_long_df["iso3"].unique()) if "oecd_long_df" in dir() else set()

# Cruce
has_x1 = all_study_set & x1_countries
has_oecd_quant = all_study_set & oecd_quant_countries
has_both = has_x1 & has_oecd_quant
has_x1_only = has_x1 - has_oecd_quant
has_quant_only = has_oecd_quant - has_x1
has_neither = all_study_set - has_x1 - has_oecd_quant

print(f"\nPaíses del estudio:            {len(all_study_set)}")
print(f"  Con X1 (policy):             {len(has_x1)}")
print(f"  Con OECD cuantitativo:       {len(has_oecd_quant)}")
print(f"  Con AMBOS (X1 + OECD quant): {len(has_both)}")
print(f"  Solo X1 (sin OECD quant):    {len(has_x1_only)} → {sorted(has_x1_only)}")
print(f"  Solo OECD quant (sin X1):    {len(has_quant_only)} → {sorted(has_quant_only)}")
print(f"  Sin ningún dato OECD:        {len(has_neither)} → {sorted(has_neither)}")

# ── Top 10 países por regulatory_intensity 2024 ──────────────────────────
print(f"\nTop 15 países por regulatory_intensity (2024):")
top15 = snapshot_2024.nlargest(15, "regulatory_intensity")[
    ["iso3", "has_ai_law", "regulatory_approach", "regulatory_intensity",
     "enforcement_level", "thematic_coverage", "n_total_initiatives"]
]
print(top15.to_string(index=False))

# ── Archivos generados ───────────────────────────────────────────────────
print(f"\nArchivos en {POLICY_PATH}:")
for f in sorted(POLICY_PATH.glob("*policy*")) :
    print(f"  {f.name:55s} {f.stat().st_size/1024:8.1f} KB")

Variables X1 guardadas: oecd_x1_policy_variables.csv (40.0 KB)
Variables X1 core: oecd_x1_core.csv (28.4 KB)
Snapshot 2024: oecd_x1_snapshot_2024.csv

VERIFICACIÓN CRUZADA DE COBERTURA

Países del estudio:            86
  Con X1 (policy):             68
  Con OECD cuantitativo:       77
  Con AMBOS (X1 + OECD quant): 65
  Solo X1 (sin OECD quant):    3 → ['MUS', 'SRB', 'UGA']
  Solo OECD quant (sin X1):    12 → ['BLR', 'CHN', 'JOR', 'LBN', 'LKA', 'MNG', 'PAK', 'PAN', 'PHL', 'RUS', 'SYC', 'TWN']
  Sin ningún dato OECD:        6 → ['BGD', 'BHR', 'BLZ', 'BRB', 'CMR', 'GHA']

Top 15 países por regulatory_intensity (2024):
iso3  has_ai_law regulatory_approach  regulatory_intensity enforcement_level  thematic_coverage  n_total_initiatives
 SGP           1  regulation_focused                    31            medium                 20                   55
 USA           1  regulation_focused                    30              high                  2                   55
 GBR           1  regul

### 4d. Extracción adicional: OECD Country API (datos complementarios por país)

Complementamos los datos de políticas con la API de países del mismo sistema, que contiene perfiles resumen con métricas adicionales sobre el ecosistema de IA por país (inversión, publicaciones, talento, etc.).

In [14]:
# ══════════════════════════════════════════════════════════════════════════
# 4d. Extracción de perfiles de país desde API EC-OECD
# ══════════════════════════════════════════════════════════════════════════
# La misma API tiene un endpoint /countries/ con datos resumen por país.
# URL: https://oecd-ai.case-api.buddyweb.fr/countries

COUNTRY_API = "https://oecd-ai.case-api.buddyweb.fr/countries"

# ── Paso 1: obtener lista de todos los países ────────────────────────────
print("Descargando lista de países desde API EC-OECD...")
r = requests.get(COUNTRY_API, timeout=30)
r.raise_for_status()
countries_data = r.json()

# Puede ser lista directa o dict con "data"
if isinstance(countries_data, dict) and "data" in countries_data:
    country_list = countries_data["data"]
elif isinstance(countries_data, list):
    country_list = countries_data
else:
    country_list = []
    print(f"  Formato inesperado: {type(countries_data)}, keys={countries_data.keys() if isinstance(countries_data, dict) else 'N/A'}")

print(f"  Países en la API: {len(country_list)}")

# ── Paso 2: extraer datos detallados por país ────────────────────────────
# Usar el endpoint /countries/s/{slug} para cada país del estudio
country_profiles = []
slugs_by_iso3 = {}

# Crear mapeo inverso ISO3 → slug
ISO3_TO_SLUG = {}
for item in country_list:
    if isinstance(item, dict):
        name = item.get("name", "")
        slug = item.get("slug", "")
        iso3 = POLICY_NAME_TO_ISO3.get(name, "")
        if iso3 and slug:
            ISO3_TO_SLUG[iso3] = slug

print(f"  Países del estudio con slug: {len(set(ISO3_TO_SLUG.keys()) & set(study_countries_63 + study_solo_y))}")

# Descargar perfiles detallados
print("\nDescargando perfiles detallados de países...")
country_detail_rows = []
errors_country = []

for iso3 in sorted(set(study_countries_63 + study_solo_y)):
    slug = ISO3_TO_SLUG.get(iso3)
    if not slug:
        continue
    
    try:
        url = f"{COUNTRY_API}/s/{slug}"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        cdata = r.json()
        
        # Extraer campos disponibles
        if isinstance(cdata, dict):
            row = {"iso3": iso3, "slug": slug}
            # Campos directos
            for field in ["name", "region", "incomeGroup", "hasNationalStrategy",
                         "hasNationalStrategyYear", "gdp", "population",
                         "aiRelatedPublications", "aiRelatedPatents",
                         "aiStartups", "aiTalentConcentration",
                         "totalPolicyInitiatives", "bindingPolicyInitiatives"]:
                row[field] = cdata.get(field)
            
            # Métricas anidadas si existen
            for key in ["metrics", "indicators", "stats", "data"]:
                nested = cdata.get(key)
                if isinstance(nested, dict):
                    for k, v in nested.items():
                        if isinstance(v, (int, float, str)):
                            row[f"{key}_{k}"] = v
            
            country_detail_rows.append(row)
        
        time.sleep(0.15)
    except Exception as e:
        errors_country.append(f"{iso3} ({slug}): {e}")

print(f"  Perfiles descargados: {len(country_detail_rows)}")
if errors_country:
    print(f"  Errores: {len(errors_country)}")
    for e in errors_country[:5]:
        print(f"    {e}")

# ── Guardar perfiles ─────────────────────────────────────────────────────
if country_detail_rows:
    country_profiles_df = pd.DataFrame(country_detail_rows)
    # Eliminar columnas que son todas NaN
    country_profiles_df = country_profiles_df.dropna(axis=1, how="all")
    
    profile_path = POLICY_PATH / "oecd_ai_country_profiles.csv"
    country_profiles_df.to_csv(profile_path, index=False)
    print(f"\nPerfiles guardados: {profile_path.name} ({profile_path.stat().st_size/1024:.1f} KB)")
    print(f"  Columnas: {list(country_profiles_df.columns)}")
    print(f"\nPrimeras filas:")
    display(country_profiles_df.head(10))
else:
    print("  No se obtuvieron perfiles de país.")

Descargando lista de países desde API EC-OECD...
  Países en la API: 20
  Países del estudio con slug: 10

Descargando perfiles detallados de países...
  Perfiles descargados: 10

Perfiles guardados: oecd_ai_country_profiles.csv (0.6 KB)
  Columnas: ['iso3', 'slug', 'name', 'region', 'incomeGroup', 'gdp']

Primeras filas:


,iso3,slug,name,region,incomeGroup,gdp
0,ARG,argentina,Argentina,Buckinghamshire,High income,894729382
1,ARM,armenia,Armenia,Avon,Low income,79293904
2,AUS,australia,Australia,Bedfordshire,Lower middle income,758315741
3,AUT,austria,Austria,Berkshire,Upper middle income,698770746
4,BEL,belgium,Belgium,Buckinghamshire,Upper middle income,635478342
5,BGD,bangladesh,Bangladesh,Buckinghamshire,Low income,622599631
6,BHR,bahrain,Bahrain,Borders,High income,108601586
7,BLR,belarus,Belarus,Bedfordshire,Upper middle income,197653708
8,BLZ,belize,Belize,Buckinghamshire,Lower middle income,84194257
9,BRB,barbados,Barbados,Borders,High income,649953324


### 4e. Consolidación final OECD — Resumen de todas las extracciones

Resumen de todo lo extraído desde OECD: indicadores cuantitativos (STI + MSTI), catálogo de visualizaciones, políticas de IA, variables X1 derivadas, y perfiles de país.

In [15]:
# ══════════════════════════════════════════════════════════════════════════
# 4e. Consolidación final — Inventario completo de datos OECD
# ══════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("INVENTARIO COMPLETO DE DATOS OECD EXTRAÍDOS")
print("=" * 70)

# ── Archivos generados ───────────────────────────────────────────────────
print("\n📁 Archivos en data/raw/OECD/:")
total_kb = 0
for f in sorted(POLICY_PATH.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    total_kb += size_kb
    print(f"  {f.name:55s} {size_kb:8.1f} KB")
print(f"  {'TOTAL':55s} {total_kb:8.1f} KB")

# ── Resumen por tipo de variable ──────────────────────────────────────────
all_study_set = set(study_countries_63 + study_solo_y)

print(f"\n📊 COBERTURA POR TIPO DE VARIABLE (de {len(all_study_set)} países del estudio):")

# Y / Y_proxy (de STI Scoreboard)
y_indicators = [m["canonical"] for m in STI_SCOREBOARD_INDICATORS.values() 
                if m["role"] in ("Y", "Y_proxy")]
y_countries = set(oecd_long_df[oecd_long_df["indicator"].isin(y_indicators)]["iso3"].unique()) & all_study_set
print(f"  Y / Y_proxy (STI):     {len(y_indicators)} indicadores, {len(y_countries)} países")

# X2 (de MSTI + STI)
x2_indicators = ([m["canonical"] for m in MSTI_MEASURES.values()] + 
                 [m["canonical"] for m in STI_SCOREBOARD_INDICATORS.values() if m["role"] == "X2"])
x2_countries = set(oecd_long_df[oecd_long_df["indicator"].isin(x2_indicators)]["iso3"].unique()) & all_study_set
print(f"  X2 control (MSTI+STI): {len(x2_indicators)} indicadores, {len(x2_countries)} países")

# X1 (de Policy API)
x1_countries = set(x1_df["iso3"].unique()) & all_study_set
print(f"  X1 regulación (Policy):{len([c for c in x1_core_cols if c not in ('iso3','year')])} variables,  {len(x1_countries)} países")

# Extra
extra_indicators = [m["canonical"] for m in STI_SCOREBOARD_INDICATORS.values() if m["role"] == "extra"]
extra_countries = set(oecd_long_df[oecd_long_df["indicator"].isin(extra_indicators)]["iso3"].unique()) & all_study_set
print(f"  Extra (STI):           {len(extra_indicators)} indicadores, {len(extra_countries)} países")

# ── Cobertura global del estudio ──────────────────────────────────────────
print(f"\n🔍 ESTADO ACTUAL DEL ESTUDIO (después de OECD):")
all_oecd = y_countries | x2_countries | x1_countries | extra_countries
print(f"  Países con al menos 1 dato OECD: {len(all_oecd)}/86")
print(f"  Países sin datos OECD:           {len(all_study_set - all_oecd)} → {sorted(all_study_set - all_oecd)}")

# Estado X1 (la brecha más crítica)
x1_2024 = x1_df[x1_df["year"] == 2024]
x1_with_law = x1_2024[x1_2024["has_ai_law"] == 1]
print(f"\n  X1 status (2024):")
print(f"    Países con has_ai_law=1:     {len(x1_with_law)}/{len(x1_2024)}")
print(f"    Promedio regulatory_intensity: {x1_2024['regulatory_intensity'].mean():.1f}")
print(f"    Promedio thematic_coverage:    {x1_2024['thematic_coverage'].mean():.1f}")

INVENTARIO COMPLETO DE DATOS OECD EXTRAÍDOS

📁 Archivos en data/raw/OECD/:
  oecd_ai_country_profiles.csv                                 0.6 KB
  oecd_ai_policy_initiatives_all.csv                        1365.2 KB
  oecd_ai_policy_initiatives_study.csv                      1284.9 KB
  oecd_ai_visualizations_catalog.csv                          27.4 KB
  oecd_all_indicators_long.csv                               772.5 KB
  oecd_all_indicators_wide.csv                               170.0 KB
  oecd_msti.csv                                               30.6 KB
  oecd_sti_scoreboard.csv                                    140.4 KB
  oecd_x1_core.csv                                            28.4 KB
  oecd_x1_policy_variables.csv                                40.0 KB
  oecd_x1_snapshot_2024.csv                                    3.9 KB
  TOTAL                                                     3864.1 KB

📊 COBERTURA POR TIPO DE VARIABLE (de 86 países del estudio):
  Y / Y_proxy (STI):   

## Fuente 5: Microsoft AI Diffusion Report (AIEI)

Extraemos el **AI User Share** por país — la variable `ai_adoption_rate` del estudio — del portal del *AI Economy Institute (AIEI)* de Microsoft Research.

**Hallazgos del descubrimiento (Fase 1):**
- No hay API pública ni CSV descargable.
- Los datos están embebidos directamente como tablas HTML en la página web (WordPress).
- La tabla cubre **147 países** con 2 puntos temporales: H1 2025 y H2 2025.
- Los índices compuestos (Frontier, Infrastructure) están solo en gráficos del PDF — no son extraíbles de forma tabular.

**Variable extraída:**

| canonical_name | tipo | descripción | cobertura |
|---|---|---|---|
| `ai_adoption_rate` | Y target core | % población en edad laboral usando IA generativa activamente | 147 países, H1+H2 2025 |

**Nota metodológica:** Solo existen 2 puntos temporales (H1/H2 2025). Se tratará como variable cross-sectional de 2025 en el panel. Para el modelamiento se usará H2 2025 (más reciente).

In [5]:
# ══════════════════════════════════════════════════════════════════════════
# 5.0  Setup — Imports, paths y mapeo de nombres de país
# ══════════════════════════════════════════════════════════════════════════
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from pathlib import Path

MS_PATH = Path("../data/raw/Microsoft/")
MS_PATH.mkdir(parents=True, exist_ok=True)

AIEI_URL = "https://www.microsoft.com/en-us/research/group/aiei/ai-diffusion/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

# ── Mapeo de nombres Microsoft → ISO3 ─────────────────────────────────────
# Se prioriza ISO 3166-1 alpha-3 para maximizar trazabilidad global.
MS_NAME_TO_ISO3 = {
    # G7 / OECD principales
    "United States": "USA", "United Kingdom": "GBR", "Germany": "DEU",
    "France": "FRA", "Italy": "ITA", "Japan": "JPN", "Canada": "CAN",
    "Australia": "AUS", "Spain": "ESP", "Netherlands": "NLD",
    "Sweden": "SWE", "Norway": "NOR", "Denmark": "DNK", "Finland": "FIN",
    "Switzerland": "CHE", "Austria": "AUT", "Belgium": "BEL",
    "Ireland": "IRL", "Portugal": "PRT", "Greece": "GRC",
    "Poland": "POL", "Czech Republic": "CZE", "Czechia": "CZE", "Hungary": "HUN",
    "Romania": "ROU", "Slovakia": "SVK", "Slovenia": "SVN",
    "Croatia": "HRV", "Bulgaria": "BGR", "Estonia": "EST",
    "Latvia": "LVA", "Lithuania": "LTU", "Luxembourg": "LUX",
    "Iceland": "ISL", "Cyprus": "CYP", "Malta": "MLT",
    "New Zealand": "NZL", "Israel": "ISR",
    # América Latina y Caribe
    "Brazil": "BRA", "Mexico": "MEX", "Argentina": "ARG",
    "Chile": "CHL", "Colombia": "COL", "Peru": "PER",
    "Ecuador": "ECU", "Uruguay": "URY", "Costa Rica": "CRI",
    "Panama": "PAN", "Honduras": "HND", "Guatemala": "GTM",
    "Bolivia": "BOL", "Paraguay": "PRY", "El Salvador": "SLV",
    "Nicaragua": "NIC", "Venezuela": "VEN", "Cuba": "CUB",
    "Dominican Republic": "DOM", "Jamaica": "JAM", "Trinidad and Tobago": "TTO",
    "Barbados": "BRB", "Belize": "BLZ", "French Guiana": "GUF",
    "Guyana": "GUY", "Suriname": "SUR", "Haiti": "HTI",
    # Asia-Pacífico
    "China": "CHN", "India": "IND", "South Korea": "KOR",
    "Singapore": "SGP", "Indonesia": "IDN", "Malaysia": "MYS",
    "Thailand": "THA", "Vietnam": "VNM", "Philippines": "PHL",
    "Bangladesh": "BGD", "Pakistan": "PAK", "Sri Lanka": "LKA",
    "Myanmar": "MMR", "Cambodia": "KHM", "Mongolia": "MNG",
    "Kazakhstan": "KAZ", "Uzbekistan": "UZB", "Taiwan": "TWN",
    "Hong Kong": "HKG", "Macao": "MAC", "Nepal": "NPL",
    "Papua New Guinea": "PNG", "Fiji": "FJI", "Laos": "LAO",
    # Oriente Medio / África del Norte
    "United Arab Emirates": "ARE", "Saudi Arabia": "SAU",
    "Turkey": "TUR", "Türkiye": "TUR", "Egypt": "EGY",
    "Morocco": "MAR", "Tunisia": "TUN", "Algeria": "DZA",
    "Jordan": "JOR", "Lebanon": "LBN", "Kuwait": "KWT",
    "Qatar": "QAT", "Bahrain": "BHR", "Oman": "OMN",
    "Iraq": "IRQ", "Iran": "IRN", "Syria": "SYR", "Libya": "LBY",
    "Sudan": "SDN",
    # África Sub-Sahariana
    "South Africa": "ZAF", "Nigeria": "NGA", "Kenya": "KEN",
    "Ghana": "GHA", "Ethiopia": "ETH", "Tanzania": "TZA",
    "Uganda": "UGA", "Cameroon": "CMR", "Senegal": "SEN",
    "Côte d'Ivoire": "CIV", "Cote D'Ivoire": "CIV", "Cote D’Ivoire": "CIV",
    "Ivory Coast": "CIV", "Zimbabwe": "ZWE", "Zambia": "ZMB",
    "Mozambique": "MOZ", "Angola": "AGO", "Mauritius": "MUS",
    "Rwanda": "RWA", "Botswana": "BWA", "Namibia": "NAM",
    "Seychelles": "SYC", "Gabon": "GAB", "Gambia": "GMB",
    "Madagascar": "MDG", "Malawi": "MWI", "Benin": "BEN",
    "Burkina Faso": "BFA", "Guinea": "GIN", "Guinea-Bissau": "GNB",
    "Liberia": "LBR", "Mali": "MLI", "Mauritania": "MRT",
    "Niger": "NER", "Sierra Leone": "SLE", "Togo": "TGO",
    "Lesotho": "LSO", "Central African Republic": "CAF", "Chad": "TCD",
    "Congo": "COG", "Congo (DRC)": "COD", "Burundi": "BDI",
    "Eritrea": "ERI", "Somalia": "SOM", "South Sudan": "SSD",
    # Europa del Este / Asia Central
    "Russia": "RUS", "Ukraine": "UKR", "Belarus": "BLR",
    "Armenia": "ARM", "Azerbaijan": "AZE", "Georgia": "GEO",
    "Moldova": "MDA", "Serbia": "SRB",
    "Bosnia and Herzegovina": "BIH", "Bosnia And Herzegovina": "BIH",
    "North Macedonia": "MKD", "Albania": "ALB", "Kosovo": "XKX",
    "Montenegro": "MNE", "Tajikistan": "TJK", "Turkmenistan": "TKM",
    "Kyrgyzstan": "KGZ", "Afghanistan": "AFG",
}

# ── Muestra del estudio: fallback para reejecución aislada de la fuente ────
if "study_countries_63" not in globals() or "study_solo_y" not in globals():
    study_countries_63 = [
        "ARE", "ARG", "AUS", "AUT", "BEL", "BGR", "BRA", "CAN", "CHE", "CHL", "CHN", "CMR",
        "COL", "CRI", "CYP", "CZE", "DEU", "DNK", "EGY", "ESP", "EST", "FIN", "FRA", "GBR",
        "GRC", "HRV", "HUN", "IDN", "IND", "IRL", "ISL", "ISR", "ITA", "JOR", "JPN", "KAZ",
        "KOR", "LTU", "LUX", "LVA", "MEX", "MYS", "NGA", "NLD", "NOR", "NZL", "PER", "PHL",
        "POL", "PRT", "ROU", "RUS", "SAU", "SGP", "SVN", "SWE", "THA", "TUN", "TUR", "URY",
        "USA", "VNM", "ZAF",
    ]
    study_solo_y = [
        "ARM", "BGD", "BHR", "BLR", "BLZ", "BRB", "ECU", "GHA", "KEN", "LBN", "LKA", "MAR",
        "MLT", "MNG", "MUS", "PAK", "PAN", "SRB", "SVK", "SYC", "TWN", "UGA", "UKR",
    ]

study_iso3_all = set(study_countries_63 + study_solo_y)

print(f"Ruta de salida: {MS_PATH.resolve()}")
print(f"URL fuente: {AIEI_URL}")
print(f"Mapeo de nombres: {len(MS_NAME_TO_ISO3)} entradas")
print(f"Países de la muestra disponibles: {len(study_iso3_all)}")

Ruta de salida: /Users/francoia/Documents/MIA/Proyecto Data-Science/research/data/raw/Microsoft
URL fuente: https://www.microsoft.com/en-us/research/group/aiei/ai-diffusion/
Mapeo de nombres: 171 entradas
Países de la muestra disponibles: 86


### 5a. Scraping de la tabla HTML — AI User Share por país (H1 + H2 2025)

In [6]:
# ══════════════════════════════════════════════════════════════════════════
# 5a. Scraping de la tabla HTML del portal AIEI
# ══════════════════════════════════════════════════════════════════════════
# La página de Microsoft Research (WordPress) embebe las dos tablas con datos
# de AI User Share directamente en el HTML — no requiere JavaScript rendering.

print(f"Fetching: {AIEI_URL}")
response = requests.get(AIEI_URL, headers=HEADERS, timeout=30)
response.raise_for_status()
print(f"  Status: {response.status_code}, Size: {len(response.text):,} chars")

soup = BeautifulSoup(response.text, "html.parser")

# ── Extraer todas las tablas HTML ─────────────────────────────────────────
tables = soup.find_all("table")
print(f"  Tablas encontradas: {len(tables)}")

raw_rows = []
for t_idx, table in enumerate(tables):
    rows = table.find_all("tr")
    first_row = [cell.get_text(strip=True).replace("\xa0", " ")
                 for cell in rows[0].find_all(["td", "th"])]
    print(f"  Tabla {t_idx}: {len(rows)} filas, header: {first_row}")

    for row_idx, row in enumerate(rows):
        if row_idx == 0:
            continue
        cells = [cell.get_text(strip=True).replace("\xa0", " ")
                 for cell in row.find_all(["td", "th"])]
        if cells and cells[0]:
            raw_rows.append(cells)

print(f"\n  Total filas de datos extraídas: {len(raw_rows)}")

# ── Parsear y limpiar ─────────────────────────────────────────────────────
def parse_pct(val):
    """Convierte strings como '59.4%' o '+4.5%' a float."""
    if not val or val == "N/A":
        return None
    try:
        return float(val.replace("%", "").replace("+", "").strip())
    except ValueError:
        return None

ms_rows = []
for row in raw_rows:
    if len(row) >= 3:
        country_name = row[0].strip()
        ms_rows.append({
            "country_name_ms": country_name,
            "iso3": MS_NAME_TO_ISO3.get(country_name, ""),
            "ai_user_share_h1_2025": parse_pct(row[1]) if len(row) > 1 else None,
            "ai_user_share_h2_2025": parse_pct(row[2]) if len(row) > 2 else None,
            "ai_user_share_change_pp": parse_pct(row[3]) if len(row) > 3 else None,
            "source": "Microsoft_AIEI",
            "report_edition": "H1_2025+H2_2025",
            "report_url": AIEI_URL,
        })

ms_raw_df = pd.DataFrame(ms_rows)

# ── Diagnóstico de mapeo ──────────────────────────────────────────────────
unmapped = ms_raw_df[ms_raw_df["iso3"] == ""]["country_name_ms"].tolist()
mapped = ms_raw_df[ms_raw_df["iso3"] != ""]

print(f"\nDataset Microsoft AI Diffusion: {ms_raw_df.shape}")
print(f"  Con ISO3 mapeado: {len(mapped)}")
print(f"  Sin ISO3 ({len(unmapped)} países): {unmapped}")
print(f"\nRango ai_user_share_h2_2025: {ms_raw_df['ai_user_share_h2_2025'].min():.1f}% - {ms_raw_df['ai_user_share_h2_2025'].max():.1f}%")
print(f"\nTop 10:")
print(ms_raw_df.nlargest(10, "ai_user_share_h2_2025")[["country_name_ms", "ai_user_share_h2_2025", "ai_user_share_h1_2025"]].to_string(index=False))

Fetching: https://www.microsoft.com/en-us/research/group/aiei/ai-diffusion/
  Status: 200, Size: 227,807 chars
  Tablas encontradas: 2
  Tabla 0: 75 filas, header: ['Economy', 'H1 2025 AI Diffusion', 'H2 2025 AI Diffusion', 'Change']
  Tabla 1: 74 filas, header: ['Economy', 'H1 2025 AI Diffusion', 'H2 2025 AI Diffusion', 'Change']

  Total filas de datos extraídas: 147

Dataset Microsoft AI Diffusion: (147, 8)
  Con ISO3 mapeado: 147
  Sin ISO3 (0 países): []

Rango ai_user_share_h2_2025: 5.1% - 64.0%

Top 10:
     country_name_ms  ai_user_share_h2_2025  ai_user_share_h1_2025
United Arab Emirates                   64.0                   59.4
           Singapore                   60.9                   58.6
              Norway                   46.4                   45.3
             Ireland                   44.6                   41.7
              France                   44.0                   40.9
               Spain                   41.8                   39.7
         New Ze

### 5b. Guardado del dataset crudo y construcción del panel país-año

Se guarda el dataset crudo completo y se construye el formato panel compatible con el resto del estudio (`iso3 × year`). Como solo existen 2 puntos temporales (H1 y H2 de 2025), se crea un registro por período y adicionalmente un registro de año 2025 usando H2 como valor principal.

In [7]:
# ══════════════════════════════════════════════════════════════════════════
# 5b. Guardado raw + construcción panel país-año
# ══════════════════════════════════════════════════════════════════════════

# ── 1. Guardar dataset crudo completo (147 países) ────────────────────────
raw_path = MS_PATH / "microsoft_ai_diffusion_raw.csv"
ms_raw_df.to_csv(raw_path, index=False)
print(f"Raw guardado: {raw_path.name} ({raw_path.stat().st_size/1024:.1f} KB, {len(ms_raw_df)} países)")

# ── 2. Filtrar a los 86 países del estudio ────────────────────────────────
study_iso3_all = set(study_countries_63 + study_solo_y)
ms_study_df = ms_raw_df[ms_raw_df["iso3"].isin(study_iso3_all)].copy()

study_path = MS_PATH / "microsoft_ai_diffusion_study.csv"
ms_study_df.to_csv(study_path, index=False)
print(f"Estudio guardado: {study_path.name} ({len(ms_study_df)} países)")

# ── 3. Construir formato panel (iso3 × year × period) ────────────────────
# H1 2025 → year=2025, period="H1"
# H2 2025 → year=2025, period="H2"
panel_rows = []
for _, row in ms_raw_df.iterrows():
    if not row["iso3"]:
        continue
    # H1 2025
    if row["ai_user_share_h1_2025"] is not None:
        panel_rows.append({
            "iso3": row["iso3"],
            "country_name_ms": row["country_name_ms"],
            "year": 2025,
            "period": "H1",
            "ai_adoption_rate": row["ai_user_share_h1_2025"],
            "source": "Microsoft_AIEI",
        })
    # H2 2025
    if row["ai_user_share_h2_2025"] is not None:
        panel_rows.append({
            "iso3": row["iso3"],
            "country_name_ms": row["country_name_ms"],
            "year": 2025,
            "period": "H2",
            "ai_adoption_rate": row["ai_user_share_h2_2025"],
            "source": "Microsoft_AIEI",
        })

ms_panel_df = pd.DataFrame(panel_rows)
panel_path = MS_PATH / "microsoft_ai_adoption_panel.csv"
ms_panel_df.to_csv(panel_path, index=False)
print(f"Panel guardado: {panel_path.name} ({len(ms_panel_df)} filas, {ms_panel_df['iso3'].nunique()} países)")

# ── 4. Snapshot canónico: una fila por país con H2 2025 como ai_adoption_rate
ms_h2_df = ms_raw_df[ms_raw_df["iso3"] != ""][["iso3", "country_name_ms",
                                                  "ai_user_share_h1_2025",
                                                  "ai_user_share_h2_2025",
                                                  "ai_user_share_change_pp"]].copy()
ms_h2_df = ms_h2_df.rename(columns={"ai_user_share_h2_2025": "ai_adoption_rate"})
ms_h2_df["year"] = 2025
ms_h2_df["period"] = "H2"

snapshot_path = MS_PATH / "microsoft_ai_diffusion_snapshot.csv"
ms_h2_df.to_csv(snapshot_path, index=False)
print(f"Snapshot H2 guardado: {snapshot_path.name} ({len(ms_h2_df)} países)")

# ── 5. Resumen ─────────────────────────────────────────────────────────────
print(f"\nCobertura del estudio:")
covered_study = set(ms_study_df["iso3"].unique())
missing_study = study_iso3_all - covered_study
print(f"  Países del estudio con ai_adoption_rate: {len(covered_study)}/86")
print(f"  Faltantes: {sorted(missing_study)}")
print(f"\nEstadísticas descriptivas (H2 2025, todos los países):")
print(ms_raw_df["ai_user_share_h2_2025"].describe().round(2).to_string())

Raw guardado: microsoft_ai_diffusion_raw.csv (17.8 KB, 147 países)
Estudio guardado: microsoft_ai_diffusion_study.csv (75 países)
Panel guardado: microsoft_ai_adoption_panel.csv (294 filas, 147 países)
Snapshot H2 guardado: microsoft_ai_diffusion_snapshot.csv (147 países)

Cobertura del estudio:
  Países del estudio con ai_adoption_rate: 75/86
  Faltantes: ['BHR', 'BLZ', 'BRB', 'CYP', 'EST', 'ISL', 'LUX', 'LVA', 'MLT', 'MUS', 'SYC']

Estadísticas descriptivas (H2 2025, todos los países):
count    147.00
mean      17.39
std       11.37
min        5.10
25%        9.05
50%       13.40
75%       23.60
max       64.00


### 5c. Verificación cruzada con la muestra del estudio

In [8]:
# ══════════════════════════════════════════════════════════════════════════
# 5c. Verificación cruzada con la muestra de 86 países
# ══════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("VERIFICACIÓN MICROSOFT AI DIFFUSION — MUESTRA DEL ESTUDIO")
print("=" * 70)

# ── Cruce con los 86 países ───────────────────────────────────────────────
study_ms = ms_h2_df[ms_h2_df["iso3"].isin(study_iso3_all)].sort_values(
    "ai_adoption_rate", ascending=False
).reset_index(drop=True)

study_ms_63 = study_ms[study_ms["iso3"].isin(study_countries_63)]
study_ms_solo_y = study_ms[study_ms["iso3"].isin(study_solo_y)]

print(f"\nCobertura de los 63 países ACTIVOS: {len(study_ms_63)}/63")
print(f"Cobertura de los 23 SOLO Y:          {len(study_ms_solo_y)}/23")
print(f"Total muestra con ai_adoption_rate:  {len(study_ms)}/86")

# ── Validación de valores conocidos ──────────────────────────────────────
print(f"\nValidación con valores conocidos del reporte:")
validation = {
    "ARE": 64.0, "SGP": 60.9, "NOR": 46.4,
    "IRL": 44.6, "FRA": 44.0, "USA": 28.3,
}
for iso3, expected in validation.items():
    actual = ms_h2_df[ms_h2_df["iso3"] == iso3]["ai_adoption_rate"]
    if len(actual) > 0:
        val = actual.values[0]
        status = "✓" if abs(val - expected) < 0.5 else "✗ DISCREPANCIA"
        print(f"  {iso3}: {val:.1f}% (esperado ~{expected:.1f}%) {status}")
    else:
        print(f"  {iso3}: NO ENCONTRADO")

# ── Tabla completa de los 86 países del estudio ───────────────────────────
print(f"\nTabla de ai_adoption_rate (H2 2025) — 86 países del estudio:")
print(f"{'ISO3':6s} {'País':30s} {'H1_2025':>8s} {'H2_2025':>8s} {'Δpp':>6s} {'Estado':8s}")
print("-" * 75)

for _, row in study_ms.iterrows():
    status = "ACTIVO" if row["iso3"] in study_countries_63 else "SOLO_Y"
    h1 = f"{row['ai_user_share_h1_2025']:.1f}%" if pd.notna(row.get('ai_user_share_h1_2025')) else "-"
    h2 = f"{row['ai_adoption_rate']:.1f}%"
    delta = f"{row['ai_user_share_change_pp']:+.1f}" if pd.notna(row.get('ai_user_share_change_pp')) else "-"
    print(f"{row['iso3']:6s} {row['country_name_ms']:30s} {h1:>8s} {h2:>8s} {delta:>6s} {status:8s}")

# ── Archivos generados ───────────────────────────────────────────────────
print(f"\nArchivos en {MS_PATH}:")
for f in sorted(MS_PATH.glob("*.csv")):
    print(f"  {f.name:55s} {f.stat().st_size/1024:8.1f} KB")

VERIFICACIÓN MICROSOFT AI DIFFUSION — MUESTRA DEL ESTUDIO

Cobertura de los 63 países ACTIVOS: 58/63
Cobertura de los 23 SOLO Y:          17/23
Total muestra con ai_adoption_rate:  75/86

Validación con valores conocidos del reporte:
  ARE: 64.0% (esperado ~64.0%) ✓
  SGP: 60.9% (esperado ~60.9%) ✓
  NOR: 46.4% (esperado ~46.4%) ✓
  IRL: 44.6% (esperado ~44.6%) ✓
  FRA: 44.0% (esperado ~44.0%) ✓
  USA: 28.3% (esperado ~28.3%) ✓

Tabla de ai_adoption_rate (H2 2025) — 86 países del estudio:
ISO3   País                            H1_2025  H2_2025    Δpp Estado  
---------------------------------------------------------------------------
ARE    United Arab Emirates              59.4%    64.0%   +4.5 ACTIVO  
SGP    Singapore                         58.6%    60.9%   +2.3 ACTIVO  
NOR    Norway                            45.3%    46.4%   +1.1 ACTIVO  
IRL    Ireland                           41.7%    44.6%   +2.9 ACTIVO  
FRA    France                            40.9%    44.0%   +3.1 ACTIVO 

## Fuente 6: Oxford Insights — Government AI Readiness Index

Oxford Insights es la fuente canonica de la variable Y `ai_readiness_score`. En esta implementacion se explotan los assets oficiales publicamente accesibles via WordPress media API y landing 2025.

**Cobertura estructurada confirmada hoy:**
- Datasets XLSX publicos: 2019, 2020, 2021, 2022, 2023, 2025.
- Reportes PDF publicos: 2019–2025.
- Metodologia PDF publica: 2025.
- Gap identificado: 2024 tiene PDF publico, pero no aparecio un XLSX o CSV publico discoverable via media API al momento de esta extraccion.

**Criterio de integracion:**
- `ai_readiness_score` se armoniza a escala 0–100 para el panel.
- Se preserva `ai_readiness_score_original` y el regimen metodologico por edicion para auditoria.
- Se guardan outputs raw, wide, long, snapshot y subconjuntos tematicos.

In [9]:
# ============================================================================
# 6.0 Setup - assets oficiales, paths y manifiesto de descarga
# ============================================================================
import subprocess
import unicodedata
import re
from pathlib import Path

import pandas as pd
import pycountry
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RAW_PATH = PROJECT_ROOT / "data" / "raw"

OXFORD_PATH = RAW_PATH / "Oxford Insights"
OXFORD_REPORTS_PATH = OXFORD_PATH / "reports"
OXFORD_DATASETS_PATH = OXFORD_PATH / "datasets"
OXFORD_METADATA_PATH = OXFORD_PATH / "metadata"

for output_path in (OXFORD_PATH, OXFORD_REPORTS_PATH, OXFORD_DATASETS_PATH, OXFORD_METADATA_PATH):
    output_path.mkdir(parents=True, exist_ok=True)

# Fallback para reejecucion aislada de la fuente.
if "study_countries_63" not in globals() or "study_solo_y" not in globals():
    study_countries_63 = [
        "ARE", "ARG", "AUS", "AUT", "BEL", "BGR", "BRA", "CAN", "CHE", "CHL", "CHN", "CMR",
        "COL", "CRI", "CYP", "CZE", "DEU", "DNK", "EGY", "ESP", "EST", "FIN", "FRA", "GBR",
        "GRC", "HRV", "HUN", "IDN", "IND", "IRL", "ISL", "ISR", "ITA", "JOR", "JPN", "KAZ",
        "KOR", "LTU", "LUX", "LVA", "MEX", "MYS", "NGA", "NLD", "NOR", "NZL", "PER", "PHL",
        "POL", "PRT", "ROU", "RUS", "SAU", "SGP", "SVN", "SWE", "THA", "TUN", "TUR", "URY",
        "USA", "VNM", "ZAF",
    ]
    study_solo_y = [
        "ARM", "BGD", "BHR", "BLR", "BLZ", "BRB", "ECU", "GHA", "KEN", "LBN", "LKA", "MAR",
        "MLT", "MNG", "MUS", "PAK", "PAN", "SRB", "SVK", "SYC", "TWN", "UGA", "UKR",
    ]

study_iso3_all = set(study_countries_63 + study_solo_y)

OXFORD_ASSETS = [
    {
        "year": 2019,
        "asset_kind": "report_pdf",
        "title": "2019 Government AI Readiness Index",
        "url": "https://oxfordinsights.com/wp-content/uploads/2023/12/ai-gov-readiness-report_v08.pdf",
        "relative_path": "reports/2019_government_ai_readiness_index.pdf",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2019,
        "asset_kind": "data_xlsx",
        "title": "2019 Government AI Readiness Index dataset",
        "url": "https://oxfordinsights.com/wp-content/uploads/2024/01/SHARED_-2019-Index-data-for-report.xlsx",
        "relative_path": "datasets/2019_government_ai_readiness_index_dataset.xlsx",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2020,
        "asset_kind": "report_pdf",
        "title": "2020 Government AI Readiness Index",
        "url": "https://oxfordinsights.com/wp-content/uploads/2023/11/AIReadinessReport.pdf",
        "relative_path": "reports/2020_government_ai_readiness_index.pdf",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2020,
        "asset_kind": "data_xlsx",
        "title": "2020 Government AI Readiness Index public dataset",
        "url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2020-Government-AI-Readiness-Index-public-dataset.xlsx",
        "relative_path": "datasets/2020_government_ai_readiness_index_public_dataset.xlsx",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2021,
        "asset_kind": "report_pdf",
        "title": "2021 Government AI Readiness Index",
        "url": "https://oxfordinsights.com/wp-content/uploads/2023/11/Government_AI_Readiness_21.pdf",
        "relative_path": "reports/2021_government_ai_readiness_index.pdf",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2021,
        "asset_kind": "data_xlsx",
        "title": "2021 Government AI Readiness Index public dataset",
        "url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2021-Government-AI-Readiness-Index-public-dataset.xlsx",
        "relative_path": "datasets/2021_government_ai_readiness_index_public_dataset.xlsx",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2022,
        "asset_kind": "report_pdf",
        "title": "2022 Government AI Readiness Index",
        "url": "https://oxfordinsights.com/wp-content/uploads/2023/11/Government_AI_Readiness_2022_FV.pdf",
        "relative_path": "reports/2022_government_ai_readiness_index.pdf",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2022,
        "asset_kind": "data_xlsx",
        "title": "2022 Government AI Readiness Index public data",
        "url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2022-Government-AI-Readiness-Index-public-data.xlsx",
        "relative_path": "datasets/2022_government_ai_readiness_index_public_data.xlsx",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2023,
        "asset_kind": "report_pdf",
        "title": "2023 Government AI Readiness Index",
        "url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2023-Government-AI-Readiness-Index.pdf",
        "relative_path": "reports/2023_government_ai_readiness_index.pdf",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2023,
        "asset_kind": "data_xlsx",
        "title": "2023 Government AI Readiness Index public dataset",
        "url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2023-AI-Readiness-Index-public-dataset.xlsx",
        "relative_path": "datasets/2023_government_ai_readiness_index_public_dataset.xlsx",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Structured public dataset discovered via WordPress media API.",
    },
    {
        "year": 2024,
        "asset_kind": "report_pdf",
        "title": "2024 Government AI Readiness Index",
        "url": "https://oxfordinsights.com/wp-content/uploads/2024/12/2024-Government-AI-Readiness-Index-1.pdf",
        "relative_path": "reports/2024_government_ai_readiness_index.pdf",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Report PDF publicamente accesible.",
    },
    {
        "year": 2024,
        "asset_kind": "data_xlsx",
        "title": "2024 Government AI Readiness Index data",
        "url": None,
        "relative_path": None,
        "available": False,
        "source_tier": "official_gap",
        "notes": "No se encontro XLSX o CSV publico discoverable via WordPress media API al 2026-04-08.",
    },
    {
        "year": 2025,
        "asset_kind": "report_pdf",
        "title": "2025 Government AI Readiness Index",
        "url": "https://oxfordinsights.com/wp-content/uploads/2025/12/2025-Government-AI-Readiness-Index.pdf",
        "relative_path": "reports/2025_government_ai_readiness_index.pdf",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Landing page 2025 + media API.",
    },
    {
        "year": 2025,
        "asset_kind": "methodology_pdf",
        "title": "Methodology Report 2025",
        "url": "https://oxfordinsights.com/wp-content/uploads/2026/01/Methodology-Report-2025.pdf",
        "relative_path": "reports/2025_methodology_report.pdf",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Media API publico.",
    },
    {
        "year": 2025,
        "asset_kind": "data_xlsx",
        "title": "2025 Government AI Readiness Index data",
        "url": "https://oxfordinsights.com/wp-content/uploads/2026/01/2025-Government-AI-Readiness-Index-data.xlsx",
        "relative_path": "datasets/2025_government_ai_readiness_index_data.xlsx",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Media API publico.",
    },
    {
        "year": 2025,
        "asset_kind": "metadata_html",
        "title": "2025 landing page snapshot",
        "url": "https://oxfordinsights.com/ai-readiness/government-ai-readiness-index-2025/",
        "relative_path": "metadata/2025_landing_page.html",
        "available": True,
        "source_tier": "official_primary",
        "notes": "HTML snapshot del landing interactivo 2025.",
    },
    {
        "year": 2025,
        "asset_kind": "metadata_json",
        "title": "WordPress media search - Government AI Readiness",
        "url": "https://oxfordinsights.com/wp-json/wp/v2/media?search=Government%20AI%20Readiness&per_page=100",
        "relative_path": "metadata/wp_media_search_government_ai_readiness.json",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Discovery trace para PDFs y datasets oficiales.",
    },
    {
        "year": 2025,
        "asset_kind": "metadata_json",
        "title": "WordPress media search - public data",
        "url": "https://oxfordinsights.com/wp-json/wp/v2/media?search=public%20data&per_page=100",
        "relative_path": "metadata/wp_media_search_public_data.json",
        "available": True,
        "source_tier": "official_primary",
        "notes": "Discovery trace para datasets XLSX publicos.",
    },
]


def download_asset(asset):
    relative_path = asset.get("relative_path")
    if not asset.get("available", True) or not asset.get("url") or not relative_path:
        return {"status": "not_publicly_discoverable", "file_size_bytes": None}

    destination = OXFORD_PATH / relative_path
    if destination.exists() and destination.stat().st_size > 0:
        return {"status": "cached", "file_size_bytes": destination.stat().st_size}

    subprocess.run(["curl", "-L", "-s", "-o", str(destination), asset["url"]], check=True)
    if not destination.exists() or destination.stat().st_size == 0:
        raise RuntimeError(f"Descarga incompleta o vacia para {asset['title']}: {asset['url']}")

    return {"status": "downloaded", "file_size_bytes": destination.stat().st_size}


manifest_rows = []
manifest_timestamp = pd.Timestamp.now(tz="UTC").isoformat()
for asset in OXFORD_ASSETS:
    outcome = download_asset(asset)
    manifest_rows.append(
        {
            **asset,
            **outcome,
            "downloaded_at_utc": manifest_timestamp,
            "absolute_path": str(OXFORD_PATH / asset["relative_path"]) if asset.get("relative_path") else "",
        }
    )

manifest_df = pd.DataFrame(manifest_rows)
manifest_df = manifest_df[
    [
        "year",
        "asset_kind",
        "title",
        "available",
        "status",
        "relative_path",
        "absolute_path",
        "url",
        "file_size_bytes",
        "source_tier",
        "downloaded_at_utc",
        "notes",
    ]
].sort_values(["year", "asset_kind", "title"], na_position="last")

manifest_path = OXFORD_PATH / "download_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print(f"Ruta Oxford: {OXFORD_PATH.resolve()}")
print(f"Assets inventariados: {len(manifest_df)}")
print(f"Assets disponibles publicamente: {manifest_df['available'].sum()}")
print(f"Structured years con XLSX: {[2019, 2020, 2021, 2022, 2023, 2025]}")
print(f"Gap estructurado identificado: 2024")
print(f"Manifest guardado: {manifest_path.relative_to(PROJECT_ROOT)}")

display(manifest_df[["year", "asset_kind", "title", "available", "status", "relative_path"]])

Ruta Oxford: /Users/francoia/Documents/MIA/Proyecto Data-Science/research/data/raw/Oxford Insights
Assets inventariados: 18
Assets disponibles publicamente: 17
Structured years con XLSX: [2019, 2020, 2021, 2022, 2023, 2025]
Gap estructurado identificado: 2024
Manifest guardado: data/raw/Oxford Insights/download_manifest.csv


,year,asset_kind,title,available,status,relative_path
1,2019,data_xlsx,2019 Government AI Readiness Index dataset,True,cached,datasets/2019_government_ai_readiness_index_da...
0,2019,report_pdf,2019 Government AI Readiness Index,True,cached,reports/2019_government_ai_readiness_index.pdf
3,2020,data_xlsx,2020 Government AI Readiness Index public dataset,True,cached,datasets/2020_government_ai_readiness_index_pu...
2,2020,report_pdf,2020 Government AI Readiness Index,True,cached,reports/2020_government_ai_readiness_index.pdf
5,2021,data_xlsx,2021 Government AI Readiness Index public dataset,True,cached,datasets/2021_government_ai_readiness_index_pu...
4,2021,report_pdf,2021 Government AI Readiness Index,True,cached,reports/2021_government_ai_readiness_index.pdf
7,2022,data_xlsx,2022 Government AI Readiness Index public data,True,cached,datasets/2022_government_ai_readiness_index_pu...
6,2022,report_pdf,2022 Government AI Readiness Index,True,cached,reports/2022_government_ai_readiness_index.pdf
9,2023,data_xlsx,2023 Government AI Readiness Index public dataset,True,cached,datasets/2023_government_ai_readiness_index_pu...
8,2023,report_pdf,2023 Government AI Readiness Index,True,cached,reports/2023_government_ai_readiness_index.pdf


In [12]:
# ============================================================================
# 6.0a Hotfix - usar el XLSX 2025 actualizado expuesto por la media API
# ============================================================================
import shutil

OXFORD_2025_DATA_INITIAL_URL = "https://oxfordinsights.com/wp-content/uploads/2026/01/2025-Government-AI-Readiness-Index-data.xlsx"
OXFORD_2025_DATA_LATEST_URL = "https://oxfordinsights.com/wp-content/uploads/2026/01/2025-Government-AI-Readiness-Index-data-1.xlsx"
OXFORD_2025_DATA_CANONICAL_PATH = OXFORD_DATASETS_PATH / "2025_government_ai_readiness_index_data.xlsx"
OXFORD_2025_DATA_ARCHIVE_PATH = OXFORD_DATASETS_PATH / "2025_government_ai_readiness_index_data_initial.xlsx"

if OXFORD_2025_DATA_CANONICAL_PATH.exists() and not OXFORD_2025_DATA_ARCHIVE_PATH.exists():
    shutil.copy2(OXFORD_2025_DATA_CANONICAL_PATH, OXFORD_2025_DATA_ARCHIVE_PATH)

subprocess.run(
    ["curl", "-L", "-s", "-o", str(OXFORD_2025_DATA_CANONICAL_PATH), OXFORD_2025_DATA_LATEST_URL],
    check=True,
)

manifest_df = manifest_df[
    ~(
        (manifest_df["year"] == 2025)
        & (manifest_df["asset_kind"] == "data_xlsx")
        & (manifest_df["title"] == "2025 Government AI Readiness Index data")
    )
].copy()

manifest_updates_df = pd.DataFrame(
    [
        {
            "year": 2025,
            "asset_kind": "data_xlsx_archive",
            "title": "2025 Government AI Readiness Index data (initial public attachment)",
            "available": OXFORD_2025_DATA_ARCHIVE_PATH.exists(),
            "status": "archived" if OXFORD_2025_DATA_ARCHIVE_PATH.exists() else "not_available",
            "relative_path": "datasets/2025_government_ai_readiness_index_data_initial.xlsx" if OXFORD_2025_DATA_ARCHIVE_PATH.exists() else None,
            "absolute_path": str(OXFORD_2025_DATA_ARCHIVE_PATH) if OXFORD_2025_DATA_ARCHIVE_PATH.exists() else "",
            "url": OXFORD_2025_DATA_INITIAL_URL,
            "file_size_bytes": OXFORD_2025_DATA_ARCHIVE_PATH.stat().st_size if OXFORD_2025_DATA_ARCHIVE_PATH.exists() else None,
            "source_tier": "official_archive",
            "downloaded_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
            "notes": "Older public attachment retained because it does not match the live 2025 dashboard scores.",
        },
        {
            "year": 2025,
            "asset_kind": "data_xlsx",
            "title": "2025 Government AI Readiness Index data",
            "available": True,
            "status": "refreshed_latest_attachment",
            "relative_path": "datasets/2025_government_ai_readiness_index_data.xlsx",
            "absolute_path": str(OXFORD_2025_DATA_CANONICAL_PATH),
            "url": OXFORD_2025_DATA_LATEST_URL,
            "file_size_bytes": OXFORD_2025_DATA_CANONICAL_PATH.stat().st_size,
            "source_tier": "official_primary",
            "downloaded_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
            "notes": "Latest media API attachment aligned with the live 2025 dashboard scores.",
        },
    ]
)
manifest_df = pd.concat([manifest_df, manifest_updates_df], ignore_index=True).sort_values(["year", "asset_kind", "title"], na_position="last")
manifest_df.to_csv(manifest_path, index=False)

print("2025 Oxford data file refreshed with latest attachment.")
print(f"Canonical: {OXFORD_2025_DATA_CANONICAL_PATH.name}")
print(f"Archive:   {OXFORD_2025_DATA_ARCHIVE_PATH.name if OXFORD_2025_DATA_ARCHIVE_PATH.exists() else 'N/A'}")

2025 Oxford data file refreshed with latest attachment.
Canonical: 2025_government_ai_readiness_index_data.xlsx
Archive:   2025_government_ai_readiness_index_data_initial.xlsx


In [13]:
# ============================================================================
# 6.1 Parsing, armonizacion y construccion del panel Oxford
# ============================================================================
OXFORD_NAME_TO_ISO3 = {
    "United States of America": "USA",
    "United States of America (the)": "USA",
    "United Kingdom of Great Britain and Northern Ireland (the)": "GBR",
    "United Arab Emirates (the)": "ARE",
    "Russian Federation": "RUS",
    "Russian Federation (the)": "RUS",
    "Republic of Moldova (the)": "MDA",
    "Moldova": "MDA",
    "Republic of Korea": "KOR",
    "Republic of Korea (the)": "KOR",
    "Korea, Republic of": "KOR",
    "Democratic People's Republic of Korea (the)": "PRK",
    "Viet Nam": "VNM",
    "Turkey": "TUR",
    "Türkiye": "TUR",
    "Taiwan": "TWN",
    "Palestine, State of": "PSE",
    "Lao People's Democratic Republic": "LAO",
    "Lao People's Democratic Republic (the)": "LAO",
    "Cote d'Ivoire": "CIV",
    "Côte d'Ivoire": "CIV",
    "Democratic Republic of the Congo": "COD",
    "Democratic Republic of the Congo (the)": "COD",
    "Congo, Democratic Republic of the": "COD",
    "Congo (the)": "COG",
    "Syrian Arab Republic": "SYR",
    "Syrian Arab Republic (the)": "SYR",
    "Iran (Islamic Republic of)": "IRN",
    "Venezuela (Bolivarian Republic of)": "VEN",
    "Bolivia (Plurinational State of)": "BOL",
    "Micronesia": "FSM",
    "Micronesia (Federated States of)": "FSM",
    "Marshall Islands (the)": "MHL",
    "Brunei Darussalam": "BRN",
    "Cabo Verde": "CPV",
    "Eswatini": "SWZ",
    "Swaziland": "SWZ",
    "Bahamas": "BHS",
    "Bahamas (the)": "BHS",
    "Gambia": "GMB",
    "Gambia (the)": "GMB",
    "Gambia (Republic of The)": "GMB",
    "Macao": "MAC",
    "Hong Kong": "HKG",
    "Central African Republic (the)": "CAF",
    "Comoros (the)": "COM",
    "Dominican Republic (the)": "DOM",
    "Guinea Bissau": "GNB",
    "Netherlands (the)": "NLD",
    "Niger (the)": "NER",
    "Philippines (the)": "PHL",
    "Sudan (the)": "SDN",
    "The former Yugoslav Republic of Macedonia": "MKD",
}

ISO3_TO_COUNTRY_STD = {
    "TWN": "Taiwan",
    "PSE": "Palestine",
    "XKX": "Kosovo",
}

YEAR_METADATA = {
    2019: {
        "source_dataset": "2019_government_ai_readiness_index_dataset.xlsx",
        "report_url": "https://oxfordinsights.com/wp-content/uploads/2023/12/ai-gov-readiness-report_v08.pdf",
        "data_asset_url": "https://oxfordinsights.com/wp-content/uploads/2024/01/SHARED_-2019-Index-data-for-report.xlsx",
        "methodology_url": None,
        "report_edition": "2019",
        "methodology_regime": "v1_2019_4_clusters_12_indicators_scale_0_to_10",
        "comparability_group": "low_comparability_2019_scale_shift",
    },
    2020: {
        "source_dataset": "2020_government_ai_readiness_index_public_dataset.xlsx",
        "report_url": "https://oxfordinsights.com/wp-content/uploads/2023/11/AIReadinessReport.pdf",
        "data_asset_url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2020-Government-AI-Readiness-Index-public-dataset.xlsx",
        "methodology_url": None,
        "report_edition": "2020",
        "methodology_regime": "v2_2020_3_pillars_9_dimensions_scale_0_to_100",
        "comparability_group": "moderate_comparability_2020_2023",
    },
    2021: {
        "source_dataset": "2021_government_ai_readiness_index_public_dataset.xlsx",
        "report_url": "https://oxfordinsights.com/wp-content/uploads/2023/11/Government_AI_Readiness_21.pdf",
        "data_asset_url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2021-Government-AI-Readiness-Index-public-dataset.xlsx",
        "methodology_url": None,
        "report_edition": "2021",
        "methodology_regime": "v2_2020_2023_3_pillars_9_dimensions_scale_0_to_100",
        "comparability_group": "moderate_comparability_2020_2023",
    },
    2022: {
        "source_dataset": "2022_government_ai_readiness_index_public_data.xlsx",
        "report_url": "https://oxfordinsights.com/wp-content/uploads/2023/11/Government_AI_Readiness_2022_FV.pdf",
        "data_asset_url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2022-Government-AI-Readiness-Index-public-data.xlsx",
        "methodology_url": None,
        "report_edition": "2022",
        "methodology_regime": "v2_2020_2023_3_pillars_9_dimensions_scale_0_to_100",
        "comparability_group": "moderate_comparability_2020_2023",
    },
    2023: {
        "source_dataset": "2023_government_ai_readiness_index_public_dataset.xlsx",
        "report_url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2023-Government-AI-Readiness-Index.pdf",
        "data_asset_url": "https://oxfordinsights.com/wp-content/uploads/2023/12/2023-AI-Readiness-Index-public-dataset.xlsx",
        "methodology_url": None,
        "report_edition": "2023",
        "methodology_regime": "v2_2020_2023_3_pillars_9_dimensions_scale_0_to_100",
        "comparability_group": "moderate_comparability_2020_2023",
    },
    2025: {
        "source_dataset": "2025_government_ai_readiness_index_data.xlsx",
        "report_url": "https://oxfordinsights.com/wp-content/uploads/2025/12/2025-Government-AI-Readiness-Index.pdf",
        "data_asset_url": "https://oxfordinsights.com/wp-content/uploads/2026/01/2025-Government-AI-Readiness-Index-data.xlsx",
        "methodology_url": "https://oxfordinsights.com/wp-content/uploads/2026/01/Methodology-Report-2025.pdf",
        "report_edition": "2025",
        "methodology_regime": "v3_2025_6_pillars_14_dimensions_scale_0_to_100",
        "comparability_group": "low_comparability_2025_methodology_change",
    },
}

PILLAR_NAMES = {
    2020: {"Government", "Technology Sector", "Data and Infrastructure"},
    2021: {"Government", "Technology Sector", "Data and Infrastructure"},
    2022: {"Government", "Technology Sector", "Data and Infrastructure"},
    2023: {"Government", "Technology Sector", "Data and Infrastructure"},
    2025: {"Policy Capacity", "AI Infrastructure", "Governance", "Public Sector Adoption", "Development & Diffusion", "Resilience"},
}

INDICATOR_ORIGINAL_LABELS = {
    "ai_readiness_score": "Overall reported score harmonized to 0-100",
    "ai_readiness_score_original": "Overall reported score in original Oxford scale",
}
INDICATOR_GROUPS = {
    "ai_readiness_score": "target",
    "ai_readiness_score_original": "target_original_scale",
}


def snake_case(value):
    value = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii")
    value = value.lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", "_", value).strip("_")
    return re.sub(r"_+", "_", value)



def country_to_iso3(name):
    name = str(name).strip()
    if name in OXFORD_NAME_TO_ISO3:
        return OXFORD_NAME_TO_ISO3[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        try:
            return pycountry.countries.search_fuzzy(name)[0].alpha_3
        except LookupError:
            return None



def iso3_to_country_std(iso3, fallback):
    if iso3 in ISO3_TO_COUNTRY_STD:
        return ISO3_TO_COUNTRY_STD[iso3]
    if not iso3:
        return fallback
    country = pycountry.countries.get(alpha_3=iso3)
    return country.name if country else fallback



def prepare_2019_dataframe(path):
    raw = pd.read_excel(path, sheet_name="Data", header=None)
    column_map = {
        0: "country_name_oxford",
        1: "indicator_privacy_laws",
        2: "indicator_ai_strategy",
        3: "indicator_data_availability",
        4: "indicator_govt_procurement_advanced_technology_products",
        5: "indicator_data_capability_in_government",
        6: "indicator_technology_skills",
        7: "indicator_ai_startups",
        8: "indicator_log_ai_startups",
        9: "indicator_private_sector_innovation_capability",
        10: "indicator_digital_public_services",
        11: "indicator_effectiveness_of_government",
        12: "indicator_importance_of_icts_to_government_vision_of_future",
        14: "normalized_privacy_laws",
        15: "normalized_ai_strategy",
        16: "normalized_data_availability",
        17: "normalized_govt_procurement_advanced_technology_products",
        18: "normalized_data_capability_in_government",
        19: "normalized_technology_skills",
        20: "normalized_log_ai_startups",
        21: "normalized_private_sector_innovation_capability",
        22: "normalized_digital_public_services",
        23: "normalized_effectiveness_of_government",
        24: "normalized_importance_of_icts_to_government_vision_of_future",
        26: "ai_readiness_score_original",
        28: "sum_of_cluster_averages",
        29: "cluster_avg_governance",
        30: "cluster_avg_infrastructure_and_data",
        31: "cluster_avg_skills_and_education",
        32: "cluster_avg_government_and_public_services",
    }
    original_labels = {
        "indicator_privacy_laws": "Privacy laws (yes = 1, no = 0)",
        "indicator_ai_strategy": "AI strategy = 2 (forthcoming = 1, none = 0)",
        "indicator_data_availability": "Data availability",
        "indicator_govt_procurement_advanced_technology_products": "Gov't procurement of advanced technology products",
        "indicator_data_capability_in_government": "Data capability (in govt)",
        "indicator_technology_skills": "Technology skills",
        "indicator_ai_startups": "AI startups",
        "indicator_log_ai_startups": "Log of AI startups",
        "indicator_private_sector_innovation_capability": "(Private sector) innovation capability",
        "indicator_digital_public_services": "Digital public services",
        "indicator_effectiveness_of_government": "Effectiveness of government",
        "indicator_importance_of_icts_to_government_vision_of_future": "Importance of ICTs to government vision of the future",
        "normalized_privacy_laws": "NORMALISED Privacy laws (yes = 1, no = 0)",
        "normalized_ai_strategy": "NORMALISED AI strategy = 2 (forthcoming = 1, none = 0)",
        "normalized_data_availability": "NORMALISED Data availability",
        "normalized_govt_procurement_advanced_technology_products": "NORMALISED Gov't procurement of advanced technology products",
        "normalized_data_capability_in_government": "NORMALISED Data capability (in govt)",
        "normalized_technology_skills": "NORMALISED Technology skills",
        "normalized_log_ai_startups": "NORMALISED Log of AI startups",
        "normalized_private_sector_innovation_capability": "NORMALISED (Private sector) innovation capability",
        "normalized_digital_public_services": "NORMALISED Digital public services",
        "normalized_effectiveness_of_government": "NORMALISED Effectiveness of government",
        "normalized_importance_of_icts_to_government_vision_of_future": "NORMALISED Importance of ICTs to government vision of the future",
        "ai_readiness_score_original": "INDEX SCORE",
        "sum_of_cluster_averages": "Sum of cluster averages",
        "cluster_avg_governance": "Average: Governance",
        "cluster_avg_infrastructure_and_data": "Average: Infrastructure and data",
        "cluster_avg_skills_and_education": "Average: Skills and education",
        "cluster_avg_government_and_public_services": "Average: Government and public services",
    }
    group_map = {}
    for key in original_labels:
        if key.startswith("indicator_"):
            group_map[key] = "raw_indicator"
        elif key.startswith("normalized_"):
            group_map[key] = "normalized_indicator"
        elif key.startswith("cluster_avg_"):
            group_map[key] = "cluster_average"
        elif key == "sum_of_cluster_averages":
            group_map[key] = "cluster_average_sum"
    group_map["ai_readiness_score"] = "target"
    group_map["ai_readiness_score_original"] = "target_original_scale"

    df = raw.iloc[7:, list(column_map)].copy()
    df.columns = [column_map[col] for col in column_map]
    df = df[df["country_name_oxford"].notna()].copy()
    df["country_name_oxford"] = df["country_name_oxford"].astype(str).str.strip()
    for col in df.columns:
        if col != "country_name_oxford":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["oxford_rank_reported"] = pd.Series(range(1, len(df) + 1), dtype="Int64")
    df["ai_readiness_score"] = df["ai_readiness_score_original"] * 10.0
    df["score_scale_original"] = "0_to_10"
    df["year"] = 2019
    return df, original_labels, group_map



def prepare_modern_dataframe(path, year, detail_sheet, pillar_names):
    raw = pd.read_excel(path, sheet_name=detail_sheet)
    subheaders = raw.iloc[0]
    df = raw.iloc[1:].copy()
    df.columns = [str(value).strip() if pd.notna(value) else str(col) for col, value in zip(raw.columns, subheaders)]
    df = df[[col for col in df.columns if not str(col).startswith("Unnamed") and col != "nan"]].copy()

    rename_map = {}
    original_labels = {}
    group_map = {}
    for column in df.columns:
        if column == "Country":
            standard = "country_name_oxford"
        elif column in {"Overall score", "Total", "Total Score"}:
            standard = "ai_readiness_score"
            original_labels["ai_readiness_score"] = column
            original_labels["ai_readiness_score_original"] = f"{column} (original scale)"
            group_map["ai_readiness_score"] = "target"
            group_map["ai_readiness_score_original"] = "target_original_scale"
        elif "Ranking" in column or column == "Ranking":
            standard = "oxford_rank_reported"
        elif column in pillar_names:
            standard = f"pillar_{snake_case(column)}"
            original_labels[standard] = column
            group_map[standard] = "pillar"
        else:
            standard = f"dimension_{snake_case(column)}"
            original_labels[standard] = column
            group_map[standard] = "dimension"
        rename_map[column] = standard

    df = df.rename(columns=rename_map)
    df["country_name_oxford"] = df["country_name_oxford"].astype(str).str.strip()
    for col in df.columns:
        if col != "country_name_oxford":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["ai_readiness_score_original"] = df["ai_readiness_score"]
    df["score_scale_original"] = "0_to_100"
    if "oxford_rank_reported" in df.columns:
        df["oxford_rank_reported"] = pd.to_numeric(df["oxford_rank_reported"], errors="coerce").astype("Int64")
    else:
        df["oxford_rank_reported"] = df["ai_readiness_score"].rank(method="first", ascending=False).astype("Int64")
    df["year"] = year
    return df, original_labels, group_map


wide_frames = []
for year, metadata in YEAR_METADATA.items():
    dataset_path = OXFORD_DATASETS_PATH / metadata["source_dataset"]
    if not dataset_path.exists():
        continue

    if year == 2019:
        year_df, original_labels, group_map = prepare_2019_dataframe(dataset_path)
    elif year == 2025:
        year_df, original_labels, group_map = prepare_modern_dataframe(dataset_path, year, "Dimensions-Pillars", PILLAR_NAMES[year])
    else:
        year_df, original_labels, group_map = prepare_modern_dataframe(dataset_path, year, "Detailed scores", PILLAR_NAMES[year])

    INDICATOR_ORIGINAL_LABELS.update(original_labels)
    INDICATOR_GROUPS.update(group_map)

    year_df["iso3"] = year_df["country_name_oxford"].apply(country_to_iso3)
    year_df["country_name_std"] = year_df.apply(
        lambda row: iso3_to_country_std(row["iso3"], row["country_name_oxford"]),
        axis=1,
    )
    year_df["source_name"] = "Oxford_Insights_Government_AI_Readiness_Index"
    year_df["source_dataset"] = metadata["source_dataset"]
    year_df["report_url"] = metadata["report_url"]
    year_df["data_asset_url"] = metadata["data_asset_url"]
    year_df["methodology_url"] = metadata["methodology_url"]
    year_df["source_tier"] = "official_primary"
    year_df["coverage_level"] = "country"
    year_df["report_edition"] = metadata["report_edition"]
    year_df["methodology_regime"] = metadata["methodology_regime"]
    year_df["comparability_group"] = metadata["comparability_group"]
    wide_frames.append(year_df)

oxford_raw_df = pd.concat(wide_frames, ignore_index=True, sort=False)
oxford_raw_df = oxford_raw_df.sort_values(["year", "oxford_rank_reported", "country_name_std"]).reset_index(drop=True)

metadata_columns = {
    "country_name_oxford",
    "country_name_std",
    "iso3",
    "year",
    "oxford_rank_reported",
    "source_name",
    "source_dataset",
    "report_url",
    "data_asset_url",
    "methodology_url",
    "source_tier",
    "coverage_level",
    "report_edition",
    "methodology_regime",
    "comparability_group",
    "score_scale_original",
}

indicator_columns = [col for col in oxford_raw_df.columns if col not in metadata_columns]
for column in indicator_columns:
    oxford_raw_df[column] = pd.to_numeric(oxford_raw_df[column], errors="coerce")

unmapped_oxford = sorted(set(oxford_raw_df.loc[oxford_raw_df["iso3"].isna(), "country_name_oxford"]))
duplicate_oxford = oxford_raw_df[oxford_raw_df["iso3"].notna()].duplicated(["year", "iso3"], keep=False)
if unmapped_oxford:
    raise RuntimeError(f"Oxford tiene paises sin ISO3: {unmapped_oxford}")
if duplicate_oxford.any():
    raise RuntimeError(
        "Oxford tiene duplicados year+iso3: "
        + str(
            oxford_raw_df.loc[duplicate_oxford, ["year", "iso3", "country_name_oxford"]]
            .sort_values(["year", "iso3"])
            .to_dict(orient="records")
        )
    )


def infer_unit(row):
    indicator = row["indicator"]
    year = row["year"]
    if indicator == "ai_readiness_score":
        return "index_0_to_100_harmonized"
    if indicator == "ai_readiness_score_original":
        return "index_0_to_10" if year == 2019 else "index_0_to_100"
    if indicator.startswith("pillar_") or indicator.startswith("dimension_"):
        return "index_0_to_100"
    if indicator.startswith("normalized_") or indicator.startswith("cluster_avg_"):
        return "index_0_to_1"
    if indicator.startswith("indicator_"):
        return "mixed_raw_source_units"
    if indicator == "sum_of_cluster_averages":
        return "sum_of_normalized_cluster_scores"
    return "reported_score"


long_id_vars = [
    "iso3",
    "country_name_std",
    "country_name_oxford",
    "year",
    "oxford_rank_reported",
    "source_name",
    "source_dataset",
    "report_url",
    "data_asset_url",
    "methodology_url",
    "source_tier",
    "coverage_level",
    "report_edition",
    "methodology_regime",
    "comparability_group",
    "score_scale_original",
]

long_indicator_columns = [col for col in indicator_columns if col != "oxford_rank_reported"]
oxford_long_df = (
    oxford_raw_df.melt(
        id_vars=long_id_vars,
        value_vars=long_indicator_columns,
        var_name="indicator",
        value_name="value",
    )
    .dropna(subset=["value"])
    .reset_index(drop=True)
)
oxford_long_df["source_variable_original"] = oxford_long_df["indicator"].map(INDICATOR_ORIGINAL_LABELS).fillna(oxford_long_df["indicator"])
oxford_long_df["indicator_group"] = oxford_long_df["indicator"].map(INDICATOR_GROUPS).fillna("other")
oxford_long_df["unit_original"] = oxford_long_df.apply(infer_unit, axis=1)

oxford_study_df = oxford_raw_df[oxford_raw_df["iso3"].isin(study_iso3_all)].copy()
latest_year = int(oxford_raw_df["year"].max())
oxford_snapshot_latest_df = oxford_raw_df[oxford_raw_df["year"] == latest_year].copy()
oxford_pillars_long_df = oxford_long_df[oxford_long_df["indicator_group"] == "pillar"].copy()
oxford_dimensions_long_df = oxford_long_df[oxford_long_df["indicator_group"] == "dimension"].copy()

print(f"Oxford rows (wide audit): {len(oxford_raw_df)}")
print(f"Oxford rows (long): {len(oxford_long_df)}")
print(f"Oxford structured years: {sorted(oxford_raw_df['year'].unique().tolist())}")
print(f"Oxford countries latest year ({latest_year}): {oxford_snapshot_latest_df['iso3'].nunique()}")
print(f"Oxford study coverage latest year: {oxford_snapshot_latest_df['iso3'].isin(study_iso3_all).sum()}/86")

Oxford rows (wide audit): 1095
Oxford rows (long): 20108
Oxford structured years: [2019, 2020, 2021, 2022, 2023, 2025]
Oxford countries latest year (2025): 195
Oxford study coverage latest year: 86/86


## Fuente 6 — Oxford Insights 2024: Extraccion desde PDF oficial

Oxford Insights no publico dataset estructurado (XLSX/CSV) para la edicion 2024.
El reporte PDF oficial contiene las tablas de scores en las paginas 43-50 con columnas:
`Country | Total | Government | Technology Sector | Data and Infrastructure`.

Esta celda extrae esos datos con `pdfplumber`, normaliza nombres de pais a ISO3 y los integra
al panel Oxford con `source_tier = 'official_pdf_extracted'` para distinguirlos de los XLSX.

**Nota metodologica:** Solo estan disponibles los 3 pilares (v2). Las 9 dimensiones no aparecen
en el PDF, por lo que los campos `dimension_*` quedaran NaN para el año 2024.
Ver `info_data/PLAN_RECUPERACION_OXFORD_2024.md` para el historial de investigacion.

In [17]:
# ============================================================================
# 6.1.a  Extraccion 2024 — PDF oficial (Oxford no publico XLSX para este año)
# ============================================================================
import pdfplumber
from datetime import timezone, datetime

PDF_2024_PATH = OXFORD_REPORTS_PATH / "2024_government_ai_readiness_index.pdf"
PDF_2024_URL  = "https://oxfordinsights.com/wp-content/uploads/2024/12/2024-Government-AI-Readiness-Index-1.pdf"

# ── Diccionario de nombres especiales del PDF 2024 ───────────────────────────
# El extractor PDF produce nombres concatenados (sin espacios) o con formato especial.
# "SriLanka" -> "Sri Lanka", "Republicof Korea" -> "Republic of Korea", etc.
PDF_2024_NAME_FIXES = {
    "SriLanka":                          "Sri Lanka",
    "StateofPalestine":                  "Palestine, State of",
    "Iran(IslamicRepublicof)":           "Iran (Islamic Republic of)",
    "Korea,RepublicOf":                  "Republic of Korea",
    "Korea(Republicof)":                 "Republic of Korea",
    "Republicof Korea":                  "Republic of Korea",
    "RepublicofKorea":                   "Republic of Korea",
    "Republicof Moldova":                "Republic of Moldova",
    "RepublicofMoldova":                 "Republic of Moldova",
    "DemocraticPeople'sRepublicofKorea": "Democratic People's Republic of Korea",
    "DemocraticRepublicoftheCongo":      "Democratic Republic of the Congo",
    "CongoDemocraticRepublicofthe":      "Democratic Republic of the Congo",
    "United Republicof Tanzania":        "United Republic of Tanzania",
    "CentralAfricanRepublic":            "Central African Republic",
    "TrinidadandTobago":                 "Trinidad and Tobago",
    "SaoTomeandPrincipe":                "Sao Tome and Principe",
    "EquatorialGuinea":                  "Equatorial Guinea",
    "GuineaBissau":                      "Guinea-Bissau",
    "SierraLeone":                       "Sierra Leone",
    "BurkinaFaso":                       "Burkina Faso",
    "CaboVerde":                         "Cabo Verde",
    "BosniaandHerzegovina":              "Bosnia and Herzegovina",
    "PapuaNewGuinea":                    "Papua New Guinea",
    "NorthMacedonia":                    "North Macedonia",
    "NorthernMacedonia":                 "North Macedonia",
    "SouthAfrica":                       "South Africa",
    "SouthSudan":                        "South Sudan",
    "CostaRica":                         "Costa Rica",
    "DominicanRepublic":                 "Dominican Republic",
    "ElSalvador":                        "El Salvador",
    "NewZealand":                        "New Zealand",
    "SaudiArabia":                       "Saudi Arabia",
    "CotedIvoire":                       "Côte d'Ivoire",
    "Côted'Ivoire":                      "Côte d'Ivoire",
    "AntiguaandBarbuda":                 "Antigua and Barbuda",
    "Antiguaand Barbuda":                "Antigua and Barbuda",
    "Saint Kittsand Nevis":              "Saint Kitts and Nevis",
    "SaintKittsandNevis":                "Saint Kitts and Nevis",
    "Saint Vincentandthe Grenadines":    "Saint Vincent and the Grenadines",
    "SaintVincentandtheGrenadines":      "Saint Vincent and the Grenadines",
    "Gambia(Republicof The)":            "Gambia",
    "Gambia(Republic of The)":           "Gambia",
    "Bolivia(Plurinational Stateof)":    "Bolivia (Plurinational State of)",
    "Bolivia(PlurinationalStateof)":     "Bolivia (Plurinational State of)",
    "Bolivia,PlurinationalStateof":      "Bolivia (Plurinational State of)",
    "UnitedStatesofAmerica":             "United States of America",
    "UnitedStates":                      "United States of America",
    "UnitedArabEmirates":                "United Arab Emirates",
    "UnitedKingdom":                     "United Kingdom of Great Britain and Northern Ireland",
    "Venezuela,BolivarianRepublicof":    "Venezuela (Bolivarian Republic of)",
    "SyrianArabRepublic":                "Syrian Arab Republic",
    "LaoPeople'sDemocraticRepublic":     "Lao People's Democratic Republic",
    "WesternSahara":                     "Western Sahara",
    "CongoBrazzaville":                  "Congo",
}

SKIP_NAMES_2024 = {
    "Country", "", "CountryTotal", "GovernmentPillar", "TechnologySectorPillar",
    "DataandInfrastructurePillar", "Total", "Government", "TechnologySector",
}


def _deconcat_2024(raw):
    """Normaliza nombres de pais concatenados del PDF 2024.
    Paso 1: lookup exacto en PDF_2024_NAME_FIXES.
    Paso 2: insertar espacio entre minuscula-MAYUSCULA (CamelCase) y re-lookup.
    Paso 3: devolver tal cual para que pycountry fuzzy lo resuelva.
    """
    raw = str(raw).strip().replace("\n", " ")
    if raw in PDF_2024_NAME_FIXES:
        return PDF_2024_NAME_FIXES[raw]
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", raw)
    if s in PDF_2024_NAME_FIXES:
        return PDF_2024_NAME_FIXES[s]
    return s


def _safe_float(v):
    try:
        return float(str(v).strip().replace("\n", ""))
    except (ValueError, TypeError):
        return None


def _parse_2024_pdf(pdf_path):
    """Extrae la tabla de scores de las paginas 43-50 del PDF 2024.
    Estructura de cada tabla: Country | Total | Government | Tech Sector | D&I.
    pdfplumber detecta las tablas correctamente; se filtran filas de encabezado
    por la incapacidad de parsear la columna Total como float.
    """
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num in range(43, 51):        # paginas 1-indexadas 43 a 50
            pg = pdf.pages[page_num - 1]
            for tbl in pg.extract_tables():
                for row in tbl:
                    if not row or len(row) < 5 or row[0] is None:
                        continue
                    country_raw = str(row[0]).strip().replace("\n", " ")
                    if not country_raw or country_raw in SKIP_NAMES_2024:
                        continue
                    total = _safe_float(row[1])
                    if total is None:            # fila de encabezado de pagina
                        continue
                    rows.append({
                        "country_name_oxford":            _deconcat_2024(country_raw),
                        "ai_readiness_score":             total,
                        "pillar_government":              _safe_float(row[2]),
                        "pillar_technology_sector":       _safe_float(row[3]),
                        "pillar_data_and_infrastructure": _safe_float(row[4]),
                    })
    return pd.DataFrame(rows)


# ── Extraer ──────────────────────────────────────────────────────────────────
df_2024 = _parse_2024_pdf(PDF_2024_PATH)
print(f"Paises extraidos del PDF 2024: {len(df_2024)}")
print(f"Score range: [{df_2024['ai_readiness_score'].min():.2f}, {df_2024['ai_readiness_score'].max():.2f}]")

# ── Mapeo ISO3 ────────────────────────────────────────────────────────────────
df_2024["iso3"]             = df_2024["country_name_oxford"].apply(country_to_iso3)
df_2024["country_name_std"] = df_2024.apply(
    lambda row: iso3_to_country_std(row["iso3"], row["country_name_oxford"]), axis=1,
)

unmapped_2024 = df_2024.loc[df_2024["iso3"].isna(), "country_name_oxford"].tolist()
if unmapped_2024:
    raise RuntimeError(f"2024 PDF: nombres sin ISO3 ({len(unmapped_2024)}): {unmapped_2024}")
print(f"Mapeo ISO3: {len(df_2024)}/{len(df_2024)} — OK")

# ── Metadatos 2024 ────────────────────────────────────────────────────────────
META_2024 = {
    "source_dataset":      "2024_government_ai_readiness_index.pdf",
    "report_url":          PDF_2024_URL,
    "data_asset_url":      PDF_2024_URL,
    "methodology_url":     None,
    "report_edition":      "2024",
    "methodology_regime":  "v2_2020_2023_3_pillars_9_dimensions_scale_0_to_100",
    "comparability_group": "moderate_comparability_2020_2023",
}
YEAR_METADATA[2024] = META_2024

df_2024["ai_readiness_score_original"] = df_2024["ai_readiness_score"]
df_2024["score_scale_original"]        = "0_to_100"
df_2024["oxford_rank_reported"]        = (
    df_2024["ai_readiness_score"].rank(method="first", ascending=False).astype("Int64")
)
df_2024["year"]             = 2024
df_2024["source_name"]      = "Oxford_Insights_Government_AI_Readiness_Index"
df_2024["source_dataset"]   = META_2024["source_dataset"]
df_2024["report_url"]       = META_2024["report_url"]
df_2024["data_asset_url"]   = META_2024["data_asset_url"]
df_2024["methodology_url"]  = META_2024["methodology_url"]
df_2024["source_tier"]      = "official_pdf_extracted"
df_2024["coverage_level"]   = "country"
df_2024["report_edition"]   = META_2024["report_edition"]
df_2024["methodology_regime"]  = META_2024["methodology_regime"]
df_2024["comparability_group"] = META_2024["comparability_group"]

# ── Spot-checks ───────────────────────────────────────────────────────────────
# Valores extraidos del texto narrativo del PDF 2024 (paginas de snapshots regionales).
SPOT_2024 = {
    "USA": 87.03, "CAN": 78.18, "FRA": 79.36,
    "GBR": 78.88, "SGP": 84.25, "KOR": 79.98,
    "BRA": 65.89, "IND": 62.81, "AUS": 76.44,
}
print("\nSpot-checks contra valores narrativos del PDF 2024:")
idx_2024 = df_2024.set_index("iso3")
all_ok = True
for iso3_check, exp in SPOT_2024.items():
    if iso3_check in idx_2024.index:
        act = float(idx_2024.loc[iso3_check, "ai_readiness_score"])
        ok  = abs(act - exp) < 0.5
        if not ok:
            all_ok = False
        print(f"  {iso3_check}: {act:.2f} (esperado {exp:.2f}) {'OK' if ok else 'FAIL'}")
    else:
        print(f"  {iso3_check}: NO ENCONTRADO")
        all_ok = False
if not all_ok:
    raise RuntimeError("Spot-checks 2024 fallaron — revisar extraccion del PDF")

# ── Integrar al panel ─────────────────────────────────────────────────────────
oxford_raw_df = pd.concat([oxford_raw_df, df_2024], ignore_index=True, sort=False)
oxford_raw_df = oxford_raw_df.sort_values(
    ["year", "oxford_rank_reported", "country_name_std"]
).reset_index(drop=True)

dupes = oxford_raw_df.duplicated(["year", "iso3"], keep=False).sum()
if dupes:
    raise RuntimeError(f"Duplicados year+iso3 tras integrar 2024: {dupes}")

# ── Regenerar tablas derivadas ────────────────────────────────────────────────
# Las columnas dimension_* de 2024 quedan NaN (datos no publicados en PDF).
all_indicator_cols = [col for col in oxford_raw_df.columns if col not in metadata_columns]
for col in all_indicator_cols:
    oxford_raw_df[col] = pd.to_numeric(oxford_raw_df[col], errors="coerce")

long_indicator_columns = [c for c in all_indicator_cols if c != "oxford_rank_reported"]
oxford_long_df = (
    oxford_raw_df.melt(
        id_vars=long_id_vars,
        value_vars=long_indicator_columns,
        var_name="indicator",
        value_name="value",
    )
    .dropna(subset=["value"])
    .reset_index(drop=True)
)
oxford_long_df["source_variable_original"] = (
    oxford_long_df["indicator"].map(INDICATOR_ORIGINAL_LABELS).fillna(oxford_long_df["indicator"])
)
oxford_long_df["indicator_group"] = oxford_long_df["indicator"].map(INDICATOR_GROUPS).fillna("other")
oxford_long_df["unit_original"]   = oxford_long_df.apply(infer_unit, axis=1)

oxford_study_df           = oxford_raw_df[oxford_raw_df["iso3"].isin(study_iso3_all)].copy()
latest_year               = int(oxford_raw_df["year"].max())   # sigue siendo 2025
oxford_snapshot_latest_df = oxford_raw_df[oxford_raw_df["year"] == latest_year].copy()
oxford_pillars_long_df    = oxford_long_df[oxford_long_df["indicator_group"] == "pillar"].copy()
oxford_dimensions_long_df = oxford_long_df[oxford_long_df["indicator_group"] == "dimension"].copy()

# ── Actualizar manifiesto ─────────────────────────────────────────────────────
now_utc = datetime.now(tz=timezone.utc).isoformat()
new_manifest_row = pd.DataFrame([{
    "year":              2024,
    "asset_kind":        "data_pdf_extracted",
    "title":             "2024 Government AI Readiness Index — datos extraidos del PDF oficial",
    "available":         True,
    "status":            "extracted_from_pdf",
    "relative_path":     "reports/2024_government_ai_readiness_index.pdf",
    "absolute_path":     str(OXFORD_REPORTS_PATH / "2024_government_ai_readiness_index.pdf"),
    "url":               PDF_2024_URL,
    "file_size_bytes":   float(PDF_2024_PATH.stat().st_size),
    "source_tier":       "official_pdf_extracted",
    "downloaded_at_utc": now_utc,
    "notes": (
        f"Scores extraidos con pdfplumber de paginas 43-50 del PDF oficial el {now_utc[:10]}. "
        "Disponibles: ai_readiness_score + 3 pilares. "
        "Sin datos de 9 dimensiones (no publicados en PDF). "
        "188 paises incluidos. source_tier=official_pdf_extracted."
    ),
}])
manifest_df = pd.concat([manifest_df, new_manifest_row], ignore_index=True)
manifest_df.to_csv(manifest_path, index=False)

# ── Resumen ────────────────────────────────────────────────────────────────────
study_2024 = oxford_study_df[oxford_study_df["year"] == 2024]["iso3"].nunique()
missing_study_2024 = sorted(
    study_iso3_all - set(oxford_study_df.loc[oxford_study_df["year"] == 2024, "iso3"])
)
print(f"\nPanel Oxford ACTUALIZADO con 2024:")
print(f"  Anos integrados: {sorted(oxford_raw_df['year'].unique().tolist())}")
print(f"  Filas totales (wide): {len(oxford_raw_df)}")
print(f"  Paises globales 2024: {(oxford_raw_df['year'] == 2024).sum()}")
print(f"  Paises muestra 2024:  {study_2024}/86  (faltantes: {missing_study_2024})")
print(f"  Manifiesto: {len(manifest_df)} entradas")
print(f"  Nota: dimension_* = NaN para 2024 (solo pilares disponibles en el PDF)")

Paises extraidos del PDF 2024: 188
Score range: [14.62, 87.03]
Mapeo ISO3: 188/188 — OK

Spot-checks contra valores narrativos del PDF 2024:
  USA: 87.03 (esperado 87.03) OK
  CAN: 78.18 (esperado 78.18) OK
  FRA: 79.36 (esperado 79.36) OK
  GBR: 78.88 (esperado 78.88) OK
  SGP: 84.25 (esperado 84.25) OK
  KOR: 79.98 (esperado 79.98) OK
  BRA: 65.89 (esperado 65.89) OK
  IND: 62.81 (esperado 62.81) OK
  AUS: 76.45 (esperado 76.44) OK

Panel Oxford ACTUALIZADO con 2024:
  Anos integrados: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Filas totales (wide): 1283
  Paises globales 2024: 188
  Paises muestra 2024:  86/86  (faltantes: [])
  Manifiesto: 20 entradas
  Nota: dimension_* = NaN para 2024 (solo pilares disponibles en el PDF)


In [15]:
# ============================================================================
# 6.1.b Alinear metadata 2025 con el adjunto oficial mas reciente
# ============================================================================
YEAR_METADATA[2025]["data_asset_url"] = OXFORD_2025_DATA_LATEST_URL

for dataframe_name in [
    "oxford_raw_df",
    "oxford_wide_export_df",
    "oxford_long_df",
    "oxford_study_df",
    "oxford_snapshot_latest_df",
    "oxford_pillars_long_df",
    "oxford_dimensions_long_df",
]:
    dataframe = globals().get(dataframe_name)
    if dataframe is not None and {"year", "data_asset_url"}.issubset(dataframe.columns):
        dataframe.loc[dataframe["year"] == 2025, "data_asset_url"] = OXFORD_2025_DATA_LATEST_URL

print("Metadata 2025 alineada con el adjunto oficial mas reciente:")
print(OXFORD_2025_DATA_LATEST_URL)

Metadata 2025 alineada con el adjunto oficial mas reciente:
https://oxfordinsights.com/wp-content/uploads/2026/01/2025-Government-AI-Readiness-Index-data-1.xlsx


In [18]:
# ============================================================================
# 6.2 Guardado de outputs y verificacion de cobertura
# ============================================================================
oxford_all_raw_path = OXFORD_PATH / "oxford_ai_readiness_all_raw.csv"
oxford_wide_path = OXFORD_PATH / "oxford_ai_readiness_wide.csv"
oxford_study_path = OXFORD_PATH / "oxford_ai_readiness_study.csv"
oxford_long_path = OXFORD_PATH / "oxford_ai_readiness_long.csv"
oxford_snapshot_path = OXFORD_PATH / "oxford_ai_readiness_snapshot_latest.csv"
oxford_pillars_path = OXFORD_PATH / "oxford_ai_readiness_pillars_long.csv"
oxford_dimensions_path = OXFORD_PATH / "oxford_ai_readiness_dimensions_long.csv"

export_metadata_drop = [
    "source_name",
    "source_dataset",
    "report_url",
    "data_asset_url",
    "methodology_url",
    "source_tier",
    "coverage_level",
]
oxford_wide_export_df = oxford_raw_df.drop(columns=export_metadata_drop)

oxford_raw_df.to_csv(oxford_all_raw_path, index=False)
oxford_wide_export_df.to_csv(oxford_wide_path, index=False)
oxford_study_df.to_csv(oxford_study_path, index=False)
oxford_long_df.to_csv(oxford_long_path, index=False)
oxford_snapshot_latest_df.to_csv(oxford_snapshot_path, index=False)
oxford_pillars_long_df.to_csv(oxford_pillars_path, index=False)
oxford_dimensions_long_df.to_csv(oxford_dimensions_path, index=False)

coverage_year_df = (
    oxford_raw_df.groupby("year")
    .agg(
        countries=("iso3", "nunique"),
        min_score=("ai_readiness_score", "min"),
        max_score=("ai_readiness_score", "max"),
        mean_score=("ai_readiness_score", "mean"),
    )
    .round(2)
)

study_coverage_year_df = (
    oxford_raw_df[oxford_raw_df["iso3"].isin(study_iso3_all)]
    .groupby("year")
    .agg(
        countries=("iso3", "nunique"),
        active_63=("iso3", lambda values: values[values.isin(study_countries_63)].nunique()),
        solo_y_23=("iso3", lambda values: values[values.isin(study_solo_y)].nunique()),
    )
    .sort_index()
)

validation_latest = {
    "USA": 88.36,
    "CAN": 74.66,
    "FRA": 80.81,
    "GBR": 77.75,
    "SAU": 71.57,
}

print("=" * 70)
print("VERIFICACION OXFORD INSIGHTS - GOVERNMENT AI READINESS INDEX")
print("=" * 70)
print(f"\nStructured years integrados: {sorted(oxford_raw_df['year'].unique().tolist())}")
print("Anio sin dataset estructurado publico: 2024 (solo PDF disponible)")
print(f"Manifest de descarga: {manifest_path.relative_to(PROJECT_ROOT)}")
print(f"All raw:             {oxford_all_raw_path.relative_to(PROJECT_ROOT)}")
print(f"Wide:                {oxford_wide_path.relative_to(PROJECT_ROOT)}")
print(f"Study:               {oxford_study_path.relative_to(PROJECT_ROOT)}")
print(f"Long:                {oxford_long_path.relative_to(PROJECT_ROOT)}")
print(f"Snapshot latest:     {oxford_snapshot_path.relative_to(PROJECT_ROOT)}")
print(f"Pillars long:        {oxford_pillars_path.relative_to(PROJECT_ROOT)}")
print(f"Dimensions long:     {oxford_dimensions_path.relative_to(PROJECT_ROOT)}")

print("\nCobertura global por anio:")
display(coverage_year_df)

print("\nCobertura de la muestra (86 paises) por anio:")
display(study_coverage_year_df)

print(f"\nValidacion rapida latest year = {latest_year}:")
latest_lookup = oxford_snapshot_latest_df.set_index("iso3")
for iso3, expected in validation_latest.items():
    if iso3 in latest_lookup.index:
        actual = latest_lookup.loc[iso3, "ai_readiness_score"]
        status = "OK" if abs(float(actual) - expected) < 0.25 else "CHECK"
        print(f"  {iso3}: {float(actual):.2f} (esperado ~{expected:.2f}) -> {status}")
    else:
        print(f"  {iso3}: NO ENCONTRADO")

latest_missing_study = sorted(study_iso3_all - set(oxford_snapshot_latest_df["iso3"]))
print(f"\nPaises de la muestra sin Oxford latest ({latest_year}): {len(latest_missing_study)}")
print(latest_missing_study)

print(f"\nTop 15 latest year ({latest_year}) por ai_readiness_score:")
print(
    oxford_snapshot_latest_df.nlargest(15, "ai_readiness_score")
    [["oxford_rank_reported", "iso3", "country_name_std", "ai_readiness_score"]]
    .to_string(index=False)
)

print("\nNota de comparabilidad:")
print("- 2019 fue armonizado de 0-10 a 0-100 solo para panel; la escala original queda en ai_readiness_score_original.")
print("- 2020-2023 comparten una estructura de 3 pilares y 9 dimensiones.")
print("- 2025 cambia a 6 pilares y 14 dimensiones; se mantiene en el panel con flag metodologico explicito.")

VERIFICACION OXFORD INSIGHTS - GOVERNMENT AI READINESS INDEX

Structured years integrados: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
Anio sin dataset estructurado publico: 2024 (solo PDF disponible)
Manifest de descarga: data/raw/Oxford Insights/download_manifest.csv
All raw:             data/raw/Oxford Insights/oxford_ai_readiness_all_raw.csv
Wide:                data/raw/Oxford Insights/oxford_ai_readiness_wide.csv
Study:               data/raw/Oxford Insights/oxford_ai_readiness_study.csv
Long:                data/raw/Oxford Insights/oxford_ai_readiness_long.csv
Snapshot latest:     data/raw/Oxford Insights/oxford_ai_readiness_snapshot_latest.csv
Pillars long:        data/raw/Oxford Insights/oxford_ai_readiness_pillars_long.csv
Dimensions long:     data/raw/Oxford Insights/oxford_ai_readiness_dimensions_long.csv

Cobertura global por anio:


,countries,min_score,max_score,mean_score
year,,,,
2019,194,1.68,91.86,40.32
2020,172,19.07,85.48,44.23
2021,160,17.93,88.16,47.42
2022,181,13.46,85.72,44.61
2023,193,9.20,84.80,44.94
2024,188,14.62,87.03,47.59
2025,195,12.12,88.36,42.52



Cobertura de la muestra (86 paises) por anio:


,countries,active_63,solo_y_23
year,,,
2019,86,63,23
2020,85,63,22
2021,86,63,23
2022,86,63,23
2023,86,63,23
2024,86,63,23
2025,86,63,23



Validacion rapida latest year = 2025:
  USA: 88.36 (esperado ~88.36) -> OK
  CAN: 74.66 (esperado ~74.66) -> OK
  FRA: 80.81 (esperado ~80.81) -> OK
  GBR: 77.75 (esperado ~77.75) -> OK
  SAU: 71.57 (esperado ~71.57) -> OK

Paises de la muestra sin Oxford latest (2025): 0
[]

Top 15 latest year (2025) por ai_readiness_score:
 oxford_rank_reported iso3   country_name_std  ai_readiness_score
                    1  USA      United States               88.36
                    2  FRA             France               80.81
                    3  GBR     United Kingdom               77.75
                    4  NLD        Netherlands               77.18
                    5  KOR Korea, Republic of               76.89
                    6  DEU            Germany               76.78
                    7  SGP          Singapore               76.42
                    8  CHN              China               76.27
                    9  AUS          Australia               75.73
            

## Fuente 7 — WIPO Global Innovation Index (GII)

WIPO GII entra al estudio como fuente **X2** para la variable `gii_score` y como fuente ampliada de
subindices e indicadores de innovacion comparables a nivel pais.

En esta primera implementacion se integran las ediciones **2024** y **2025**, que si exponen
un dataset oficial estructurado en formato XLSX (`tech1.xlsx`).

Las ediciones **2019-2023** tienen landings oficiales y reportes PDF visibles, pero su descarga
binaria automatizada aun presenta friccion tecnica desde terminal. Por eso esta seccion separa:
- capa operativa inmediata: 2024-2025 con XLSX oficial;
- capa historica pendiente: 2019-2023, documentada en el manifiesto como descubrimiento oficial
  aun no resuelto para descarga reproducible.

**Cobertura del XLSX oficial 2024-2025**
- 2024: 133 economias, 109 indicadores
- 2025: 139 economias, 109 indicadores

El output preserva tanto el score compuesto `Global Innovation Index` como los subindices,
pilares y metadatos de economia (ingreso, region UN, poblacion, PPP GDP y PPP per capita).

In [1]:
# ============================================================================
# 7.0 Setup, rutas y discovery operativo de WIPO GII
# ============================================================================
from pathlib import Path
from datetime import datetime, timezone
import re
import pandas as pd

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

WIPO_PATH = PROJECT_ROOT / "data" / "raw" / "WIPO Global Innovation Index"
WIPO_REPORTS_PATH = WIPO_PATH / "reports"
WIPO_DATASETS_PATH = WIPO_PATH / "datasets"
WIPO_METADATA_PATH = WIPO_PATH / "metadata"
WIPO_MANIFEST_PATH = WIPO_PATH / "download_manifest.csv"

for path in [WIPO_PATH, WIPO_REPORTS_PATH, WIPO_DATASETS_PATH, WIPO_METADATA_PATH]:
    path.mkdir(parents=True, exist_ok=True)

# Intentar reutilizar la muestra del estudio ya cargada; si no existe en memoria,
# reconstruirla desde outputs ya confirmados del repo.
if "study_iso3_all" not in globals():
    oxford_study_path = PROJECT_ROOT / "data" / "raw" / "Oxford Insights" / "oxford_ai_readiness_study.csv"
    study_iso3_all = set(pd.read_csv(oxford_study_path)["iso3"].dropna().unique())

if "study_countries_63" not in globals():
    wdi_wide_path = PROJECT_ROOT / "data" / "raw" / "World Bank WDI" / "wdi_all_indicators_wide.csv"
    study_countries_63 = sorted(pd.read_csv(wdi_wide_path)["iso3"].dropna().unique().tolist())

if "study_solo_y" not in globals():
    study_solo_y = sorted(study_iso3_all - set(study_countries_63))

WIPO_ASSETS = [
    {
        "year": 2025,
        "asset_kind": "data_xlsx",
        "title": "WIPO GII 2025 structured dataset (tech1)",
        "relative_path": "datasets/2025_gii_tech1.xlsx",
        "url": "https://www.wipo.int/edocs/pubdocs/en/wipo-pub-2000-2025-tech1.xlsx",
        "source_tier": "official_primary",
    },
    {
        "year": 2025,
        "asset_kind": "report_pdf",
        "title": "WIPO GII 2025 main report",
        "relative_path": "reports/2025_global_innovation_index.pdf",
        "url": "https://www.wipo.int/web-publications/global-innovation-index-2025/assets/80937/global-innovation-index-2025-en.pdf",
        "source_tier": "official_primary",
    },
    {
        "year": 2024,
        "asset_kind": "data_xlsx",
        "title": "WIPO GII 2024 structured dataset (tech1)",
        "relative_path": "datasets/2024_gii_tech1.xlsx",
        "url": "https://www.wipo.int/edocs/pubdocs/en/wipo-pub-2000-2024-tech1.xlsx",
        "source_tier": "official_primary",
    },
    {
        "year": 2024,
        "asset_kind": "report_pdf",
        "title": "WIPO GII 2024 main report",
        "relative_path": "reports/2024_global_innovation_index.pdf",
        "url": "https://www.wipo.int/edocs/pubdocs/en/wipo-pub-2000-2024-en-global-innovation-index-2024.pdf",
        "source_tier": "official_primary",
    },
    {
        "year": 2023,
        "asset_kind": "report_pdf",
        "title": "WIPO GII 2023 main report",
        "relative_path": None,
        "url": "https://www.wipo.int/documents/d/global-innovation-index/docs-en-wipo-pub-2000-2023-en-main-report-global-innovation-index-2023-16th-edition.pdf",
        "source_tier": "official_discovered_pending_download",
    },
    {
        "year": 2022,
        "asset_kind": "report_pdf",
        "title": "WIPO GII 2022 main report",
        "relative_path": None,
        "url": "https://www.wipo.int/documents/d/global-innovation-index/docs-en-wipo-pub-2000-2022-en-main-report-global-innovation-index-2022-15th-edition.pdf",
        "source_tier": "official_discovered_pending_download",
    },
    {
        "year": 2021,
        "asset_kind": "report_pdf",
        "title": "WIPO GII 2021 main report",
        "relative_path": None,
        "url": "https://www.wipo.int/documents/d/global-innovation-index/docs-en-2021-wipo_pub_gii_2021.pdf",
        "source_tier": "official_discovered_pending_download",
    },
    {
        "year": 2020,
        "asset_kind": "report_pdf",
        "title": "WIPO GII 2020 main report",
        "relative_path": None,
        "url": "https://www.wipo.int/documents/d/global-innovation-index/docs-en-2020-wipo_pub_gii_2020.pdf",
        "source_tier": "official_discovered_pending_download",
    },
    {
        "year": 2019,
        "asset_kind": "report_landing",
        "title": "WIPO GII 2019 report landing / publication page",
        "relative_path": None,
        "url": "https://www.wipo.int/publications/en/details.jsp?id=4435",
        "source_tier": "official_discovered_pending_download",
    },
]

print("WIPO GII — assets inventariados:")
for asset in WIPO_ASSETS:
    asset_path = WIPO_PATH / asset["relative_path"] if asset["relative_path"] else None
    exists = asset_path.exists() if asset_path else False
    size = asset_path.stat().st_size if exists else None
    print(f"  {asset['year']} | {asset['asset_kind']:<12} | exists={exists:<5} | size={size}")

print(f"\nCobertura esperada de muestra: {len(study_iso3_all)} paises")
print(f"ACTIVOS via WDI: {len(study_countries_63)}")
print(f"SOLO Y: {len(study_solo_y)}")

WIPO GII — assets inventariados:
  2025 | data_xlsx    | exists=1     | size=935909
  2025 | report_pdf   | exists=1     | size=11969
  2024 | data_xlsx    | exists=1     | size=938609
  2024 | report_pdf   | exists=1     | size=48450231
  2023 | report_pdf   | exists=0     | size=None
  2022 | report_pdf   | exists=0     | size=None
  2021 | report_pdf   | exists=0     | size=None
  2020 | report_pdf   | exists=0     | size=None
  2019 | report_landing | exists=0     | size=None

Cobertura esperada de muestra: 86 paises
ACTIVOS via WDI: 63
SOLO Y: 23


In [2]:
# ============================================================================
# 7.1 Carga, normalizacion y ensamblado del panel WIPO 2024-2025
# ============================================================================

def wipo_slugify(name):
    slug = re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower())
    return re.sub(r"_+", "_", slug).strip("_")


def validate_pdf_signature(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    with open(path, "rb") as handle:
        return handle.read(4) == b"%PDF"


structured_assets = [asset for asset in WIPO_ASSETS if asset["asset_kind"] == "data_xlsx"]
wipo_frames = []

for asset in structured_assets:
    dataset_path = WIPO_PATH / asset["relative_path"]
    year = asset["year"]
    if not dataset_path.exists():
        raise FileNotFoundError(f"Falta el dataset WIPO esperado: {dataset_path}")

    data_df = pd.read_excel(dataset_path, sheet_name="Data")
    economies_df = pd.read_excel(dataset_path, sheet_name="Economies")
    structure_df = pd.read_excel(dataset_path, sheet_name="Index Structure")

    # Unir metadatos de economia e informacion descriptiva de cada indicador.
    merged_df = (
        data_df.merge(
            economies_df,
            on=["ISO3", "ECONOMY_NAME"],
            how="left",
            suffixes=("", "_economy"),
        )
        .merge(
            structure_df[["NUM", "NAME", "LEVEL", "TYPE", "PROFILE", "DESCRIPTION", "SOURCE", "WEBSITE"]],
            on=["NUM", "NAME"],
            how="left",
            suffixes=("", "_structure"),
        )
    )

    merged_df["year"] = year
    merged_df["indicator"] = merged_df["NAME"].apply(wipo_slugify)
    merged_df["iso3"] = merged_df["ISO3"].astype(str).str.upper()
    merged_df["country_name_std"] = merged_df["ECONOMY_NAME"]
    merged_df["country_name_wipo"] = merged_df["ECONOMY_NAME"]
    merged_df["indicator_code"] = merged_df["NUM"].fillna("GII")
    merged_df["indicator_name_wipo"] = merged_df["NAME"]
    merged_df["value"] = pd.to_numeric(merged_df["SCORE"], errors="coerce")
    merged_df["rank_reported"] = pd.to_numeric(merged_df["RANK"], errors="coerce").astype("Int64")
    merged_df["source_variable_original"] = merged_df["NAME"]
    merged_df["source_name"] = "WIPO_Global_Innovation_Index"
    merged_df["source_dataset"] = dataset_path.name
    merged_df["report_url"] = next((a["url"] for a in WIPO_ASSETS if a["year"] == year and a["asset_kind"] == "report_pdf"), None)
    merged_df["data_asset_url"] = asset["url"]
    merged_df["methodology_url"] = None
    merged_df["source_tier"] = asset["source_tier"]
    merged_df["coverage_level"] = "country"
    merged_df["report_edition"] = str(year)
    merged_df["methodology_regime"] = f"gii_{year}_official_release_109_indicators"
    merged_df["comparability_group"] = "review_pending_cross_year"
    merged_df["unit_original"] = "index_score_or_indicator_score"
    merged_df["notes_mapping"] = "ISO3 provisto por WIPO; score preservado desde sheet Data"
    merged_df["indicator_level"] = merged_df["LEVEL"]
    merged_df["indicator_type"] = merged_df["TYPE"]
    merged_df["indicator_profile"] = merged_df["PROFILE"]
    merged_df["indicator_description"] = merged_df["DESCRIPTION"]
    merged_df["indicator_source_wipo"] = merged_df["SOURCE"]
    merged_df["indicator_website"] = merged_df["WEBSITE"]
    merged_df["income_group_wipo"] = merged_df["INCOME"]
    merged_df["region_un"] = merged_df["REG_UN"]
    merged_df["region_un_code"] = merged_df["REG_UN_CODE"]
    merged_df["population_thousands"] = pd.to_numeric(merged_df["POP"], errors="coerce")
    merged_df["ppp_gdp_billions"] = pd.to_numeric(merged_df["PPPGDP"], errors="coerce")
    merged_df["ppp_per_capita"] = pd.to_numeric(merged_df["PPPPC"], errors="coerce")
    merged_df["data_year_original"] = pd.to_numeric(merged_df["DATAYR"], errors="coerce").astype("Int64")
    merged_df["value_screen_original"] = merged_df["VALUE_SCREEN"]
    merged_df["score_screen_warning_overall"] = merged_df["SW_OVERALL"]
    merged_df["score_screen_warning_income_group"] = merged_df["SW_INCGRP"]
    merged_df["outdated_flag"] = merged_df["OUTDATED"]
    merged_df["dmc_flag"] = merged_df["DMC"]

    keep_cols = [
        "iso3", "country_name_std", "country_name_wipo", "year", "indicator", "indicator_code",
        "indicator_name_wipo", "indicator_level", "indicator_type", "indicator_profile",
        "indicator_description", "indicator_source_wipo", "indicator_website", "value",
        "rank_reported", "value_screen_original", "data_year_original", "income_group_wipo",
        "region_un", "region_un_code", "population_thousands", "ppp_gdp_billions", "ppp_per_capita",
        "source_name", "source_dataset", "source_variable_original", "report_url", "data_asset_url",
        "methodology_url", "source_tier", "coverage_level", "report_edition", "methodology_regime",
        "comparability_group", "unit_original", "notes_mapping", "score_screen_warning_overall",
        "score_screen_warning_income_group", "outdated_flag", "dmc_flag",
    ]
    wipo_frames.append(merged_df[keep_cols].copy())

wipo_all_raw_df = pd.concat(wipo_frames, ignore_index=True)
wipo_all_raw_df = wipo_all_raw_df.sort_values(["year", "iso3", "indicator_name_wipo"]).reset_index(drop=True)

wipo_long_df = wipo_all_raw_df.copy()
wipo_study_df = wipo_all_raw_df[wipo_all_raw_df["iso3"].isin(study_iso3_all)].copy()

wipo_overall_df = wipo_all_raw_df[wipo_all_raw_df["indicator_name_wipo"] == "Global Innovation Index"].copy()
wipo_overall_df = wipo_overall_df.rename(columns={"value": "gii_score", "rank_reported": "gii_rank"})

wide_id_cols = [
    "iso3", "country_name_std", "country_name_wipo", "year", "income_group_wipo",
    "region_un", "region_un_code", "population_thousands", "ppp_gdp_billions", "ppp_per_capita",
]
wipo_wide_df = (
    wipo_all_raw_df.pivot_table(
        index=wide_id_cols,
        columns="indicator",
        values="value",
        aggfunc="first",
    )
    .reset_index()
)
wipo_wide_df.columns.name = None

latest_year_wipo = int(wipo_overall_df["year"].max())
wipo_snapshot_latest_df = wipo_overall_df[wipo_overall_df["year"] == latest_year_wipo].copy()

duplicate_wipo = wipo_overall_df.duplicated(["year", "iso3"], keep=False).sum()
if duplicate_wipo:
    raise RuntimeError(f"Duplicados en WIPO overall por year+iso3: {duplicate_wipo}")

print("WIPO panel ensamblado:")
print(f"  Filas raw: {len(wipo_all_raw_df)}")
print(f"  Economias 2024: {wipo_overall_df[wipo_overall_df['year'] == 2024]['iso3'].nunique()}")
print(f"  Economias 2025: {wipo_overall_df[wipo_overall_df['year'] == 2025]['iso3'].nunique()}")
print(f"  Indicadores unicos: {wipo_all_raw_df['indicator_name_wipo'].nunique()}")
print(f"  Cobertura muestra 2024: {wipo_overall_df[(wipo_overall_df['year'] == 2024) & (wipo_overall_df['iso3'].isin(study_iso3_all))]['iso3'].nunique()}/86")
print(f"  Cobertura muestra 2025: {wipo_overall_df[(wipo_overall_df['year'] == 2025) & (wipo_overall_df['iso3'].isin(study_iso3_all))]['iso3'].nunique()}/86")

WIPO panel ensamblado:
  Filas raw: 29648
  Economias 2024: 133
  Economias 2025: 139
  Indicadores unicos: 118
  Cobertura muestra 2024: 83/86
  Cobertura muestra 2025: 84/86


In [4]:
# ============================================================================
# 7.2 Guardado, manifiesto y verificacion WIPO
# ============================================================================
wipo_all_raw_path = WIPO_PATH / "wipo_gii_all_raw.csv"
wipo_study_path = WIPO_PATH / "wipo_gii_study.csv"
wipo_long_path = WIPO_PATH / "wipo_gii_long.csv"
wipo_wide_path = WIPO_PATH / "wipo_gii_wide.csv"
wipo_snapshot_path = WIPO_PATH / "wipo_gii_snapshot_latest.csv"

wipo_all_raw_df.to_csv(wipo_all_raw_path, index=False)
wipo_study_df.to_csv(wipo_study_path, index=False)
wipo_long_df.to_csv(wipo_long_path, index=False)
wipo_wide_df.to_csv(wipo_wide_path, index=False)
wipo_snapshot_latest_df.to_csv(wipo_snapshot_path, index=False)

manifest_rows = []
manifest_timestamp = datetime.now(tz=timezone.utc).isoformat()
for asset in WIPO_ASSETS:
    asset_path = WIPO_PATH / asset["relative_path"] if asset["relative_path"] else None
    exists = asset_path.exists() if asset_path else False
    file_size_bytes = float(asset_path.stat().st_size) if exists else None

    status = "discovered_only"
    available = False
    notes = ""

    if asset["asset_kind"] == "data_xlsx" and exists:
        status = "downloaded_valid"
        available = True
        notes = "XLSX oficial valido; integra score, rank y estructura rica por economia."
    elif asset["asset_kind"] == "report_pdf" and exists:
        if validate_pdf_signature(asset_path):
            status = "downloaded_valid"
            available = True
            notes = "PDF oficial valido archivado localmente."
        else:
            status = "downloaded_invalid_html_or_blocked"
            available = False
            notes = "La URL oficial respondio pero el archivo local no es PDF binario valido (posible challenge HTML / WAF)."
    elif asset["source_tier"] == "official_discovered_pending_download":
        status = "official_url_discovered_pending_download"
        available = False
        notes = "Landing o URL oficial confirmada; descarga binaria reproducible pendiente por friccion tecnica."

    manifest_rows.append({
        "year": asset["year"],
        "asset_kind": asset["asset_kind"],
        "title": asset["title"],
        "available": available,
        "status": status,
        "relative_path": asset["relative_path"],
        "absolute_path": str(asset_path) if asset_path else None,
        "url": asset["url"],
        "file_size_bytes": file_size_bytes,
        "source_tier": asset["source_tier"],
        "downloaded_at_utc": manifest_timestamp if exists else None,
        "notes": notes,
    })

manifest_df = pd.DataFrame(manifest_rows).sort_values(["year", "asset_kind", "title"]).reset_index(drop=True)
manifest_df.to_csv(WIPO_MANIFEST_PATH, index=False)

coverage_wipo_df = (
    wipo_overall_df.groupby("year")
    .agg(
        economies=("iso3", "nunique"),
        min_score=("gii_score", "min"),
        max_score=("gii_score", "max"),
        mean_score=("gii_score", "mean"),
    )
    .round(2)
)

study_coverage_wipo_df = (
    wipo_overall_df[wipo_overall_df["iso3"].isin(study_iso3_all)]
    .groupby("year")
    .agg(
        countries=("iso3", "nunique"),
        active_63=("iso3", lambda values: values[values.isin(study_countries_63)].nunique()),
        solo_y_23=("iso3", lambda values: values[values.isin(study_solo_y)].nunique()),
    )
    .sort_index()
)

validation_rank_order = {
    2024: ["CHE", "SWE", "USA", "SGP", "GBR", "KOR", "FIN", "NLD", "DEU", "DNK"],
    2025: ["CHE", "SWE", "USA", "KOR", "SGP", "GBR", "FIN", "NLD", "DNK", "CHN"],
}

print("=" * 70)
print("VERIFICACION WIPO GLOBAL INNOVATION INDEX")
print("=" * 70)
print(f"\nManifest de descarga: {WIPO_MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print(f"All raw:         {wipo_all_raw_path.relative_to(PROJECT_ROOT)}")
print(f"Study:           {wipo_study_path.relative_to(PROJECT_ROOT)}")
print(f"Long:            {wipo_long_path.relative_to(PROJECT_ROOT)}")
print(f"Wide:            {wipo_wide_path.relative_to(PROJECT_ROOT)}")
print(f"Snapshot latest: {wipo_snapshot_path.relative_to(PROJECT_ROOT)}")

print("\nCobertura global por anio (GII overall):")
display(coverage_wipo_df)

print("\nCobertura de la muestra por anio (GII overall):")
display(study_coverage_wipo_df)

print("\nValidacion de ranking oficial WIPO:")
for year, expected_top10 in validation_rank_order.items():
    actual_top10 = (
        wipo_overall_df[wipo_overall_df["year"] == year]
        .nsmallest(10, "gii_rank")
        ["iso3"]
        .tolist()
    )
    status = "OK" if actual_top10 == expected_top10 else "CHECK"
    print(f"  {year}: {status}")
    print(f"    esperado: {expected_top10}")
    print(f"    obtenido: {actual_top10}")

invalid_reports = manifest_df.loc[manifest_df["status"] == "downloaded_invalid_html_or_blocked", "year"].tolist()
pending_hist = manifest_df.loc[manifest_df["status"] == "official_url_discovered_pending_download", "year"].tolist()
print(f"\nReports invalidos/bloqueados: {invalid_reports}")
print(f"Historico oficial pendiente de descarga reproducible: {pending_hist}")
print("\nTop 10 latest year por gii_score:")
print(
    wipo_snapshot_latest_df.nsmallest(10, "gii_rank")
    [["gii_rank", "iso3", "country_name_std", "gii_score"]]
    .to_string(index=False)
)
print("\nNota metodologica:")
print("- `gii_score` se toma del row 'Global Innovation Index' del sheet Data.")
print("- 2024-2025 quedan integrados con dataset estructurado oficial WIPO.")
print("- 2019-2023 quedan registrados en el manifiesto como descubrimiento oficial pendiente de descarga reproducible.")

VERIFICACION WIPO GLOBAL INNOVATION INDEX

Manifest de descarga: data/raw/WIPO Global Innovation Index/download_manifest.csv
All raw:         data/raw/WIPO Global Innovation Index/wipo_gii_all_raw.csv
Study:           data/raw/WIPO Global Innovation Index/wipo_gii_study.csv
Long:            data/raw/WIPO Global Innovation Index/wipo_gii_long.csv
Wide:            data/raw/WIPO Global Innovation Index/wipo_gii_wide.csv
Snapshot latest: data/raw/WIPO Global Innovation Index/wipo_gii_snapshot_latest.csv

Cobertura global por anio (GII overall):


,economies,min_score,max_score,mean_score
year,,,,
2024,133,10.22,67.47,31.44
2025,139,11.94,65.96,31.49



Cobertura de la muestra por anio (GII overall):


,countries,active_63,solo_y_23
year,,,
2024,83,63,20
2025,84,63,21



Validacion de ranking oficial WIPO:
  2024: OK
    esperado: ['CHE', 'SWE', 'USA', 'SGP', 'GBR', 'KOR', 'FIN', 'NLD', 'DEU', 'DNK']
    obtenido: ['CHE', 'SWE', 'USA', 'SGP', 'GBR', 'KOR', 'FIN', 'NLD', 'DEU', 'DNK']
  2025: OK
    esperado: ['CHE', 'SWE', 'USA', 'KOR', 'SGP', 'GBR', 'FIN', 'NLD', 'DNK', 'CHN']
    obtenido: ['CHE', 'SWE', 'USA', 'KOR', 'SGP', 'GBR', 'FIN', 'NLD', 'DNK', 'CHN']

Reports invalidos/bloqueados: [2025]
Historico oficial pendiente de descarga reproducible: [2019, 2020, 2021, 2022, 2023]

Top 10 latest year por gii_score:
 gii_rank iso3             country_name_std  gii_score
        1  CHE                  Switzerland  65.961951
        2  SWE                       Sweden  62.578191
        3  USA     United States of America  61.689211
        4  KOR            Republic of Korea  59.997635
        5  SGP                    Singapore  59.943421
        6  GBR               United Kingdom  59.115376
        7  FIN                      Finland  57.706365
   

### 7.3 Extracción histórica GII desde PDFs oficiales (2020–2023)

Panel de score y rank GII extraído directamente de los PDFs oficiales WIPO.
- **2020**: formato columna única — `Country Score GlobalRank IncomeCode IncomeRank RegionCode RegionRank`
- **2021–2023**: formato doble columna — dos economías por línea con ranks enteros de subgrupo
- **2019**: solo disponible como "Key Findings" (20 páginas), sin tabla de ranking completa → excluido


In [7]:

# ============================================================================
# 7.3  Extracción histórica GII desde PDFs oficiales (2020–2023)
# ============================================================================
import pdfplumber
import pycountry

# ── ISO3 overrides para nombres propios de WIPO ─────────────────────────────
WIPO_ISO3_OVERRIDES = {
    "United States of America": "USA",
    "United States": "USA",
    "Republic of Korea": "KOR",
    "Korea, Republic of": "KOR",
    "United Kingdom": "GBR",
    "Viet Nam": "VNM",
    "Iran (Islamic Republic of)": "IRN",
    "Iran, Islamic Republic of": "IRN",
    "Syrian Arab Republic": "SYR",
    "Lao People's Democratic Republic": "LAO",
    "Lao People\u2019s Democratic Republic": "LAO",
    "Bolivia (Plurinational State of)": "BOL",
    "Venezuela (Bolivarian Republic of)": "VEN",
    "Netherlands (Kingdom of the)": "NLD",
    "Côte d'Ivoire": "CIV",
    "Côte d\u2019Ivoire": "CIV",
    "Cote d'Ivoire": "CIV",
    "Congo": "COG",
    "Democratic Republic of the Congo": "COD",
    "United Republic of Tanzania": "TZA",
    "Tanzania, United Republic of": "TZA",
    "Czech Republic": "CZE",
    "Czechia": "CZE",
    "Moldova": "MDA",
    "Republic of Moldova": "MDA",
    "North Macedonia": "MKD",
    "The former Yugoslav Republic of Macedonia": "MKD",
    "Brunei Darussalam": "BRN",
    "Palestine, State of": "PSE",
    "State of Palestine": "PSE",
    "Hong Kong, China": "HKG",
    "Macao, China": "MAC",
    "Cabo Verde": "CPV",
    "Cape Verde": "CPV",
    "Russian Federation": "RUS",
    "Chinese Taipei": "TWN",
    "Taiwan, Province of China": "TWN",
    "Sao Tome and Principe": "STP",
    "São Tomé and Príncipe": "STP",
    "Eswatini": "SWZ",
    "Swaziland": "SWZ",
    "Bosnia and Herzegovina": "BIH",
    "China": "CHN",
    "Turkey": "TUR",
    "Türkiye": "TUR",
    "Kosovo": "XKX",
    "North Korea": "PRK",
    "South Korea": "KOR",
    "Vietnam": "VNM",
}


def _iso3_from_name(name_raw: str):
    name = name_raw.strip()
    if name in WIPO_ISO3_OVERRIDES:
        return WIPO_ISO3_OVERRIDES[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None


# ── Tokenizer y parser para formato doble columna (2021–2023) ────────────────
def _tokenize(text: str):
    """Clasifica cada token como 'int', 'float', o 'word'."""
    toks = []
    for t in text.split():
        if re.match(r'^\d{1,3}$', t):
            toks.append(('int',   int(t)))
        elif re.match(r'^\d{1,2}\.\d{1,2}$', t):
            toks.append(('float', float(t)))
        else:
            toks.append(('word',  t))
    return toks


def _parse_two_col_tokens(tokens):
    """Extrae entradas con el patrón: int, word+, float, int, int."""
    entries = []
    i, n = 0, len(tokens)
    while i < n:
        if tokens[i][0] != 'int':
            i += 1; continue
        rank = tokens[i][1]
        i += 1
        name_parts = []
        while i < n and tokens[i][0] == 'word':
            name_parts.append(tokens[i][1]); i += 1
        if not name_parts or i >= n or tokens[i][0] != 'float':
            continue
        score = tokens[i][1]; i += 1
        if i >= n or tokens[i][0] != 'int':
            continue
        income_rank = tokens[i][1]; i += 1
        if i >= n or tokens[i][0] != 'int':
            continue
        region_rank = tokens[i][1]; i += 1
        # validación de rango razonable
        if 1 <= rank <= 200 and 10.0 <= score <= 75.0:
            entries.append({
                'economy':     ' '.join(name_parts),
                'gii_rank':    rank,
                'gii_score':   score,
                'income_rank': income_rank,
                'region_rank': region_rank,
            })
    return entries


# ── Parser 2020: columna única con código de ingreso/región ─────────────────
_PAT_2020 = re.compile(
    r'^(.+?)\s+([\d.]+)\s+(\d+)\s+(HI|UM|LM|LI)\s+(\d+)\s+([A-Z]{2,4})\s+(\d+)\s*$'
)
_INCOME_MAP_2020 = {
    "HI": "High income",
    "UM": "Upper middle income",
    "LM": "Lower middle income",
    "LI": "Low income",
}


def _parse_2020_page(page):
    rows = []
    for line in (page.extract_text() or "").split('\n'):
        m = _PAT_2020.match(line.strip())
        if m:
            economy, score, rank, inc_code, inc_rank, reg_code, reg_rank = m.groups()
            rows.append({
                'economy':          economy.strip(),
                'gii_rank':         int(rank),
                'gii_score':        float(score),
                'income_group_code': inc_code,
                'income_group_wipo': _INCOME_MAP_2020[inc_code],
                'income_rank':      int(inc_rank),
                'region_code':      reg_code,
                'region_rank':      int(reg_rank),
            })
    return rows


def parse_gii_pdf(path, year: int, start_page: int, max_pages: int = 8):
    """Extrae el ranking GII de un PDF oficial WIPO dado el año y página de inicio."""
    rows = []
    with pdfplumber.open(path) as pdf:
        total = len(pdf.pages)
        end = min(start_page - 1 + max_pages, total)
        for p_idx in range(start_page - 1, end):
            page = pdf.pages[p_idx]
            if year == 2020:
                rows.extend(_parse_2020_page(page))
            else:
                text = page.extract_text() or ""
                for line in text.split('\n'):
                    rows.extend(_parse_two_col_tokens(_tokenize(line)))

    df = (
        pd.DataFrame(rows)
        .drop_duplicates(subset=['gii_rank'])
        .sort_values('gii_rank')
        .reset_index(drop=True)
    )
    return df


# ── Configuración por año: archivo, página de inicio ────────────────────────
PDF_CFG = {
    2023: {'file': 'wipo-pub-2000-2023-en-main-report-global-innovation-index-2023-16th-edition.pdf', 'start': 19},
    2022: {'file': 'wipo-pub-2000-2022-en-main-report-global-innovation-index-2022-15th-edition.pdf', 'start': 21},
    2021: {'file': 'wipo_pub_gii_2021.pdf', 'start': 24},
    2020: {'file': 'wipo_pub_gii_2020.pdf', 'start': 33},
}

# ── Columnas estándar alineadas con wipo_overall_df ─────────────────────────
_OVERALL_COLS = list(wipo_overall_df.columns)   # referencia desde Cell 7.1

pdf_frames = []

for yr, cfg in sorted(PDF_CFG.items()):
    pdf_path = WIPO_DATASETS_PATH / cfg['file']
    if not pdf_path.exists():
        print(f"  AVISO: {pdf_path.name} no encontrado — saltando {yr}")
        continue

    print(f"\n>> Extrayendo {yr} desde {pdf_path.name} ...")
    raw = parse_gii_pdf(pdf_path, yr, start_page=cfg['start'], max_pages=10)

    if raw.empty:
        print(f"   ERROR: Sin datos extraídos para {yr}")
        continue

    # Resolver ISO3
    raw['iso3'] = raw['economy'].apply(_iso3_from_name)
    unmatched = raw[raw['iso3'].isna()]['economy'].tolist()
    if unmatched:
        print(f"   SIN ISO3 ({len(unmatched)}): {unmatched}")

    # Construir DataFrame alineado con wipo_overall_df
    inc_group = raw['income_group_wipo'] if 'income_group_wipo' in raw.columns else pd.Series([pd.NA] * len(raw))

    hist_df = pd.DataFrame({
        'iso3':                          raw['iso3'],
        'country_name_std':              raw['economy'],
        'country_name_wipo':             raw['economy'],
        'year':                          yr,
        'indicator':                     'global_innovation_index',
        'indicator_code':                'GII',
        'indicator_name_wipo':           'Global Innovation Index',
        'indicator_level':               pd.NA,
        'indicator_type':                pd.NA,
        'indicator_profile':             pd.NA,
        'indicator_description':         pd.NA,
        'indicator_source_wipo':         pd.NA,
        'indicator_website':             pd.NA,
        'gii_score':                     raw['gii_score'],
        'gii_rank':                      raw['gii_rank'].astype('Int64'),
        'value_screen_original':         pd.NA,
        'data_year_original':            pd.NA,
        'income_group_wipo':             inc_group.values,
        'region_un':                     pd.NA,
        'region_un_code':                pd.NA,
        'population_thousands':          pd.NA,
        'ppp_gdp_billions':              pd.NA,
        'ppp_per_capita':                pd.NA,
        'source_name':                   'WIPO_Global_Innovation_Index',
        'source_dataset':                pdf_path.name,
        'source_variable_original':      'Global Innovation Index',
        'report_url':                    pd.NA,
        'data_asset_url':                pd.NA,
        'methodology_url':               pd.NA,
        'source_tier':                   'pdf_historical',
        'coverage_level':                'country',
        'report_edition':                str(yr),
        'methodology_regime':            f'gii_{yr}_official_release_pdf_extracted',
        'comparability_group':           'gii_historical_pdf_only',
        'unit_original':                 'index_score_0_100',
        'notes_mapping':                 'Extraído de PDF oficial WIPO; solo score y rank GII total disponible',
        'score_screen_warning_overall':  pd.NA,
        'score_screen_warning_income_group': pd.NA,
        'outdated_flag':                 pd.NA,
        'dmc_flag':                      pd.NA,
    })

    # Reordenar columnas para que coincidan exactamente con wipo_overall_df
    for c in _OVERALL_COLS:
        if c not in hist_df.columns:
            hist_df[c] = pd.NA
    hist_df = hist_df[_OVERALL_COLS]

    n_total  = len(hist_df)
    n_iso3   = hist_df['iso3'].notna().sum()
    n_study  = hist_df['iso3'].isin(study_iso3_all).sum()
    print(f"   Economías: {n_total} | con ISO3: {n_iso3} | en muestra 86: {n_study}")
    print(f"   Top 5:")
    print(raw[['gii_rank','economy','gii_score']].head().to_string(index=False, col_space=4))
    pdf_frames.append(hist_df)

if not pdf_frames:
    raise RuntimeError("No se extrajo ningún año histórico; revisa las rutas de los PDFs.")



>> Extrayendo 2020 desde wipo_pub_gii_2020.pdf ...
   Economías: 131 | con ISO3: 131 | en muestra 86: 82
   Top 5:
 gii_rank                  economy  gii_score
        1              Switzerland      66.08
        2                   Sweden      62.47
        3 United States of America      60.56
        4           United Kingdom      59.78
        5              Netherlands      58.76

>> Extrayendo 2021 desde wipo_pub_gii_2021.pdf ...
   Economías: 132 | con ISO3: 132 | en muestra 86: 82
   Top 5:
 gii_rank                  economy  gii_score
        1              Switzerland       65.5
        2                   Sweden       63.1
        3 United States of America       61.3
        4           United Kingdom       59.8
        5        Republic of Korea       59.3

>> Extrayendo 2022 desde wipo-pub-2000-2022-en-main-report-global-innovation-index-2022-15th-edition.pdf ...
   Economías: 132 | con ISO3: 132 | en muestra 86: 81
   Top 5:
 gii_rank        economy  gii_score
      

In [8]:

# ============================================================================
# 7.4  Extensión del panel WIPO overall + guardado del histórico GII 2020–2025
# ============================================================================

# Extender wipo_overall_df con el historial extraído de PDFs
wipo_pdf_hist_df = pd.concat(pdf_frames, ignore_index=True)

wipo_overall_panel_df = pd.concat(
    [wipo_overall_df, wipo_pdf_hist_df], ignore_index=True
).sort_values(['year', 'gii_rank']).reset_index(drop=True)

# Verificar duplicados year+iso3
dup_panel = wipo_overall_panel_df.dropna(subset=['iso3']).duplicated(['year', 'iso3'], keep=False).sum()
if dup_panel:
    print(f"AVISO: {dup_panel} duplicados year+iso3 en panel extendido")

# ── Guardar panel histórico completo ────────────────────────────────────────
wipo_panel_path = WIPO_PATH / "wipo_gii_overall_panel.csv"
wipo_overall_panel_df.to_csv(wipo_panel_path, index=False)
print(f"Guardado: {wipo_panel_path}")
print(f"  Filas totales: {len(wipo_overall_panel_df)}")
print(f"  Años cubiertos: {sorted(wipo_overall_panel_df['year'].unique())}")

# ── Cobertura por año ────────────────────────────────────────────────────────
print("\nCobertura de muestra 86 por año:")
for y in sorted(wipo_overall_panel_df['year'].unique()):
    yr_df = wipo_overall_panel_df[wipo_overall_panel_df['year'] == y]
    n_total  = len(yr_df)
    n_study  = yr_df['iso3'].isin(study_iso3_all).sum()
    n_study63 = yr_df['iso3'].isin(study_countries_63).sum()
    src = yr_df['source_tier'].iloc[0] if len(yr_df) else '?'
    print(f"  {y}: {n_total} economías totales | {n_study}/86 muestra | {n_study63}/63 activos | fuente={src}")

# ── Validación de ranking: top-3 esperado por año ───────────────────────────
EXPECTED_TOP3 = {
    2020: ["Switzerland", "Sweden", "United States of America"],
    2021: ["Switzerland", "Sweden", "United States of America"],
    2022: ["Switzerland", "United States", "Sweden"],
    2023: ["Switzerland", "Sweden", "United States"],
    2024: ["Switzerland", "Sweden", "United States"],
    2025: ["Switzerland", "Sweden", "United States"],
}
print("\nValidación top-3 por año:")
all_ok = True
for y, expected in sorted(EXPECTED_TOP3.items()):
    yr_df = wipo_overall_panel_df[wipo_overall_panel_df['year'] == y].sort_values('gii_rank').head(3)
    actual = yr_df['country_name_wipo'].tolist()
    # La comparación es flexible: buscar los esperados entre top-3 con tolerancia a variantes de nombre
    expected_iso3 = [_iso3_from_name(n) for n in expected]
    actual_iso3   = yr_df['iso3'].tolist()
    match = set(expected_iso3) == set(actual_iso3) if None not in expected_iso3 else False
    status_marker = "OK" if match else "CHECK"
    if not match:
        all_ok = False
    print(f"  {y}: {actual} {status_marker}")
print(f"\nValidación global: {'PASS' if all_ok else 'REVISAR (ver CHECK arriba)'}")


Guardado: /Users/francoia/Documents/MIA/Proyecto Data-Science/research/data/raw/WIPO Global Innovation Index/wipo_gii_overall_panel.csv
  Filas totales: 799
  Años cubiertos: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Cobertura de muestra 86 por año:
  2020: 131 economías totales | 82/86 muestra | 63/63 activos | fuente=pdf_historical
  2021: 132 economías totales | 82/86 muestra | 63/63 activos | fuente=pdf_historical
  2022: 132 economías totales | 81/86 muestra | 63/63 activos | fuente=pdf_historical
  2023: 132 economías totales | 82/86 muestra | 63/63 activos | fuente=pdf_historical
  2024: 133 economías totales | 83/86 muestra | 63/63 activos | fuente=official_primary
  2025: 139 economías totales | 84/86 muestra | 63/63 activos | fuente=official_primary

Validación top-3 por año:
  2020: ['Switzerland', 'Sweden', 'United States of America'] OK
  2021: ['Switzerland', 'Sweden', 'United States of America'] OK
  2022: ['Switzer

/var/folders/gt/2tw2v2nd5d16pv2z34s77m4h0000gn/T/ipykernel_28598/3230422814.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wipo_overall_panel_df = pd.concat(


## Fuente 8 — IAPP Global AI Law & Policy Tracker (X1 Regulación)

Fuente principal: **IAPP Global AI Law and Policy Tracker** (PDF, Feb 2026).
Entra al estudio como fuente **complementaria y de validación cruzada de X1** frente a OECD.

**Rol en el estudio:**
- **Validación cruzada:** Confirmar/corregir `has_ai_law`, `regulatory_approach`, `year_enacted` vs OECD
- **Gap-filling:** Cubrir los 18 países sin datos X1 en OECD (BGD, BHR, BLR, BLZ, BRB, CHN, CMR, GHA, JOR, LBN, LKA, MNG, PAK, PAN, PHL, RUS, SYC, TWN)
- **Actualización temporal:** IAPP (Feb 2026) es más reciente que OECD (2024) — captura EU AI Act, KOR AI Basic Act, JPN AI Promotion Act

**Cobertura directa del tracker:** 29 jurisdicciones (Argentina, Australia, Bangladesh, Brazil, Canada, Chile, China, Colombia, Egypt, EU, Hong Kong, India, Indonesia, Israel, Japan, Kenya, Mauritius, New Zealand, Nigeria, Peru, Saudi Arabia, Singapore, South Korea, Taiwan, Turkey, UAE, UK, US, Vietnam).

**Propagación EU:** El EU AI Act (entrada en vigor agosto 2024) se propaga a los 27 estados miembros de la UE en la muestra.

**Variables X1 codificadas:** `has_ai_law`, `regulatory_approach`, `regulatory_intensity`, `year_enacted`, `enforcement_level`, `thematic_coverage`

In [9]:
# ═══════════════════════════════════════════════════════════════════════
# 8.1 — IAPP Setup, rutas y discovery de activos
# ═══════════════════════════════════════════════════════════════════════
import pdfplumber
import json
import re

IAPP_PATH = PROJECT_ROOT / "data" / "raw" / "IAPP"
IAPP_PATH.mkdir(parents=True, exist_ok=True)
for subdir in ["reports", "metadata", "manual_coding", "snapshots"]:
    (IAPP_PATH / subdir).mkdir(exist_ok=True)

IAPP_PDF_PATH = PROJECT_ROOT / "context_llm" / "IAPP-global_ai_law_policy_tracker.pdf"
IAPP_RAW_JSON = PROJECT_ROOT / "data" / "raw" / "iapp_tracker_raw_extracted.json"

# ─── IAPP Assets ───
IAPP_ASSETS = [
    {
        "name": "Global AI Law and Policy Tracker (PDF)",
        "kind": "report_pdf",
        "url": "https://assets.contentstack.io/v3/assets/bltd4dd5b2d705252bc/blt34a8e3844fb44942/global_ai_law_policy_tracker.pdf",
        "local_path": str(IAPP_PDF_PATH),
        "source_tier": "official_primary",
        "date_updated": "2026-02",
        "coverage": "29 jurisdictions",
        "role": "core_x1",
    },
    {
        "name": "Jurisdiction Overviews 2025 (PDF)",
        "kind": "report_pdf",
        "url": "https://assets.contentstack.io/v3/assets/bltd4dd5b2d705252bc/blt03a98a47faf09eb7/global_ai_governance_law_policy_series_2025.pdf",
        "local_path": str(IAPP_PATH / "reports" / "jurisdiction_overviews_2025.pdf"),
        "source_tier": "official_complement",
        "date_updated": "2025",
        "coverage": "11 jurisdictions",
        "role": "coding_support",
    },
    {
        "name": "IAPP Tracker Landing Page",
        "kind": "webpage",
        "url": "https://iapp.org/resources/article/global-ai-legislation-tracker/",
        "local_path": None,
        "source_tier": "official_reference",
        "date_updated": "2026-02",
        "coverage": "29 jurisdictions",
        "role": "reference",
    },
]

# ─── ISO3 Mapping ───
IAPP_TO_ISO3 = {
    'Argentina': 'ARG', 'Australia': 'AUS', 'Bangladesh': 'BGD',
    'Brazil': 'BRA', 'Canada': 'CAN', 'Chile': 'CHL',
    'China': 'CHN', 'Colombia': 'COL', 'Egypt': 'EGY',
    'Hong Kong': 'HKG', 'India': 'IND', 'Indonesia': 'IDN',
    'Israel': 'ISR', 'Japan': 'JPN', 'Kenya': 'KEN',
    'Mauritius': 'MUS', 'New Zealand': 'NZL', 'Nigeria': 'NGA',
    'Peru': 'PER', 'Saudi Arabia': 'SAU', 'Singapore': 'SGP',
    'South Korea': 'KOR', 'Taiwan': 'TWN', 'Turkey': 'TUR',
    'United Arab Emirates': 'ARE', 'United Kingdom': 'GBR',
    'United States': 'USA', 'Vietnam': 'VNM',
}

EU_MEMBERS_IN_STUDY = {
    'AUT': 'Austria', 'BEL': 'Belgium', 'BGR': 'Bulgaria',
    'CYP': 'Cyprus', 'CZE': 'Czechia', 'DEU': 'Germany',
    'DNK': 'Denmark', 'ESP': 'Spain', 'EST': 'Estonia',
    'FIN': 'Finland', 'FRA': 'France', 'GRC': 'Greece',
    'HRV': 'Croatia', 'HUN': 'Hungary', 'IRL': 'Ireland',
    'ITA': 'Italy', 'LTU': 'Lithuania', 'LUX': 'Luxembourg',
    'LVA': 'Latvia', 'MLT': 'Malta', 'NLD': 'Netherlands',
    'POL': 'Poland', 'PRT': 'Portugal', 'ROU': 'Romania',
    'SVK': 'Slovakia', 'SVN': 'Slovenia', 'SWE': 'Sweden',
}

# Inventory
print("IAPP ASSET INVENTORY:")
for asset in IAPP_ASSETS:
    lp = Path(asset["local_path"]) if asset["local_path"] else None
    exists = lp.exists() if lp else False
    size = f"{lp.stat().st_size / 1024:.0f} KB" if exists else "N/A"
    print(f"  [{asset['source_tier']:20s}] {asset['name']:45s} | exists={exists} | size={size} | role={asset['role']}")

print(f"\nStudy sample ISO3: {len(study_iso3_all)} countries")
print(f"IAPP direct jurisdictions: {len(IAPP_TO_ISO3)} (excl. EU)")
print(f"EU member states in study: {len(EU_MEMBERS_IN_STUDY)}")
print(f"Total potential IAPP coverage: {len(IAPP_TO_ISO3) + len(EU_MEMBERS_IN_STUDY) - 1} (EU counted once)")

IAPP ASSET INVENTORY:
  [official_primary    ] Global AI Law and Policy Tracker (PDF)        | exists=True | size=2360 KB | role=core_x1
  [official_complement ] Jurisdiction Overviews 2025 (PDF)             | exists=False | size=N/A | role=coding_support
  [official_reference  ] IAPP Tracker Landing Page                     | exists=False | size=N/A | role=reference

Study sample ISO3: 86 countries
IAPP direct jurisdictions: 28 (excl. EU)
EU member states in study: 27
Total potential IAPP coverage: 54 (EU counted once)


In [10]:
# ═══════════════════════════════════════════════════════════════════════
# 8.2 — Extracción completa del PDF del IAPP Tracker
# ═══════════════════════════════════════════════════════════════════════
# El PDF tiene tabla estructurada con 5 columnas por jurisdicción:
# [Country, Specific AI governance law/policy, Relevant authorities,
#  Other relevant laws/policies, Wider AI context]
# Los nombres de países están en texto espejo (reversed).

COUNTRY_DECODE = {
    'ANITNEGRA': 'Argentina',
    'AILARTSUA': 'Australia', 'deunitnoc\n,AILARTSUA': 'Australia',
    'HSEDALGNAB': 'Bangladesh',
    'LIZARB': 'Brazil',
    'ADANAC': 'Canada', 'ADANAC\n': 'Canada',
    'ELIHC': 'Chile',
    'ANIHC': 'China', 'ANIHC\n': 'China', 'deunitnoc\n,ANIHC': 'China',
    'AIBMOLOC': 'Colombia',
    'TPYGE': 'Egypt',
    'UE': 'EU', 'UE\n': 'EU', 'deunitnoc\n,UE': 'EU',
    'GNOK\nGNOH': 'Hong Kong',
    'AIDNI': 'India', 'deunitnoc\n,AIDNI': 'India',
    'AISENODNI': 'Indonesia',
    'LEARSI': 'Israel',
    'NAPAJ': 'Japan', 'NAPAJ\n': 'Japan', 'NAPAJ\n\uf069': 'Japan', 'deunitnoc\n,NAPAJ': 'Japan',
    'AYNEK': 'Kenya',
    'SUITIRUAM': 'Mauritius',
    'DNALAEZ\nWEN': 'New Zealand',
    'AIREGIN': 'Nigeria',
    'UREP': 'Peru',
    'AIBARA\nIDUAS': 'Saudi Arabia',
    'EROPAGNIS': 'Singapore', 'EROPAGNIS\n': 'Singapore', 'EROPAGNIS\n\uf069': 'Singapore', 'deunitnoc\n,EROPAGNIS': 'Singapore',
    'AEROK\nHTUOS': 'South Korea',
    'NAWIAT': 'Taiwan',
    'YEKRUT': 'Turkey',
    'SETARIME\nBARA\nDETINU': 'United Arab Emirates', 'SETARIME\nBARA\nDETINU\n': 'United Arab Emirates', 'SETARIME\nBARA\nDETINU\n\uf069': 'United Arab Emirates',
    '.K.U': 'United Kingdom', '.K.U\n': 'United Kingdom', 'deunitnoc\n,.K.U': 'United Kingdom',
    ')LAREDEF(\n.S.U': 'United States', ')LAREDEF(\n.S.U\n': 'United States', 'deunitnoc\n,)LAREDEF(\n.S.U': 'United States',
    'MANTEIV': 'Vietnam',
}

jurisdictions_raw = {}
with pdfplumber.open(IAPP_PDF_PATH) as pdf:
    print(f"IAPP PDF: {len(pdf.pages)} pages")
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        for table in tables:
            if not table or len(table) < 2 or len(table[0]) != 5:
                continue
            for row in table[1:]:
                raw_name = (row[0] or "").strip()
                policy = (row[1] or "").strip()
                authorities = (row[2] or "").strip()
                other_laws = (row[3] or "").strip()
                context = (row[4] or "").strip()
                if not policy and not authorities:
                    continue
                country = None
                for key, name in COUNTRY_DECODE.items():
                    if raw_name == key or raw_name.startswith(key):
                        country = name
                        break
                if country is None:
                    clean = raw_name.replace('\n', '').replace('\uf069', '')
                    country = clean[::-1] if clean else f"UNKNOWN_p{i+1}"
                if country not in jurisdictions_raw:
                    jurisdictions_raw[country] = {'policy': '', 'authorities': '', 'other_laws': '', 'context': '', 'pages': []}
                for field, val in [('policy', policy), ('authorities', authorities), ('other_laws', other_laws), ('context', context)]:
                    if val:
                        jurisdictions_raw[country][field] += ('\n' if jurisdictions_raw[country][field] else '') + val
                jurisdictions_raw[country]['pages'].append(i + 1)

# Save raw JSON for traceability
with open(IAPP_RAW_JSON, "w") as f:
    json.dump(jurisdictions_raw, f, indent=2, ensure_ascii=False)

print(f"\nJurisdictions extracted: {len(jurisdictions_raw)}")
for name in sorted(jurisdictions_raw):
    d = jurisdictions_raw[name]
    print(f"  {name:25s} | pages={d['pages']} | policy={len(d['policy']):5d}ch | auth={len(d['authorities']):4d}ch | laws={len(d['other_laws']):4d}ch | ctx={len(d['context']):5d}ch")

IAPP PDF: 41 pages

Jurisdictions extracted: 29
  Argentina                 | pages=[4] | policy= 1181ch | auth= 259ch | laws= 403ch | ctx=  971ch
  Australia                 | pages=[5, 6] | policy= 2241ch | auth= 325ch | laws= 454ch | ctx= 1971ch
  Bangladesh                | pages=[7] | policy=  451ch | auth=  51ch | laws= 304ch | ctx=  104ch
  Brazil                    | pages=[8] | policy= 1295ch | auth=  55ch | laws= 202ch | ctx=  913ch
  Canada                    | pages=[9] | policy=  943ch | auth= 243ch | laws= 482ch | ctx= 1360ch
  Chile                     | pages=[11] | policy=  478ch | auth= 310ch | laws= 614ch | ctx= 1177ch
  China                     | pages=[12, 13] | policy= 1453ch | auth= 206ch | laws= 265ch | ctx= 2204ch
  Colombia                  | pages=[14] | policy=  418ch | auth= 344ch | laws= 244ch | ctx=  566ch
  EU                        | pages=[16, 17] | policy= 1743ch | auth=1084ch | laws= 299ch | ctx= 2108ch
  Egypt                     | pages=[15] | pol

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 8.3 — Codificación manual X1 por jurisdicción IAPP
# ═══════════════════════════════════════════════════════════════════════
# Rúbrica de codificación:
# has_ai_law (0/1): 1 si existe ≥1 ley vinculante AI-específica EN VIGOR
# regulatory_approach: none|light_touch|strategy_led|regulation_focused|comprehensive
# regulatory_intensity (0-10): score compuesto de profundidad regulatoria
# year_enacted: año de primera regulación vinculante AI-específica
# enforcement_level: none|low|medium|high
# thematic_coverage (0-15): conteo de temas cubiertos
#
# Fuente primaria: IAPP PDF text content analyzed rigorously per jurisdiction
# Los países adicionales (no cubiertos por IAPP) se codifican con
# evidencia de UNESCO, OECD AI Policy Observatory, Stanford HAI, fuentes oficiales.
# ═══════════════════════════════════════════════════════════════════════

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.iapp_coding import (
    build_additional_countries,
    build_iapp_raw_table,
    build_iapp_x1,
    IAPP_CODING,
    study_iso3,
 )

# Build IAPP raw + coded + additional
iapp_raw_structured_df = build_iapp_raw_table()
iapp_direct_df = build_iapp_x1()
iapp_additional_df = build_additional_countries()
iapp_all_df = pd.concat([iapp_direct_df, iapp_additional_df], ignore_index=True)
iapp_study_df = iapp_all_df[iapp_all_df['iso3'].isin(study_iso3)].sort_values('iso3').reset_index(drop=True)

print(f"IAPP X1 CODING RESULTS")
print(f"  Structured raw tracker jurisdictions: {iapp_raw_structured_df['jurisdiction_iapp'].nunique()}")
print(f"  Direct IAPP jurisdictions: {iapp_direct_df.shape[0]} rows ({iapp_direct_df[iapp_direct_df['iso3'].isin(study_iso3)].shape[0]} in study)")
print(f"  Additional research-coded: {iapp_additional_df.shape[0]} countries")
print(f"  Study sample coverage: {iapp_study_df['iso3'].nunique()}/{len(study_iso3)} ({iapp_study_df['iso3'].nunique()/len(study_iso3)*100:.0f}%)")

# Distributions
print(f"\n  has_ai_law:         {dict(iapp_study_df['has_ai_law'].value_counts())}")
print(f"  regulatory_approach: {dict(iapp_study_df['regulatory_approach'].value_counts())}")
print(f"  enforcement_level:   {dict(iapp_study_df['enforcement_level'].value_counts())}")
print(f"  reg_intensity mean:  {iapp_study_df['regulatory_intensity'].mean():.1f}")
print(f"  thematic_cov mean:   {iapp_study_df['thematic_coverage'].mean():.1f}")

# Countries with binding AI law
has_law = iapp_study_df[iapp_study_df['has_ai_law'] == 1].sort_values('year_enacted')
print(f"\nCountries with binding AI-specific law ({len(has_law)}):")
for _, r in has_law.iterrows():
    yr = int(r['year_enacted']) if pd.notna(r['year_enacted']) else 'N/A'
    print(f"  {r['iso3']} ({r['jurisdiction_iapp'][:30]:30s}) | year={yr} | {r['regulatory_approach']}")

IAPP X1 CODING RESULTS
  Direct IAPP jurisdictions: 55 rows (54 in study)
  Additional research-coded: 32 countries
  Study sample coverage: 86/86 (100%)

  has_ai_law:         {0: np.int64(54), 1: np.int64(32)}
  regulatory_approach: {'strategy_led': np.int64(39), 'comprehensive': np.int64(32), 'light_touch': np.int64(10), 'none': np.int64(5)}
  enforcement_level:   {'high': np.int64(31), 'low': np.int64(30), 'medium': np.int64(14), 'none': np.int64(11)}
  reg_intensity mean:  5.3
  thematic_cov mean:   8.3

Countries with binding AI-specific law (32):
  RUS (Additional research (RUS)     ) | year=2020 | comprehensive
  CHN (China                         ) | year=2021 | comprehensive
  PER (Peru                          ) | year=2023 | comprehensive
  AUT (EU (Austria)                  ) | year=2024 | comprehensive
  SVK (EU (Slovakia)                 ) | year=2024 | comprehensive
  ROU (EU (Romania)                  ) | year=2024 | comprehensive
  PRT (EU (Portugal)                 )

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 8.3 — Estado de consolidación X1 (OECD + IAPP)
# ═══════════════════════════════════════════════════════════════════════
# NOTA (abril 2025): La reconciliación OECD + IAPP ya fue ejecutada
# por src/consolidate_x1.py → data/interim/x1_consolidated.csv
# y cerrada oficialmente por src/build_source_masters.py → data/interim/x1_master.csv
#
# En esta etapa (Extract), cada fuente se guarda cruda e independiente:
#   data/raw/OECD/oecd_x1_core.csv   ← solo datos OECD, sin mezcla
#   data/raw/IAPP/iapp_x1_core.csv   ← solo datos IAPP, sin mezcla
# ═══════════════════════════════════════════════════════════════════════

oecd_x1_path = PROJECT_ROOT / "data" / "raw" / "OECD" / "oecd_x1_core.csv"
oecd_x1_df = pd.read_csv(oecd_x1_path)

print("FUENTES X1 DISPONIBLES:")
print(f"  OECD  → data/raw/OECD/oecd_x1_core.csv     : {oecd_x1_df['iso3'].nunique()} países")
print(f"  IAPP  → data/raw/IAPP/iapp_x1_core.csv      : {iapp_study_df['iso3'].nunique()} países")
print(f"\n  Reconciliación OECD + IAPP → COMPLETADA (src/consolidate_x1.py)")
print(f"  Output consolidado         → data/interim/x1_consolidated.csv")
print(f"  Master oficial             → data/interim/x1_master.csv (86 países, 4 grupos regulatorios)")

RECONCILIATION SUMMARY
  Status: {'DIVERGE': np.int64(32), 'PARTIAL_AGREE': np.int64(22), 'IAPP_FILL_GAP': np.int64(18), 'AGREE': np.int64(14)}
  Source: {'OECD+IAPP': np.int64(36), 'IAPP_override': np.int64(32), 'IAPP_fill_gap': np.int64(18)}
  Confidence: {'high': np.int64(36), 'medium_needs_review': np.int64(32), 'low': np.int64(15), 'medium': np.int64(3)}

  has_ai_law: {0: np.int64(54), 1: np.int64(32)}
  approach: {'strategy_led': np.int64(39), 'comprehensive': np.int64(32), 'light_touch': np.int64(10), 'none': np.int64(5)}

DIVERGENCES (32 countries — IAPP override used as more recent):
  AUS: OECD(law=1,regulation_focused) → IAPP(law=0,strategy_led)
  BGR: OECD(law=0,strategy_led) → IAPP(law=1,comprehensive)
  BRA: OECD(law=1,comprehensive) → IAPP(law=0,strategy_led)
  CHL: OECD(law=1,light_touch) → IAPP(law=0,strategy_led)
  COL: OECD(law=1,comprehensive) → IAPP(law=0,strategy_led)
  CRI: OECD(law=1,comprehensive) → IAPP(law=0,strategy_led)
  CZE: OECD(law=0,strategy_led) → IA

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 8.4 — Guardar outputs IAPP (raw) y QA de extracción
# ═══════════════════════════════════════════════════════════════════════
# Solo se guardan datos crudos IAPP — sin mezclar con ninguna otra fuente.
# ═══════════════════════════════════════════════════════════════════════
from datetime import datetime

# ── Guardar raw IAPP ─────────────────────────────────────────────────
# Tabla raw estructurada del tracker (29 jurisdicciones directas, con texto y páginas)
iapp_raw_structured_df.to_csv(IAPP_PATH / "iapp_tracker_structured_raw.csv", index=False)

# Todos los jurisdicciones codificados (incluyendo HKG fuera del estudio)
iapp_all_df.to_csv(IAPP_PATH / "iapp_all_coded.csv", index=False)

# Solo los 86 países del estudio, valores IAPP directos sin modificar
iapp_study_df.to_csv(IAPP_PATH / "iapp_x1_core.csv", index=False)

# Archivo plano solicitado por la guía metodológica (solo columnas core)
iapp_reg_df = iapp_study_df[[
    'iso3', 'has_ai_law', 'regulatory_approach', 'regulatory_intensity',
    'year_enacted', 'enforcement_level', 'thematic_coverage',
    'evidence_summary', 'source'
]].copy()
iapp_reg_df.to_csv(PROJECT_ROOT / "data" / "raw" / "iapp_regulatory.csv", index=False)

# Manifesto de activos descargados
manifest_rows = []
for asset in IAPP_ASSETS:
    lp = Path(asset["local_path"]) if asset["local_path"] else None
    manifest_rows.append({
        'asset_name': asset['name'],
        'kind': asset['kind'],
        'url': asset['url'],
        'local_path': str(lp) if lp else '',
        'exists': lp.exists() if lp else False,
        'size_bytes': lp.stat().st_size if (lp and lp.exists()) else 0,
        'source_tier': asset['source_tier'],
        'role': asset['role'],
        'date_updated': asset['date_updated'],
        'download_timestamp': datetime.now().isoformat(),
    })
iapp_manifest_df = pd.DataFrame(manifest_rows)
iapp_manifest_df.to_csv(IAPP_PATH / "download_manifest.csv", index=False)

print("OUTPUTS RAW IAPP GUARDADOS:")
for f in [
    IAPP_PATH / "iapp_tracker_structured_raw.csv",
    IAPP_PATH / "iapp_all_coded.csv",
    IAPP_PATH / "iapp_x1_core.csv",
    IAPP_PATH / "download_manifest.csv",
    PROJECT_ROOT / "data" / "raw" / "iapp_regulatory.csv",
]:
    df_tmp = pd.read_csv(f)
    rel = f.relative_to(PROJECT_ROOT / "data" / "raw")
    print(f"  {rel}: {df_tmp.shape[0]} filas × {df_tmp.shape[1]} cols")

# ── QA de extracción IAPP ────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"QA EXTRACCIÓN IAPP (sobre datos crudos IAPP, sin otras fuentes)")
print(f"{'='*60}")

# Cobertura
n_covered = iapp_study_df['iso3'].nunique()
print(f"  Cobertura X1 estudio: {n_covered}/{len(study_iso3)} países ({n_covered/len(study_iso3)*100:.0f}%)")
print(f"  Jurisdicciones raw directas del tracker: {iapp_raw_structured_df['jurisdiction_iapp'].nunique()}")

# Duplicados
dup = iapp_study_df[iapp_study_df.duplicated(subset=['iso3'], keep=False)]
print(f"  Duplicados iso3: {len(dup)}")

# NaN en campos críticos
for col in ['has_ai_law', 'regulatory_approach', 'enforcement_level']:
    n_na = iapp_study_df[col].isna().sum()
    print(f"  NaN en {col}: {n_na}/{len(iapp_study_df)}")

# Validar que el raw estructurado preserva texto en las 4 secciones del tracker
for col in ['policy_text', 'authorities_text', 'other_laws_text', 'context_text']:
    n_empty = (iapp_raw_structured_df[col].fillna('').str.strip() == '').sum()
    print(f"  Vacíos en {col}: {n_empty}/{len(iapp_raw_structured_df)}")

# Distribuciones IAPP puras
print(f"\n  has_ai_law:          {dict(iapp_study_df['has_ai_law'].value_counts())}")
print(f"  regulatory_approach: {dict(iapp_study_df['regulatory_approach'].value_counts())}")
print(f"  enforcement_level:   {dict(iapp_study_df['enforcement_level'].value_counts())}")

# Spot-checks sobre valores IAPP directos
ANCHOR_CHECKS = {
    'CHN': {'has_ai_law': 1, 'approach': 'comprehensive'},
    'USA': {'has_ai_law': 0, 'approach': 'strategy_led'},
    'GBR': {'has_ai_law': 0, 'approach': 'strategy_led'},
    'FRA': {'has_ai_law': 1, 'approach': 'comprehensive'},  # EU AI Act
    'DEU': {'has_ai_law': 1, 'approach': 'comprehensive'},  # EU AI Act
    'KOR': {'has_ai_law': 1, 'approach': 'comprehensive'},  # AI Basic Act 2025
    'JPN': {'has_ai_law': 1, 'approach': 'comprehensive'},  # AI Promotion Act 2025
    'BRA': {'has_ai_law': 0, 'approach': 'strategy_led'},
    'IND': {'has_ai_law': 0, 'approach': 'light_touch'},
    'SGP': {'has_ai_law': 0, 'approach': 'strategy_led'},
    'RUS': {'has_ai_law': 1, 'approach': 'comprehensive'},
    'ARG': {'has_ai_law': 0, 'approach': 'strategy_led'},
}

print(f"\n  Spot-checks sobre valores IAPP:")
all_pass = True
for iso3, expected in ANCHOR_CHECKS.items():
    row = iapp_study_df[iapp_study_df['iso3'] == iso3]
    if len(row) == 0:
        print(f"    {iso3}: MISSING!")
        all_pass = False
        continue
    r = row.iloc[0]
    law_ok = r['has_ai_law'] == expected['has_ai_law']
    app_ok = r['regulatory_approach'] == expected['approach']
    ok = law_ok and app_ok
    if not ok:
        all_pass = False
    print(f"    {iso3}: law={r['has_ai_law']} ({'OK' if law_ok else 'FAIL'}), approach={r['regulatory_approach']} ({'OK' if app_ok else 'FAIL'}) {'✓' if ok else '✗'}")

print(f"\n  QA STATUS: {'PASS ✓' if all_pass and len(dup)==0 else 'REVISAR ✗'}")
print(f"
  → Reconciliación con OECD COMPLETADA (src/consolidate_x1.py → data/interim/x1_master.csv)")

OUTPUTS SAVED:
  IAPP/download_manifest.csv: 3 rows × 10 cols
  IAPP/iapp_all_coded.csv: 87 rows × 11 cols
  IAPP/iapp_oecd_reconciliation.csv: 86 rows × 16 cols
  IAPP/iapp_x1_consolidated.csv: 86 rows × 10 cols
  IAPP/iapp_x1_core.csv: 86 rows × 11 cols
  IAPP/snapshots/iapp_x1_snapshot_2026.csv: 86 rows × 10 cols
  iapp_regulatory.csv: 86 rows × 9 cols

QA CHECKS
  Coverage: 86/86 countries (100%)
  Duplicates iso3: 0
  NaN in has_ai_law: 0/86
  NaN in regulatory_approach: 0/86
  NaN in enforcement_level: 0/86

  Anchor spot-checks:
    CHN: law=1 (OK), approach=comprehensive (OK) ✓
    USA: law=0 (OK), approach=strategy_led (OK) ✓
    GBR: law=0 (OK), approach=strategy_led (OK) ✓
    FRA: law=1 (OK), approach=comprehensive (OK) ✓
    DEU: law=1 (OK), approach=comprehensive (OK) ✓
    KOR: law=1 (OK), approach=comprehensive (OK) ✓
    JPN: law=1 (OK), approach=comprehensive (OK) ✓
    BRA: law=0 (OK), approach=strategy_led (OK) ✓
    IND: law=0 (OK), approach=light_touch (OK) ✓
    

'iso3  has_ai_law regulatory_approach  regulatory_intensity enforcement_level  thematic_coverage     x1_source          confidence\n ARE           0        strategy_led                   5.0            medium                9.0     OECD+IAPP                high\n ARG           0        strategy_led                   8.0               low               10.0     OECD+IAPP                high\n ARM           0         light_touch                   1.0              none                2.0     OECD+IAPP                high\n AUS           0        strategy_led                   5.0            medium               11.0 IAPP_override medium_needs_review\n AUT           1       comprehensive                  25.0              high               25.0     OECD+IAPP                high\n BEL           1       comprehensive                  14.0              high               28.0     OECD+IAPP                high\n BGD           0        strategy_led                   2.0               low      

## Cierre de Recolección — Estado Final

Todas las **8 fuentes** del estudio han sido extraídas, validadas y guardadas como datos crudos independientes en `data/raw/`. La consolidación y cierre oficial se ejecuta mediante scripts en `src/`:

| Fuente | Variable principal | Script de cierre | Master output |
|---|---|---|---|
| Stanford AI Index 2025 | ai_investment, ai_startups, ai_patents | `build_source_masters.py` | `y_stanford_master.csv` |
| Microsoft AIEI | ai_adoption_rate | `build_source_masters.py` | `y_microsoft_master.csv` |
| Oxford Insights | ai_readiness_score | `build_source_masters.py` | `y_oxford_master.csv` |
| WIPO GII | gii_score, region_un | `build_source_masters.py` | `x2_wipo_master.csv` |
| World Bank WDI/WGI | gdp, internet, R&D, education, gov_eff | `build_source_masters.py` | `x2_wb_master.csv` |
| OECD STI/MSTI | vc_proxy, publications | `build_source_masters.py` | `oecd_robustness_master.csv` |
| EC-OECD + IAPP | has_ai_law, regulatory_approach, etc. | `consolidate_x1.py` + `build_source_masters.py` | `x1_master.csv` |

**Dataset definitivo:** `data/interim/sample_ready_cross_section.csv` (86 países, 72 con modelo PRINCIPAL completo).

**Pipeline de ejecución:** ver `info_data/ETL_RUNBOOK.md` y `info_data/DATA_DECISIONS_LOG.md`.

**Siguiente paso:** `02_limpieza.ipynb` carga el dataset sample-ready para iniciar limpieza y análisis.